# EPIC-HAR — **E**fficient, **P**runed, **I**nterpretable, **C**alibrated & compliant Human Activity Recognition
*A human-centred framework for activity and gesture recognition on pervasive devices under user heterogeneity, device heterogeneity and label noise. Evaluated on HHAR (Stisen et al., SenSys'15) and LMDHG.*

**Core idea.** Accuracy, compute, fairness and accountability are usually tuned one at a time, each in its own pipeline. EPIC-HAR puts them in **one constrained objective** and **one auditable decision policy**:

| Pillar | Mechanism in this notebook | What it guarantees or measures |
|---|---|---|
| **A — Feature selection** | Hard-Concrete L0 gates on *feature groups* (sensor axes, magnitudes, derivatives / hand joints) with a data-minimisation price | Which sensors and joints are needed at all (GDPR Art. 5(1)(c)); fewer inputs to collect |
| **B — Adaptive large model** | The **same budget** gates attention heads, MLP neurons and whole layers (light transformer) or LLM depth (frozen GPT-2/Qwen + LoRA). Gates are **folded into the weights** after training | A physically smaller model that matches the gated model's output exactly (tested to ~1e-15 in float64) |
| **C — HCAI** | Inherently interpretable prototype head (Rudin'19); calibrated **conformal sets**; boundary-condition and OOD guardrails; **abstention**; Art. 22 human review; counterfactual recourse; fairness audit including **abstention parity**; hash-chained ledger; **bias-audited override loop** | Every automated output is justified (in-distribution, valid input, unambiguous). Every non-automated one is routed, explained, logged and appealable |

Run order: Cells 1–6 define everything (Cell 2b fetches the data). Cell 7 trains and evaluates on HHAR and LMDHG. Cells 8–9 produce figures and the HCAI demo. Cell 10 runs the ablations. Cell 11 saves the memory/speed/accuracy benchmark.


### Cell 1 — Setup and configuration

In [ ]:
# ============================================================================
# CELL 1 — Environment, paths, configuration
# ============================================================================
import os, sys, math, json, time, glob, re, copy, random, hashlib, hmac, warnings
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple, Callable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

warnings.filterwarnings('ignore', category=UserWarning)

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/working')
if os.environ.get('EPIC_PROJECT_DIR'):             # Kaggle / scripted runs: no Drive, explicit folder
    PROJECT_DIR = os.environ['EPIC_PROJECT_DIR']
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/EPIC_HAR'
else:
    PROJECT_DIR = os.path.abspath('./EPIC_HAR')
os.makedirs(PROJECT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')   # needed for deterministic cuBLAS; set before CUDA init


def set_seed(seed: int = 42, deterministic: bool = False):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:      # identical reruns (slower); GPU atomics otherwise make same-seed runs differ
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=True)


@dataclass
class EPICConfig:
    # ---------------- data ----------------
    dataset: str = 'hhar'            # 'hhar' | 'lmdhg' | 'synthetic'
    data_dir: str = f'{PROJECT_DIR}/data/hhar'   # folder holding the CSVs / .mat files
    out_dir: str = f'{PROJECT_DIR}/runs'
    hhar_stream: str = 'Phones'      # 'Phones' | 'Watch'
    fs: float = 50.0                 # common resampling rate (Hz) — removes sampling-rate heterogeneity
    win: int = 128                   # 2.56 s windows
    stride: int = 64                 # 50 % overlap
    min_purity: float = 0.8          # window label purity; impure-but-kept windows are down-weighted
    keep_null: bool = False
    allow_missing_gyro: bool = True  # keep accelerometer-only devices (gyro = 0) instead of dropping them
    max_gap_s: float = 0.2           # windows containing larger sensor gaps are dropped
    lmdhg_T: int = 64                # frames per resampled gesture (segmented mode)
    lmdhg_mode: str = 'segmented'    # 'segmented' (isolated gestures) | 'continuous' (sliding windows, IDLE = class)
    lmdhg_fps: float = 30.0          # nominal frame rate used for velocity features (only scales them)
    lmdhg_label_policy: str = 'as_is'  # 'as_is' | 'merge' (applies LMDHG_MERGE)
    lmdhg_subject_csv: str = ''      # optional csv: file,subject (from ComplementaryInfo_LMDHG)
    # ---------------- protocol ----------------
    protocol: str = 'LOSO'           # 'LOSO' (leave-one-subject-out) | 'LODO' (leave-one-device-model-out)
    max_folds: Optional[int] = None  # limit folds for quick runs
    group_kfold: int = 5             # used when there are many subjects (LMDHG)
    # ---------------- model ----------------
    backbone: str = 'light'          # 'light' (gated transformer) | 'llm' (frozen pretrained LM + LoRA)
    llm_name: str = 'gpt2'           # e.g. 'gpt2', 'Qwen/Qwen2.5-0.5B', 'HuggingFaceTB/SmolLM2-135M'
    llm_layers: int = 6              # truncate the LM to its first k blocks (One-Fits-All style)
    lora_r: int = 8
    d_model: int = 128
    n_layers: int = 4
    n_heads: int = 4
    mlp_ratio: float = 4.0
    patch: int = 16
    patch_stride: int = 8
    emb_dim: int = 64
    protos_per_class: int = 3
    dropout: float = 0.1
    # ---------------- pillars A + B: joint feature/weight selection ----------------
    use_gates: bool = True
    budget: float = 0.35             # target compute (expected FLOPs) as a fraction of the dense model
    lambda_lr: float = 2.0           # dual-ascent step for the budget multiplier (per epoch of constraint violation)
    gate_warmup_epochs: int = 3
    budget_f1_tol: float = 0.03      # max val-F1 we are willing to pay to meet the budget
    minimization_weight: float = 0.05  # GDPR Art. 5(1)(c): price on every feature group kept (privacy cost, not FLOPs)
    gate_lr: float = 0.05            # gates need a much larger step than weights to open/close
    feature_gates: Optional[bool] = None   # pillar A: learned selection of input feature groups
    struct_gates: Optional[bool] = None    # pillar B: learned selection of heads / neurons / layers
    fixed_groups: Optional[List[str]] = None  # keep exactly these feature groups, not learned (random / filter controls)
    # ---------------- heterogeneity / noise ----------------
    use_dro: bool = True             # group-DRO over training subjects
    dro_eta: float = 0.05
    use_adv: bool = True             # subject- and device-adversarial invariance (GRL)
    adv_weight: float = 0.1
    use_consistency: bool = True     # JSD consistency across device-physics views
    cons_weight: float = 0.5
    class_balance: bool = True       # inverse-sqrt class-frequency weights (IDLE dominates continuous LMDHG)
    gce_q: float = 0.7               # generalised cross-entropy (label-noise robust); 0 -> plain CE
    boundary_weight: float = 0.5
    max_rot_deg: float = 30.0
    sensor_dropout: float = 0.2      # P(zero an optional sensor in a training view): robustness to missing gyro
    # ---------------- optimisation ----------------
    epochs: int = 30
    batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 1e-2
    patience: int = 6
    num_workers: int = 2
    # ---------------- pillar C: trust layer ----------------
    alpha: float = 0.1               # conformal miscoverage (target coverage 90 %)
    decision_alpha: Optional[float] = None  # alpha used for the auto/abstain decision (None = alpha)
    conformal_mode: str = 'class'    # 'marginal' | 'class' (Mondrian, class-conditional)
    ood_quantile: float = 0.99
    critical_classes: Tuple[str, ...] = ()   # classes that always need human sign-off (GDPR Art. 22)
    pseudonym_salt: str = 'CHANGE-ME-AND-STORE-SECRETLY'
    seed: int = 42                   # training seed (vary for >=3 seeds)
    deterministic: bool = True       # deterministic GPU kernels: same seed -> same result
    split_seed: int = 42             # fold / calibration split seed (keep FIXED across seeds -> paired tests)


def gate_flags(cfg) -> Tuple[bool, bool, bool]:
    """(feature gates on, structural gates on, any gate learned under the budget).
    feature_gates / struct_gates default to use_gates, so older configs behave exactly as before."""
    fg = cfg.use_gates if cfg.feature_gates is None else bool(cfg.feature_gates)
    sg = cfg.use_gates if cfg.struct_gates is None else bool(cfg.struct_gates)
    learned = (fg and cfg.fixed_groups is None) or sg
    return fg, sg, learned


print(f'Project dir: {PROJECT_DIR} | device: {DEVICE} | torch {torch.__version__}')


### Cell 2 — Data loaders (HHAR, LMDHG, simulator), caching and feature lifting

In [ ]:
# ============================================================================
# CELL 2 — Data: HHAR (device + user heterogeneity), LMDHG (user heterogeneity),
#          and a heterogeneity simulator used only for smoke tests.
# ============================================================================
from numpy.lib.stride_tricks import sliding_window_view

HHAR_CLASSES = ['bike', 'sit', 'stand', 'walk', 'stairsup', 'stairsdown']


def _to_seconds(t: np.ndarray) -> np.ndarray:
    """HHAR timestamps come in different units per device; infer the unit so the
    median sampling interval lands in a physically plausible 2 ms – 200 ms range."""
    d = np.median(np.diff(np.sort(t)))
    for s in (1e-9, 1e-6, 1e-3, 1.0):
        if 0.002 <= d * s <= 0.2:
            return t.astype(np.float64) * s
    return t.astype(np.float64) * 1e-9


def _resample_stream(t, v, grid):
    """Linear interpolation of each axis onto a common grid + per-point gap size."""
    out = np.stack([np.interp(grid, t, v[:, k]) for k in range(v.shape[1])], 1)
    idx = np.clip(np.searchsorted(t, grid), 1, len(t) - 1)
    gap = np.minimum(np.abs(t[idx] - grid), np.abs(t[idx - 1] - grid))
    return out.astype(np.float32), gap


def _read_hhar_csv(path: str) -> pd.DataFrame:
    """Chunked, dtype-compact reader: peak RAM ~1/3 of a one-shot read of the 1.3 GB phone files."""
    cols = ['Creation_Time', 'x', 'y', 'z', 'User', 'Model', 'Device', 'gt']
    dt = {'Creation_Time': 'int64', 'x': 'float32', 'y': 'float32', 'z': 'float32',
          'User': 'category', 'Model': 'category', 'Device': 'category', 'gt': 'category'}
    parts = []
    for ch in pd.read_csv(path, usecols=cols, dtype=dt, chunksize=2_000_000, keep_default_na=False):
        parts.append(ch)
    df = pd.concat(parts, ignore_index=True)
    for c in ('User', 'Device', 'gt'):                     # unify categories across chunks
        df[c] = df[c].astype(str).astype('category')
    return df


def load_hhar(cfg: EPICConfig) -> Dict:
    """Per (user, device): sort, de-duplicate, resample acc + gyro to a common fs grid, window.
    Devices whose gyroscope is absent or covers < 50 % of the accelerometer span are kept as
    ACCELEROMETER-ONLY (gyro channels = 0, meta.has_gyro = False) instead of being dropped —
    sensor availability is itself a device heterogeneity. A per-device diagnostic table is printed."""
    pa = os.path.join(cfg.data_dir, f'{cfg.hhar_stream}_accelerometer.csv')
    pg = os.path.join(cfg.data_dir, f'{cfg.hhar_stream}_gyroscope.csv')
    print(f'Reading {pa}\n        {pg}  (chunked; a few minutes for the phone files)')
    acc = _read_hhar_csv(pa); gyr = _read_hhar_csv(pg)
    classes = HHAR_CLASSES + (['null'] if cfg.keep_null else [])
    cidx = {c: i for i, c in enumerate(classes)}
    W, S, fs = cfg.win, cfg.stride, cfg.fs
    Xs, ys, metas, diag = [], [], [], []
    gyr_groups = {k: v for k, v in gyr.groupby(['User', 'Device'], observed=True)}
    del gyr
    for (user, dev), a in acc.groupby(['User', 'Device'], observed=True):
        ta = _to_seconds(a['Creation_Time'].values); oa = np.argsort(ta); ta = ta[oa]
        va = a[['x', 'y', 'z']].values[oa]; la = a['gt'].astype(str).values[oa]
        ta, ua = np.unique(ta, return_index=True); va, la = va[ua], la[ua]
        g = gyr_groups.get((user, dev))
        has_gyro, tg, vg = False, None, None
        if g is not None and len(g) > 10:
            tg = _to_seconds(g['Creation_Time'].values); og = np.argsort(tg); tg = tg[og]
            vg = g[['x', 'y', 'z']].values[og]; tg, ug = np.unique(tg, return_index=True); vg = vg[ug]
            ov = min(ta[-1], tg[-1]) - max(ta[0], tg[0])
            has_gyro = ov > 0.5 * (ta[-1] - ta[0])
        if not has_gyro and not cfg.allow_missing_gyro:
            diag.append((str(user), str(dev), len(a), 0 if g is None else len(g), 0, 'dropped (no usable gyro)')); continue
        t0, t1 = (max(ta[0], tg[0]), min(ta[-1], tg[-1])) if has_gyro else (ta[0], ta[-1])
        if t1 - t0 < W / fs:
            diag.append((str(user), str(dev), len(a), 0 if g is None else len(g), 0, 'too short')); continue
        grid = np.arange(t0, t1, 1.0 / fs)
        A, gapa = _resample_stream(ta, va, grid)
        if has_gyro:
            G, gapg = _resample_stream(tg, vg, grid)
        else:
            G, gapg = np.zeros_like(A), np.zeros(len(grid))
        lab = la[np.clip(np.searchsorted(ta, grid), 0, len(ta) - 1)]
        lab_i = np.array([cidx.get(l, -1) for l in lab])
        bad = (gapa > cfg.max_gap_s) | (gapg > cfg.max_gap_s)
        sig = np.concatenate([A, G], 1)
        sw = sliding_window_view(sig, (W, 6))[::S, 0]; lw = sliding_window_view(lab_i, W)[::S]
        bw = sliding_window_view(bad, W)[::S].any(1)
        kept = 0
        for k in range(len(sw)):
            if bw[k]:
                continue
            vals, cnt = np.unique(lw[k], return_counts=True)
            j = cnt.argmax(); lbl, pur = vals[j], cnt[j] / W
            if lbl < 0 or pur < cfg.min_purity:
                continue
            Xs.append(sw[k].astype(np.float16)); ys.append(lbl); kept += 1
            metas.append((str(user), str(dev), str(dev).rsplit('_', 1)[0], pur < 1.0, float(pur), has_gyro))
        diag.append((str(user), str(dev), len(a), 0 if g is None else len(g), kept,
                     'acc+gyro' if has_gyro else 'ACC-ONLY'))
    del acc, gyr_groups
    X = np.stack(Xs); y = np.array(ys, dtype=np.int64)
    meta = pd.DataFrame(metas, columns=['user', 'device', 'model', 'boundary', 'purity', 'has_gyro'])
    d = pd.DataFrame(diag, columns=['user', 'device', 'acc_rows', 'gyro_rows', 'windows', 'mode'])
    print(d.groupby(['device', 'mode'])[['acc_rows', 'gyro_rows', 'windows']].sum().to_string())
    spec = dict(name='hhar', raw_names=['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z'],
                triads=[(0, 1, 2), (3, 4, 5)], triad_names=['acc', 'gyr'],
                add_mag=True, add_deriv=True, group_mode='per_channel',
                sensor_of_triad=['accelerometer', 'gyroscope'], optional_triads=[1],
                rot='3d', fs=fs, acc_sat=78.0)
    print(f'HHAR {cfg.hhar_stream}: {len(X)} windows | users {meta.user.nunique()} | '
          f'models {sorted(meta.model.unique())} | acc-only windows {int((~meta.has_gyro).sum())}')
    return dict(X=X, y=y, meta=meta, classes=classes, spec=spec)


def load_hhar_both(cfg: EPICConfig, cache_dir=None) -> Dict:
    """Phones + Watch in one dataset (meta.stream) for the cross-device protocols."""
    out = []
    for st in ('Phones', 'Watch'):
        c = copy.deepcopy(cfg); c.hhar_stream = st; c.dataset = 'hhar'
        d = load_dataset(c, cache_dir); d['meta'] = d['meta'].assign(stream=st); out.append(d)
    return dict(X=np.concatenate([o['X'] for o in out]), y=np.concatenate([o['y'] for o in out]),
                meta=pd.concat([o['meta'] for o in out], ignore_index=True),
                classes=out[0]['classes'], spec=out[0]['spec'])


# ---------------------------------------------------------------------------
# LMDHG: 50 unsegmented sequences, 46 joints (23 per hand) x 3D, 13 gestures,
# 21 subjects. The .mat layout differs between releases, so the loader inspects
# each file and prints what it found. CHECK THE PRINTOUT on the first run.
# ---------------------------------------------------------------------------
def _natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]


def _as_TxJ3(a):
    a = np.asarray(a)
    if a.dtype == object:                                   # cell array of frames, e.g. 13756 x (46, 3)
        frames = []
        for f in a.ravel():
            try:
                f = np.asarray(f, dtype=np.float32)
            except (ValueError, TypeError):
                return None                                 # not numeric (e.g. the labels cell array)
            frames.append(f if f.shape == (46, 3) else np.zeros((46, 3), np.float32))   # untracked frame -> zeros
        if not frames:
            return None
        a = np.stack(frames)
    a = a.astype(np.float32)
    if a.ndim == 2 and 138 in a.shape:
        return a if a.shape[1] == 138 else a.T
    if a.ndim == 3 and 46 in a.shape and 3 in a.shape:
        ja, ca = a.shape.index(46), a.shape.index(3)
        ta = [i for i in range(3) if i not in (ja, ca)][0]
        return np.transpose(a, (ta, ja, ca)).reshape(a.shape[ta], 138)
    return None


# Label names observed in the real LMDHG annotations.
LMDHG_IDLE = {'IDLE', 'idle', 'Idle', 'REPOS'}
LMDHG_EN = {'POINTER': 'point_to', 'ATTRAPER': 'catch', 'SECOUER_BAS': 'shake_down', 'SECOUER': 'shake',
            'C': 'draw_C', 'ZOOM': 'zoom', 'DEFILER_DOIGT': 'scroll', 'LIGNE': 'draw_line',
            'TRANCHER': 'slice', 'PIVOTER': 'rotate', 'ATTRAPER_MAIN_LEVEE': 'catch_raised_hand',
            'SECOUER_POING_LEVE': 'shake_raised_fist', 'POINTER_MAIN_LEVEE': 'point_raised_hand',
            'POINTER_PROLONGE': 'point_prolonged', 'CISEAUX': 'scissors'}
LMDHG_MERGE = {}   # e.g. {'POINTER_PROLONGE': 'POINTER_MAIN_LEVEE'}: ONLY if the dataset documentation confirms it


def _read_lmdhg_file(f):
    from scipy.io import loadmat
    m = loadmat(f, squeeze_me=True, struct_as_record=False)
    skel, ann, labs = None, None, None
    for k in [k for k in m if not k.startswith('__')]:
        v = m[k]
        if not isinstance(v, np.ndarray):
            continue
        if skel is None:
            cand = _as_TxJ3(v)
            if cand is not None and cand.shape[0] > 50:
                skel = cand; continue
        if v.ndim == 2 and v.shape[1] == 2 and np.issubdtype(v.dtype, np.number):
            ann = v.astype(int)
        elif v.dtype.kind in 'US' or (v.dtype == object and all(isinstance(e, str) for e in v.ravel()[:5])):
            labs = [str(x).strip() for x in np.ravel(v)]
    return skel, ann, labs


def load_lmdhg(cfg: EPICConfig) -> Dict:
    """lmdhg_mode='segmented': one sample per annotated gesture, time-normalised to lmdhg_T frames.
    lmdhg_mode='continuous': fixed-length sliding windows (win, stride) over the raw stream with
    IDLE as a class; window label = label of its CENTRE frame, purity kept in meta. Training
    uses only pure windows; evaluation uses ALL windows, i.e. frame-level accuracy sampled every
    `stride` frames — the unsegmented setting. Training on time-normalised segments but testing on
    fixed-length windows is a train/test mismatch, so continuous mode trains on windows too."""
    files = sorted(glob.glob(os.path.join(cfg.data_dir, '**', '*.mat'), recursive=True), key=_natural_key)
    if not files:
        raise FileNotFoundError(f'No .mat files under {cfg.data_dir}')
    subj = {}
    if cfg.lmdhg_subject_csv and os.path.exists(cfg.lmdhg_subject_csv):
        s = pd.read_csv(cfg.lmdhg_subject_csv); subj = dict(zip(s.iloc[:, 0].astype(str), s.iloc[:, 1].astype(str)))
    cont = cfg.lmdhg_mode == 'continuous'
    Xs, ys, metas, raw_counts = [], [], [], {}
    names = {'IDLE': 0} if (cont or cfg.keep_null) else {}
    def cls_of(lbl):
        if lbl in LMDHG_IDLE:
            return 'IDLE'
        return LMDHG_MERGE.get(lbl, lbl) if cfg.lmdhg_label_policy == 'merge' else lbl
    for fi, f in enumerate(files):
        skel, ann, labs = _read_lmdhg_file(f)
        if fi == 0:
            print(f'[LMDHG] {os.path.basename(f)} skeleton={None if skel is None else skel.shape} '
                  f'annotations={None if ann is None else ann.shape} labels={labs[:4] if labs else None}')
        if skel is None or ann is None or labs is None or len(labs) != len(ann):
            print(f'  ! skipped {os.path.basename(f)} — run inspect_lmdhg() and send me the output'); continue
        stem = os.path.splitext(os.path.basename(f))[0]
        user = subj.get(stem, stem)
        for l in labs:
            raw_counts[l] = raw_counts.get(l, 0) + 1
        if not cont:
            for (s0, s1), l in zip(ann, labs):
                c = cls_of(l)
                if c == 'IDLE' and not cfg.keep_null:
                    continue
                s0, s1 = max(int(s0) - 1, 0), min(int(s1), len(skel))
                if s1 - s0 < 5:
                    continue
                seg = skel[s0:s1]; tt = np.linspace(0, len(seg) - 1, cfg.lmdhg_T)
                seg = np.stack([np.interp(tt, np.arange(len(seg)), seg[:, c_]) for c_ in range(138)], 1)
                names.setdefault(c, len(names))
                Xs.append(seg.astype(np.float32)); ys.append(names[c])
                metas.append((user, 'leap', 'leap', False, 1.0, fi, s0))
        else:
            fl = np.zeros(len(skel), dtype=np.int64)            # frame labels; unannotated frames = IDLE
            for (s0, s1), l in zip(ann, labs):
                c = cls_of(l); names.setdefault(c, len(names))
                fl[max(int(s0) - 1, 0):min(int(s1), len(skel))] = names[c]
            W, S = cfg.win, cfg.stride
            if len(skel) < W:
                continue
            sw = sliding_window_view(skel, (W, 138))[::S, 0]; lw = sliding_window_view(fl, W)[::S]
            lab = lw[:, W // 2]; pur = (lw == lab[:, None]).mean(1)
            Xs.append(sw.astype(np.float16))                    # whole file at once, float16: ~2.3 GB for all of LMDHG
            ys.extend(lab.tolist())
            metas.extend((user, 'leap', 'leap', bool(p_ < 1.0), float(p_), fi, k * S) for k, p_ in enumerate(pur))
    print(f'  raw annotation counts: {dict(sorted(raw_counts.items(), key=lambda t: -t[1]))}')
    if not Xs:
        raise RuntimeError('LMDHG loader found no samples — run inspect_lmdhg() and adapt _read_lmdhg_file.')
    if not subj:
        print('  ! No subject map: folds are split by sequence file. Subjects with several sequences can leak — '
              'supply lmdhg_subject_csv (file,subject) from ComplementaryInfo_LMDHG.')
    classes = sorted(names, key=names.get)
    joints = [f'{h}{j:02d}' for h in ('L', 'R') for j in range(23)]
    spec = dict(name='lmdhg', raw_names=[f'{j}_{a}' for j in joints for a in 'xyz'],
                triads=[(3 * j, 3 * j + 1, 3 * j + 2) for j in range(46)], triad_names=joints,
                add_mag=False, add_deriv=True, group_mode='per_triad',
                sensor_of_triad=['left_hand'] * 23 + ['right_hand'] * 23, rot='vertical',
                fs=float(cfg.lmdhg_fps), fov_mm=600.0)
    meta = pd.DataFrame(metas, columns=['user', 'device', 'model', 'boundary', 'purity', 'seq', 'start'])
    y = np.array(ys)
    print(f'LMDHG ({cfg.lmdhg_mode}): {len(Xs)} samples | {len(classes)} classes | groups {meta.user.nunique()} | '
          f'majority-class share {np.bincount(y).max() / len(y):.3f}')
    X = np.concatenate(Xs) if cont else np.stack(Xs).astype(np.float16)
    del Xs
    return dict(X=X, y=y, meta=meta, classes=[LMDHG_EN.get(c, c) for c in classes], spec=spec)


def simulate_heterogeneous_har(n_users=9, models=('nexus4', 's3', 's3mini', 'samsungold'),
                               per=40, win=128, fs=50.0, seed=0) -> Dict:
    """SMOKE-TEST ONLY. Synthetic IMU windows with user style, device gain/bias/noise,
    sampling-rate mismatch and orientation. Numbers obtained on it mean nothing."""
    rng = np.random.default_rng(seed)
    f0 = np.array([1.2, 0.0, 0.0, 1.8, 1.5, 2.0]); amp = np.array([3., .05, .1, 4., 5., 6.])
    Xs, ys, metas = [], [], []
    for u in range(n_users):
        ustyle = rng.uniform(0.8, 1.25)
        for m_i, m in enumerate(models):
            dev_rate = fs * rng.uniform(0.7, 1.6); gain = rng.uniform(.85, 1.15, 6)
            bias = rng.normal(0, .3, 6); noise = rng.uniform(.02, .2)
            for d in (1, 2):
                for c in range(6):
                    for _ in range(per // 2):
                        t = np.arange(int(win * dev_rate / fs)) / dev_rate
                        ph = rng.uniform(0, 2 * np.pi)
                        base = amp[c] * np.sin(2 * np.pi * f0[c] * ustyle * t[:, None] + ph + np.arange(6) * .7)
                        grav = np.array([0, 0, 9.81]) if c != 1 else np.array([0, 9.81, 0])
                        sig = base.copy(); sig[:, :3] += grav
                        th = rng.uniform(-.5, .5); R = np.array([[np.cos(th), -np.sin(th), 0], [np.sin(th), np.cos(th), 0], [0, 0, 1]])
                        sig[:, :3] = sig[:, :3] @ R.T; sig[:, 3:] = sig[:, 3:] @ R.T
                        sig = sig * gain + bias + rng.normal(0, noise, sig.shape)
                        tt = np.linspace(0, t[-1], win)
                        sig = np.stack([np.interp(tt, t, sig[:, k]) for k in range(6)], 1)
                        Xs.append(sig.astype(np.float32)); ys.append(c)
                        metas.append((chr(97 + u), f'{m}_{d}', m, bool(rng.random() < .1), 1.0, True))
    spec = dict(name='synthetic', raw_names=['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z'],
                triads=[(0, 1, 2), (3, 4, 5)], triad_names=['acc', 'gyr'], add_mag=True, add_deriv=True,
                group_mode='per_channel', sensor_of_triad=['accelerometer', 'gyroscope'], optional_triads=[1], rot='3d', fs=fs, acc_sat=78.0)
    return dict(X=np.stack(Xs), y=np.array(ys), meta=pd.DataFrame(metas, columns=['user', 'device', 'model', 'boundary', 'purity', 'has_gyro']),
                classes=HHAR_CLASSES, spec=spec)


def _cache_key(cfg: EPICConfig) -> str:
    if cfg.dataset == 'hhar':
        k = f'hhar_v3_{cfg.hhar_stream}_fs{cfg.fs:g}_w{cfg.win}_s{cfg.stride}_p{cfg.min_purity}_n{int(cfg.keep_null)}_g{int(cfg.allow_missing_gyro)}'
    else:
        k = (f'lmdhg_{cfg.lmdhg_mode}_T{cfg.lmdhg_T}_w{cfg.win}_s{cfg.stride}_n{int(cfg.keep_null)}_'
             f'{cfg.lmdhg_label_policy}{len(LMDHG_MERGE)}')
    return k


def load_dataset(cfg: EPICConfig, cache_dir: Optional[str] = None) -> Dict:
    """Parses once, then caches windows (float16 on disk -> half the Drive space) to cache_dir."""
    if cfg.dataset == 'synthetic':
        print('⚠ Using SIMULATED data — pipeline test only, not a result.')
        return simulate_heterogeneous_har(win=cfg.win, fs=cfg.fs, seed=cfg.seed)
    path = os.path.join(cache_dir, _cache_key(cfg)) if cache_dir else None
    if path and os.path.exists(path + '.npz'):
        z = np.load(path + '.npz', allow_pickle=False)
        side = json.load(open(path + '.json'))
        data = dict(X=z['X'], y=z['y'], meta=pd.read_csv(path + '_meta.csv', dtype={'user': str}),
                    classes=side['classes'], spec=side['spec'])
        data['spec']['triads'] = [tuple(t) for t in data['spec']['triads']]
        if 'has_gyro' in data['meta']:
            data['meta']['has_gyro'] = data['meta']['has_gyro'].astype(bool)
        print(f'Loaded cached {os.path.basename(path)}: {data["X"].shape}')
        return data
    data = load_hhar(cfg) if cfg.dataset == 'hhar' else load_lmdhg(cfg)
    if path and data['X'].nbytes > 1e9:
        print(f'Not caching ({data["X"].nbytes / 1e9:.1f} GB): re-parsing the .mat files is faster than Drive I/O')
    elif path:
        np.savez(path + '.npz', X=data['X'].astype(np.float16, copy=False), y=data['y'])
        data['meta'].to_csv(path + '_meta.csv', index=False)
        json.dump(dict(classes=data['classes'], spec=data['spec']), open(path + '.json', 'w'))
        print(f'Cached to {path}.npz ({os.path.getsize(path + ".npz") / 1e6:.0f} MB)')
    return data


# ---------------------------------------------------------------------------
# Channel lifting (done inside the model so the deployed artefact is self-contained)
# and the channel -> feature-group map that the input gates act on.
# ---------------------------------------------------------------------------
def build_channel_groups(spec: Dict):
    names, groups, gnames, gsensor = list(spec['raw_names']), [], [], []
    tri_of = {c: i for i, tr in enumerate(spec['triads']) for c in tr}
    if spec['group_mode'] == 'per_channel':
        for c, n in enumerate(spec['raw_names']):
            groups.append(len(gnames)); gnames.append(n); gsensor.append(spec['sensor_of_triad'][tri_of[c]])
    else:
        for c in range(len(spec['raw_names'])):
            groups.append(tri_of[c])
        gnames = list(spec['triad_names']); gsensor = list(spec['sensor_of_triad'])
    for i, tn in enumerate(spec['triad_names']):
        for kind, flag in (('mag', spec['add_mag']), ('deriv', spec['add_deriv'])):
            if not flag:
                continue
            names.append(f'{tn}_{kind}')
            if spec['group_mode'] == 'per_channel':
                groups.append(len(gnames)); gnames.append(f'{tn}_{kind}'); gsensor.append(spec['sensor_of_triad'][i])
            else:
                groups.append(i)
    return names, groups, gnames, gsensor


class FeatureLift(nn.Module):
    """raw [B,T,C_raw] -> [raw, |triad|, |d triad/dt|] — magnitudes are orientation-invariant,
    which directly targets device-placement heterogeneity (Stisen et al., SenSys'15)."""
    def __init__(self, spec):
        super().__init__()
        self.register_buffer('tri', torch.tensor(spec['triads'], dtype=torch.long))
        self.add_mag, self.add_deriv, self.fs = spec['add_mag'], spec['add_deriv'], float(spec['fs'])

    def forward(self, x):
        v = x[..., self.tri]                                         # [B,T,K,3]
        feats = [x]
        mag = v.norm(dim=-1) if self.add_mag else None
        # first difference written with slicing (== torch.diff with prepend) so it exports to ONNX / mobile
        der = (torch.cat([torch.zeros_like(v[:, :1]), v[:, 1:] - v[:, :-1]], 1) * self.fs).norm(dim=-1) if self.add_deriv else None
        per = [t for t in (mag, der) if t is not None]
        if per:                                                      # interleave per triad: mag, deriv
            feats.append(torch.stack(per, -1).flatten(2))
        return torch.cat(feats, -1)


### Cell 2b — Get the data: HHAR from UCI, LMDHG from your Drive

In [ ]:
# ============================================================================
# CELL 1b — Data acquisition (run once per Colab session)
#   * HHAR: downloaded from UCI into the fast local disk (/content)
#   * LMDHG: copied from Drive to /content (Drive FUSE reads are slow file-by-file).
# ============================================================================
import zipfile, shutil, urllib.request

HHAR_URL = 'https://archive.ics.uci.edu/static/public/344/heterogeneity+activity+recognition.zip'
HHAR_LOCAL = os.environ.get('EPIC_HHAR_DIR') or ('/content/hhar' if IN_COLAB else os.path.abspath('./hhar'))
# LMDHG location in your Drive: a folder of .mat files OR a .zip. The first one that exists is used.
LMDHG_CANDIDATES = ['/content/drive/MyDrive/GMN_Datasets/LMDHG',
                    '/content/drive/MyDrive/GMN_Datasets/LMDHG.zip',
                    f'{PROJECT_DIR}/data/lmdhg']
LMDHG_DRIVE = next((p for p in LMDHG_CANDIDATES if os.path.exists(p)), LMDHG_CANDIDATES[0])
LMDHG_LOCAL = '/content/LMDHG' if IN_COLAB else os.path.abspath('./LMDHG')
CACHE_DIR = f'{PROJECT_DIR}/cache'; os.makedirs(CACHE_DIR, exist_ok=True)
HHAR_FILES = ['Phones_accelerometer.csv', 'Phones_gyroscope.csv', 'Watch_accelerometer.csv', 'Watch_gyroscope.csv']


def _progress(block, bsize, total):
    if total > 0 and block % 200 == 0:
        print(f'\r  downloaded {block * bsize / 1e6:7.1f} / {total / 1e6:.1f} MB', end='')


def _extract_wanted(zf: zipfile.ZipFile, dest: str, wanted: List[str]) -> List[str]:
    got = []
    for n in zf.namelist():
        if os.path.basename(n) in wanted and not n.startswith('__MACOSX'):
            with zf.open(n) as src, open(os.path.join(dest, os.path.basename(n)), 'wb') as dst:
                shutil.copyfileobj(src, dst, length=16 << 20)
            got.append(os.path.basename(n))
    return got


def fetch_hhar(dest: str = HHAR_LOCAL, wanted: List[str] = HHAR_FILES) -> str:
    os.makedirs(dest, exist_ok=True)
    if all(os.path.exists(os.path.join(dest, f)) for f in wanted):
        print(f'HHAR already present in {dest}'); return dest
    outer = os.path.join(dest, 'hhar_uci.zip')
    if not os.path.exists(outer):
        print(f'Downloading HHAR from UCI ...'); urllib.request.urlretrieve(HHAR_URL, outer, _progress); print()
    with zipfile.ZipFile(outer) as z:
        got = _extract_wanted(z, dest, wanted)
        inner = [n for n in z.namelist() if n.lower().endswith('.zip') and 'activity' in n.lower()]
        for n in inner:                                   # the CSVs sit inside "Activity recognition exp.zip"
            ip = os.path.join(dest, 'inner.zip')
            with z.open(n) as src, open(ip, 'wb') as dst:
                shutil.copyfileobj(src, dst, length=16 << 20)
            with zipfile.ZipFile(ip) as zi:
                got += _extract_wanted(zi, dest, wanted)
            os.remove(ip)
    os.remove(outer)                                      # free local disk
    missing = [f for f in wanted if f not in got and not os.path.exists(os.path.join(dest, f))]
    if missing:
        raise FileNotFoundError(f'Not found in the UCI archive: {missing}. Download manually from '
                                'https://archive.ics.uci.edu/dataset/344 and put the CSVs in ' + dest)
    print('HHAR files:', {f: f'{os.path.getsize(os.path.join(dest, f)) / 1e6:.0f} MB' for f in wanted})
    return dest


def copy_lmdhg_from_drive(src: str = LMDHG_DRIVE, dst: str = LMDHG_LOCAL) -> str:
    """src may be a folder (with .mat files or .zip archives inside) or a single .zip file."""
    def n_mat(d):
        return len(glob.glob(os.path.join(d, '**', '*.mat'), recursive=True))
    if os.path.isdir(dst) and n_mat(dst) > 0:
        print(f'LMDHG: {n_mat(dst)} .mat files already in {dst}'); return dst
    if not os.path.exists(src):
        raise FileNotFoundError(f'{src} not found — set LMDHG_DRIVE to the LMDHG folder or .zip in your Drive '
                                f'(tried: {LMDHG_CANDIDATES})')
    os.makedirs(dst, exist_ok=True)
    if os.path.isfile(src) and src.lower().endswith('.zip'):
        zips = [src]
    elif n_mat(src) > 0:
        shutil.copytree(src, dst, dirs_exist_ok=True); zips = []
    else:
        zips = glob.glob(os.path.join(src, '**', '*.zip'), recursive=True)
    for zp in zips:
        print(f'Extracting {zp} -> {dst}')
        with zipfile.ZipFile(zp) as z:
            z.extractall(dst)
    for zp in glob.glob(os.path.join(dst, '**', '*.zip'), recursive=True):   # zip inside the zip
        with zipfile.ZipFile(zp) as z:
            z.extractall(os.path.dirname(zp))
    n = n_mat(dst)
    if n == 0:
        raise FileNotFoundError(f'No .mat files found after copying/extracting {src}. Contents of {dst}: '
                                f'{os.listdir(dst)[:20]}')
    print(f'LMDHG: {n} .mat files in {dst}')
    return dst


def inspect_lmdhg(path: str = LMDHG_LOCAL, n_files: int = 1):
    """Print the structure of the first LMDHG file(s). Paste this output to adapt the loader."""
    from scipy.io import loadmat
    files = sorted(glob.glob(os.path.join(path, '**', '*.mat'), recursive=True), key=_natural_key)
    for f in files[:n_files]:
        m = loadmat(f, squeeze_me=True, struct_as_record=False)
        print(f'== {os.path.relpath(f, path)}')
        for k, v in m.items():
            if k.startswith('__'):
                continue
            a = np.asarray(v)
            ex = a.ravel()[:4] if a.dtype != object else [e if isinstance(e, str) else f'{type(e).__name__}{np.shape(e)}' for e in a.ravel()[:4]]
            print(f'   {k:<14} dtype={a.dtype} shape={a.shape} first={ex}')


In [ ]:
# Check the LMDHG file structure (send me this output if the loader skips files)
copy_lmdhg_from_drive(LMDHG_DRIVE, LMDHG_LOCAL)
inspect_lmdhg(LMDHG_LOCAL, n_files=1)


### Cell 3 — EPIC-Net: coupled feature and weight gating, prototype head, exact pruned export

In [ ]:
# ============================================================================
# CELL 3 — EPIC-Net: one budget couples input-feature selection (Pillar A)
#          with structural weight selection in the backbone (Pillar B).
# ============================================================================
class HardConcrete(nn.Module):
    def __init__(self, n: int, init_open: float = 0.9, beta=2 / 3, gamma=-0.1, zeta=1.1):
        super().__init__()
        self.beta, self.gamma, self.zeta = beta, gamma, zeta
        la = math.log(init_open / (1 - init_open)) + beta * math.log(-gamma / zeta)
        self.log_alpha = nn.Parameter(torch.full((n,), la))
        self.active = True

    def forward(self):
        if not self.active:
            return torch.ones_like(self.log_alpha)
        if self.training:
            u = torch.rand_like(self.log_alpha).clamp(1e-6, 1 - 1e-6)
            s = torch.sigmoid((torch.log(u) - torch.log1p(-u) + self.log_alpha) / self.beta)
        else:
            s = torch.sigmoid(self.log_alpha)
        return (s * (self.zeta - self.gamma) + self.gamma).clamp(0, 1)

    def p_open(self):
        if not self.active:
            return torch.ones_like(self.log_alpha)
        return torch.sigmoid(self.log_alpha - self.beta * math.log(-self.gamma / self.zeta))

    def hard_open(self):
        with torch.no_grad():
            was = self.training; self.eval(); g = self.forward(); self.train(was)
        return (g > 0).float()


def all_gates(module):
    return [m for m in module.modules() if isinstance(m, HardConcrete)]


class HeteroNorm(nn.Module):
    """Global standardisation (statistics fitted on the TRAINING fold only — the original
    notebook fitted a StandardScaler on all data, leaking test statistics) followed by a
    *learned partial* instance centring: x - sigmoid(kappa) * mean_t(x).
    kappa lets the model decide per channel how much device offset / gravity
    orientation to remove, instead of RevIN's all-or-nothing normalisation, which
    erases the amplitude cues that separate e.g. sit from walk."""
    def __init__(self, C):
        super().__init__()
        self.register_buffer('mu', torch.zeros(C)); self.register_buffer('sd', torch.ones(C))
        self.kappa = nn.Parameter(torch.zeros(C))
        self.a = nn.Parameter(torch.ones(C)); self.b = nn.Parameter(torch.zeros(C))

    def forward(self, x):
        x = (x - self.mu) / self.sd
        x = x - torch.sigmoid(self.kappa) * x.mean(1, keepdim=True)
        return x * self.a + self.b


class PatchTokenizer(nn.Module):
    def __init__(self, C, P, S, d, max_tokens=512):
        super().__init__()
        self.C, self.P, self.S = C, P, S
        self.proj = nn.Linear(C * P, d)
        self.pos = nn.Parameter(torch.zeros(1, max_tokens, d)); nn.init.trunc_normal_(self.pos, std=.02)

    def forward(self, x):                                   # [B,T,C]
        p = x.unfold(1, self.P, self.S)                     # [B,N,C,P]  (channel-major -> sliceable)
        B, N = p.shape[:2]
        return self.proj(p.reshape(B, N, -1)) + self.pos[:, :N]


class GatedMHA(nn.Module):
    def __init__(self, d, n_heads, head_dim, drop=0.0, gated=True):
        super().__init__()
        self.H, self.dh, self.drop = n_heads, head_dim, drop
        self.qkv = nn.Linear(d, 3 * n_heads * head_dim)
        self.out = nn.Linear(n_heads * head_dim, d)
        self.gate = HardConcrete(n_heads) if gated else None

    def forward(self, x):
        B, N, _ = x.shape
        q, k, v = self.qkv(x).view(B, N, 3, self.H, self.dh).permute(2, 0, 3, 1, 4)
        o = F.scaled_dot_product_attention(q, k, v, dropout_p=self.drop if self.training else 0.0)
        if self.gate is not None:
            o = o * self.gate().view(1, -1, 1, 1)
        return self.out(o.transpose(1, 2).reshape(B, N, self.H * self.dh))

    def cost(self, N, d, heads):
        return heads * (3 * N * d * self.dh + 2 * N * N * self.dh + N * self.dh * d)


class GatedMLP(nn.Module):
    def __init__(self, d, m, drop=0.0, gated=True):
        super().__init__()
        self.fc1, self.fc2, self.dp = nn.Linear(d, m), nn.Linear(m, d), nn.Dropout(drop)
        self.gate = HardConcrete(m) if gated else None

    def forward(self, x):
        h = F.gelu(self.fc1(x))
        if self.gate is not None:
            h = h * self.gate()
        return self.fc2(self.dp(h))


class ConstBranch(nn.Module):           # a branch whose every unit was pruned leaves only its bias
    def __init__(self, bias):
        super().__init__(); self.register_buffer('bias', bias.detach().clone())

    def forward(self, x):
        return self.bias.expand_as(x)


class GatedBlock(nn.Module):
    def __init__(self, d, H, mlp_ratio, drop):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = GatedMHA(d, H, d // H, drop)
        self.mlp = GatedMLP(d, int(d * mlp_ratio), drop)
        self.layer_gate = HardConcrete(1)
        self.dp = nn.Dropout(drop)

    def forward(self, x):
        lg = self.layer_gate() if self.layer_gate is not None else 1.0
        x = x + lg * self.dp(self.attn(self.ln1(x)))
        return x + lg * self.dp(self.mlp(self.ln2(x)))


class GatedEncoder(nn.Module):
    def __init__(self, cfg: EPICConfig):
        super().__init__()
        self.d = cfg.d_model
        self.blocks = nn.ModuleList([GatedBlock(cfg.d_model, cfg.n_heads, cfg.mlp_ratio, cfg.dropout)
                                     for _ in range(cfg.n_layers)])
        self.ln = nn.LayerNorm(cfg.d_model)

    def forward(self, x):
        for b in self.blocks:
            x = b(x)
        return self.ln(x)

    def cost(self, N, mode='expected'):
        """mode: 'expected' (differentiable, from P(gate open)), 'hard' (eval gates), 'dense'."""
        tot = 0.0
        for b in self.blocks:
            def stat(g, n):
                if mode == 'dense' or g is None:
                    return torch.tensor(float(n))
                return g.p_open().sum() if mode == 'expected' else g.hard_open().sum()
            heads = stat(b.attn.gate, b.attn.H) if isinstance(b.attn, GatedMHA) else torch.tensor(0.)
            neur = stat(b.mlp.gate, b.mlp.fc1.out_features) if isinstance(b.mlp, GatedMLP) else torch.tensor(0.)
            pl = stat(b.layer_gate, 1)
            a = b.attn.cost(N, self.d, heads) if isinstance(b.attn, GatedMHA) else 0.0
            tot = tot + pl * (a + 2 * N * self.d * neur)
        return tot


def _find_block_list(model):
    best = None
    for name, mod in model.named_modules():
        if isinstance(mod, nn.ModuleList) and (best is None or len(mod) > len(best[1])):
            best = (name, mod)
    return best


class HFEncoder(nn.Module):
    """Pillar B with a *pretrained* language model (One-Fits-All / Time-LLM style):
    frozen, truncated LM, LoRA adapters, and Hard-Concrete DEPTH gates that decide how
    many blocks are worth their cost. Kept as an ablation arm on purpose: Tan et al.
    (NeurIPS'24) report that removing the LLM often does not hurt time-series models,
    so EPIC-HAR treats 'does the LLM earn its FLOPs?' as a question to measure."""
    def __init__(self, cfg: EPICConfig):
        super().__init__()
        from transformers import AutoModel
        self.lm = AutoModel.from_pretrained(cfg.llm_name)
        name, blocks = _find_block_list(self.lm)
        k = min(cfg.llm_layers, len(blocks))
        parent = self.lm.get_submodule(name.rsplit('.', 1)[0]) if '.' in name else self.lm
        setattr(parent, name.rsplit('.', 1)[-1], nn.ModuleList(list(blocks)[:k]))
        for attr in ('n_layer', 'num_hidden_layers'):
            if hasattr(self.lm.config, attr):
                setattr(self.lm.config, attr, k)
        if hasattr(self.lm.config, 'layer_types') and self.lm.config.layer_types:
            self.lm.config.layer_types = self.lm.config.layer_types[:k]
        for p in self.lm.parameters():
            p.requires_grad = False
        try:
            from peft import LoraConfig, get_peft_model
            tm = ['c_attn'] if 'gpt2' in self.lm.config.model_type else ['q_proj', 'v_proj']
            self.lm = get_peft_model(self.lm, LoraConfig(r=cfg.lora_r, lora_alpha=2 * cfg.lora_r,
                                                         target_modules=tm, lora_dropout=0.05))
        except Exception as e:                               # fall back to LayerNorm tuning (OFA)
            print(f'[HFEncoder] PEFT unavailable ({e}); tuning LayerNorms only')
            for n, p in self.lm.named_parameters():
                if 'ln' in n or 'norm' in n:
                    p.requires_grad = True
        self.L, self.hidden = k, self.lm.config.hidden_size
        self.inp, self.out = nn.Linear(cfg.d_model, self.hidden), nn.Linear(self.hidden, cfg.d_model)
        self.depth_gate = HardConcrete(k)

    def forward(self, t):
        hs = self.lm(inputs_embeds=self.inp(t), output_hidden_states=True).hidden_states[1:]
        g = self.depth_gate()
        h = sum(g[l] * hs[l] for l in range(self.L)) / (g.sum() + 1e-6)
        return self.out(h)

    def cost(self, N, mode='expected'):
        per = N * 12 * self.hidden ** 2 + 2 * N * N * self.hidden
        if mode == 'dense':
            return torch.tensor(float(self.L * per))
        p = self.depth_gate.p_open() if mode == 'expected' else self.depth_gate.hard_open()
        # expected depth = sum_l P(some gate at depth >= l is open)  (deeper blocks must still run)
        depth = sum(1 - torch.prod(1 - p[l:]) for l in range(self.L))
        return depth * per


class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__(); self.s = nn.Linear(d, 1)

    def forward(self, h):
        w = torch.softmax(self.s(h), 1)
        return (w * h).sum(1), w.squeeze(-1)


class PrototypeHead(nn.Module):
    def __init__(self, e, K, M):
        super().__init__()
        self.K, self.M = K, M
        self.protos = nn.Parameter(torch.randn(K * M, e) * 0.1)
        self.log_scale = nn.Parameter(torch.tensor(math.log(10.0)))
        self.register_buffer('proto_src', torch.full((K * M,), -1, dtype=torch.long))  # training window index

    def forward(self, z):
        sim = F.normalize(z, dim=-1) @ F.normalize(self.protos, dim=-1).T          # [B, K*M]
        logits = torch.logsumexp(self.log_scale.exp() * sim.view(-1, self.K, self.M), -1)
        return logits, sim


class _GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, l):
        ctx.l = l; return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return -ctx.l * g, None


class EPICNet(nn.Module):
    def __init__(self, cfg: EPICConfig, spec: Dict, n_classes: int, n_users: int, n_devices: int, T: int):
        super().__init__()
        self.cfg, self.spec = cfg, spec
        names, groups, gnames, gsensor = build_channel_groups(spec)
        self.channel_names, self.group_names, self.group_sensor = names, gnames, gsensor
        C = len(names)
        self.lift = FeatureLift(spec)
        self.register_buffer('ch2grp', torch.tensor(groups))
        self.register_buffer('in_idx', torch.arange(C))       # channels kept after export
        self.in_gate = HardConcrete(len(gnames))
        self.norm = HeteroNorm(C)
        self.tok = PatchTokenizer(C, cfg.patch, cfg.patch_stride, cfg.d_model)
        self.N = (T - cfg.patch) // cfg.patch_stride + 1
        self.encoder = GatedEncoder(cfg) if cfg.backbone == 'light' else HFEncoder(cfg)
        self.pool = AttnPool(cfg.d_model)
        self.proj = nn.Sequential(nn.LayerNorm(cfg.d_model), nn.Linear(cfg.d_model, cfg.emb_dim))
        self.head = PrototypeHead(cfg.emb_dim, n_classes, cfg.protos_per_class)
        self.adv_user = nn.Sequential(nn.Linear(cfg.emb_dim, 64), nn.ReLU(), nn.Linear(64, max(n_users, 2)))
        self.adv_dev = nn.Sequential(nn.Linear(cfg.emb_dim, 64), nn.ReLU(), nn.Linear(64, max(n_devices, 2)))
        fg, sg, _ = gate_flags(cfg)
        if not sg:                                            # pillar B off: dense heads / neurons / layers
            for g in all_gates(self.encoder):
                g.active = False
        if cfg.fixed_groups is not None:                      # fixed, non-learned subset (random / filter controls)
            keep = torch.tensor([n in set(cfg.fixed_groups) for n in gnames])
            with torch.no_grad():
                self.in_gate.log_alpha.copy_(torch.where(keep, 12.0, -12.0))
            self.in_gate.log_alpha.requires_grad_(False)
        elif not fg:                                          # pillar A off: every feature group kept
            self.in_gate.active = False

    def features(self, x_raw):
        return self.lift(x_raw.float())[..., self.in_idx]          # windows are stored as float16 in RAM

    def forward(self, x_raw, group_mask=None, grl=None):
        h = self.norm(self.features(x_raw))
        g = self.in_gate()[self.ch2grp]
        if group_mask is not None:                            # occlusion of feature groups (explanations)
            gm = group_mask if group_mask.dim() == 2 else group_mask.unsqueeze(0)
            g = g * gm[:, self.ch2grp].unsqueeze(1)
        h = h * g
        t = self.encoder(self.tok(h))
        pooled, attn = self.pool(t)
        z = self.proj(pooled)
        logits, sim = self.head(z)
        out = dict(logits=logits, z=z, sim=sim, attn=attn)
        if grl is not None:
            zr = _GRL.apply(z, grl)
            out['adv_user'], out['adv_dev'] = self.adv_user(zr), self.adv_dev(zr)
        return out

    def cost(self, mode='expected'):
        cpg = torch.bincount(self.ch2grp, minlength=len(self.group_names)).float().to(self.in_gate.log_alpha.device)
        if mode == 'dense':
            ch = cpg.sum()
        else:
            p = self.in_gate.p_open() if mode == 'expected' else self.in_gate.hard_open()
            ch = (p * cpg).sum()
        tok = self.N * self.cfg.d_model * self.cfg.patch * ch
        return tok + self.encoder.cost(self.N, mode)

    def cost_ratio(self, mode='expected'):
        return self.cost(mode) / self.cost('dense').detach()

    def input_cost_ratio(self, mode='expected'):
        cpg = torch.bincount(self.ch2grp, minlength=len(self.group_names)).float().to(self.in_gate.log_alpha.device)
        p = self.in_gate.p_open() if mode == 'expected' else self.in_gate.hard_open()
        return (p * cpg).sum() / cpg.sum()

    def budget_ratio(self, mode='expected'):
        """The quantity held to the budget. With both gate families on it is the FLOPs ratio of the whole
        network. With only the feature gates on it is the share of input channels kept, so pillar A alone
        is held to the same budget instead of chasing a FLOPs target it cannot reach."""
        fg, sg, _ = gate_flags(self.cfg)
        if fg and not sg and self.cfg.fixed_groups is None:
            return self.input_cost_ratio(mode)
        return self.cost_ratio(mode)

    def gate_report(self) -> Dict:
        g = self.in_gate.hard_open().cpu().numpy()
        kept = [n for n, k in zip(self.group_names, g) if k > 0]
        sensors = sorted({s for s, k in zip(self.group_sensor, g) if k > 0})
        rep = dict(feature_groups_kept=kept, n_groups_kept=int(g.sum()), n_groups=len(g),
                   sensors_needed=sensors, sensors_all=sorted(set(self.group_sensor)))
        if isinstance(self.encoder, GatedEncoder):
            rep['layers'] = [dict(on=int(b.layer_gate.hard_open().item()),
                                  heads=int(b.attn.gate.hard_open().sum()), neurons=int(b.mlp.gate.hard_open().sum()))
                             for b in self.encoder.blocks if b.layer_gate is not None]
        else:
            rep['llm_depth_gates'] = self.encoder.depth_gate.hard_open().int().tolist()
        rep['hard_cost_ratio'] = float(self.cost_ratio('hard'))
        return rep


@torch.no_grad()
def export_pruned(model: EPICNet) -> EPICNet:
    """Fold every eval-mode gate into the weights and physically remove pruned
    channels / heads / neurons / layers. Output equals the gated model exactly."""
    model = model.eval()
    m = copy.deepcopy(model).cpu().eval()
    gin = m.in_gate()[m.ch2grp]                                  # per-channel gate values
    keep = (gin > 0).nonzero().squeeze(1)
    if len(keep) == 0:
        keep = gin.argmax().view(1)
    nm = m.norm
    m.norm = HeteroNorm(len(keep))
    m.norm.mu.copy_(nm.mu[keep]); m.norm.sd.copy_(nm.sd[keep]); m.norm.kappa.copy_(nm.kappa[keep])
    m.norm.a.copy_((nm.a * gin)[keep]); m.norm.b.copy_((nm.b * gin)[keep])
    P = m.tok.P; W = m.tok.proj.weight.view(-1, m.tok.C, P)
    newtok = PatchTokenizer(len(keep), P, m.tok.S, W.shape[0], m.tok.pos.shape[1])
    newtok.proj.weight.copy_(W[:, keep].reshape(W.shape[0], -1)); newtok.proj.bias.copy_(m.tok.proj.bias)
    newtok.pos.copy_(m.tok.pos); m.tok = newtok
    m.in_idx = m.in_idx[keep]; m.ch2grp = m.ch2grp[keep]
    m.in_gate.active = False
    if isinstance(m.encoder, GatedEncoder):
        blocks = []
        for b in m.encoder.blocks:
            lg = float(b.layer_gate())
            if lg == 0:
                continue
            hg = b.attn.gate(); kh = (hg > 0).nonzero().squeeze(1)
            d, dh = b.attn.out.out_features, b.attn.dh
            if len(kh) == 0:
                b.attn = ConstBranch(b.attn.out.bias * lg)
            else:
                na = GatedMHA(d, len(kh), dh, b.attn.drop, gated=False)
                qw = b.attn.qkv.weight.view(3, b.attn.H, dh, d)[:, kh].reshape(-1, d)
                qb = b.attn.qkv.bias.view(3, b.attn.H, dh)[:, kh].reshape(-1)
                ow = (b.attn.out.weight.view(d, b.attn.H, dh) * (hg * lg).view(1, -1, 1))[:, kh].reshape(d, -1)
                na.qkv.weight.copy_(qw); na.qkv.bias.copy_(qb); na.out.weight.copy_(ow); na.out.bias.copy_(b.attn.out.bias * lg)
                b.attn = na
            ng = b.mlp.gate(); kn = (ng > 0).nonzero().squeeze(1)
            if len(kn) == 0:
                b.mlp = ConstBranch(b.mlp.fc2.bias * lg)
            else:
                nmlp = GatedMLP(d, len(kn), 0.0, gated=False)
                nmlp.fc1.weight.copy_(b.mlp.fc1.weight[kn]); nmlp.fc1.bias.copy_(b.mlp.fc1.bias[kn])
                nmlp.fc2.weight.copy_((b.mlp.fc2.weight * (ng * lg))[:, kn]); nmlp.fc2.bias.copy_(b.mlp.fc2.bias * lg)
                b.mlp = nmlp
            b.layer_gate = None
            blocks.append(b)
        m.encoder.blocks = nn.ModuleList(blocks)
    else:                                                        # LLM: drop blocks deeper than the last open gate
        g = m.encoder.depth_gate()
        last = int((g > 0).nonzero().max()) + 1 if (g > 0).any() else 1
        print(f'[export] LLM depth {m.encoder.L} -> {last} (re-truncate with llm_layers={last} for deployment)')
    for mod in (m.adv_user, m.adv_dev):                          # training-only heads
        for p in mod.parameters():
            p.requires_grad = False
    return m.eval()      # modules created above default to train mode (attention dropout!)


def count_params(model, inference_only=True):
    skip = ('adv_user', 'adv_dev')
    return sum(p.numel() for n, p in model.named_parameters() if not (inference_only and n.startswith(skip)))


### Cell 4 — Training: device-physics views, group-DRO, GCE, adversarial invariance, budget dual ascent

In [ ]:
# ============================================================================
# CELL 4 — Training under heterogeneity and label noise
# ============================================================================
def random_rotation(B, max_deg, vertical_only, device):
    ang = (torch.rand(B, device=device) * 2 - 1) * math.radians(max_deg)
    if vertical_only:
        axis = torch.tensor([0., 1., 0.], device=device).expand(B, 3)
    else:
        axis = F.normalize(torch.randn(B, 3, device=device), dim=-1)
    Kx = torch.zeros(B, 3, 3, device=device)
    Kx[:, 0, 1], Kx[:, 0, 2], Kx[:, 1, 2] = -axis[:, 2], axis[:, 1], -axis[:, 0]
    Kx = Kx - Kx.transpose(1, 2)
    I = torch.eye(3, device=device).expand(B, 3, 3)
    s, c = torch.sin(ang).view(B, 1, 1), torch.cos(ang).view(B, 1, 1)
    return I + s * Kx + (1 - c) * Kx @ Kx


def device_physics_augment(x, spec, raw_sd, cfg: EPICConfig):
    """Simulates the heterogeneities measured by Stisen et al.: orientation/placement,
    sensor gain and bias, sampling-rate mismatch / clock drift, and sensor noise."""
    B, T, C = x.shape
    dev = x.device
    x_in_absent = None
    if spec.get('optional_triads'):                                   # sensors that are exactly 0 = not present
        x_in_absent = torch.stack([(x[..., list(spec['triads'][t])] == 0).all(2).all(1)
                                   for t in range(len(spec['triads']))], 1)
    tri = torch.tensor(spec['triads'], device=dev)
    R = random_rotation(B, cfg.max_rot_deg, spec['rot'] == 'vertical', dev)
    v = x[..., tri]                                                   # [B,T,K,3]
    v = torch.einsum('bij,btkj->btki', R, v)
    gain = 1 + 0.1 * (torch.rand(B, 1, len(spec['triads']), 1, device=dev) * 2 - 1)
    v = v * gain
    x = x.clone(); x[..., tri] = v
    x = x + torch.randn(B, 1, C, device=dev) * 0.02 * raw_sd         # bias
    r = float(np.random.uniform(0.85, 1.15))                          # resampling-rate mismatch
    L = max(int(round(T * r)), 8)
    xr = F.interpolate(x.transpose(1, 2), size=L, mode='linear', align_corners=True).transpose(1, 2)
    if L >= T:
        s0 = np.random.randint(0, L - T + 1); x = xr[:, s0:s0 + T]
    else:
        pad = T - L; x = torch.cat([xr, xr[:, -1:].expand(B, pad, C)], 1)
    x = x + torch.randn_like(x) * 0.02 * raw_sd                       # sensor noise
    for t in spec.get('optional_triads', []):                         # sensor availability heterogeneity
        ch = list(spec['triads'][t])
        absent = x_in_absent[:, t] if x_in_absent is not None else torch.zeros(B, dtype=torch.bool, device=dev)
        drop = absent | (torch.rand(B, device=dev) < cfg.sensor_dropout)
        x[:, :, ch] = x[:, :, ch] * (~drop).view(B, 1, 1).float()
    return x


def gce_loss(logits, y, q):
    if q <= 0:
        return F.cross_entropy(logits, y, reduction='none')
    p = torch.softmax(logits, -1).gather(1, y[:, None]).squeeze(1).clamp_min(1e-7)
    return (1 - p ** q) / q


def jsd(l1, l2):
    p1, p2 = torch.softmax(l1, -1), torch.softmax(l2, -1)
    m = ((p1 + p2) / 2).clamp_min(1e-7).log()
    return (F.kl_div(m, p1, reduction='batchmean') + F.kl_div(m, p2, reduction='batchmean')) / 2


@torch.no_grad()
def fit_norm_stats(model: EPICNet, X: torch.Tensor, idx, bs=2048):
    s, s2, n = 0., 0., 0
    for i in range(0, len(idx), bs):
        f = model.lift(X[idx[i:i + bs]].to(DEVICE).float()).double()
        s = s + f.sum((0, 1)); s2 = s2 + (f ** 2).sum((0, 1)); n += f.shape[0] * f.shape[1]
    mu = s / n; sd = (s2 / n - mu ** 2).clamp_min(1e-8).sqrt()
    model.norm.mu.copy_(mu.float()); model.norm.sd.copy_(sd.float())


@torch.no_grad()
def predict(model, X, bs=1024, group_mask=None, idx=None):
    """idx: optional row indices — avoids materialising a copy of a large subset in RAM."""
    model.eval()
    idx = np.arange(len(X)) if idx is None else np.asarray(idx)
    L, Z = [], []
    for i in range(0, len(idx), bs):
        o = model(X[idx[i:i + bs]].to(DEVICE), group_mask=group_mask)
        L.append(o['logits'].float().cpu()); Z.append(o['z'].float().cpu())
    return torch.cat(L), torch.cat(Z)


def train_model(cfg: EPICConfig, data: Dict, tr_idx, va_idx, verbose=True):
    set_seed(cfg.seed, cfg.deterministic)
    X = torch.from_numpy(data['X']); y = torch.from_numpy(data['y']).long()
    meta = data['meta']
    users = {u: i for i, u in enumerate(sorted(meta.user.iloc[tr_idx].unique()))}
    devs = {d: i for i, d in enumerate(sorted(meta.device.iloc[tr_idx].unique()))}
    u_all = torch.tensor([users.get(u, -1) for u in meta.user])
    d_all = torch.tensor([devs.get(d, -1) for d in meta.device])
    w_all = torch.tensor(np.where(meta.boundary.values, cfg.boundary_weight, 1.0), dtype=torch.float32)
    if cfg.class_balance:                                          # inverse-sqrt class frequency (train fold)
        cnt = np.bincount(data['y'][tr_idx], minlength=len(data['classes'])).astype(float)
        cw = 1.0 / np.sqrt(np.maximum(cnt, 1)); cw = cw / (cw[data['y'][tr_idx]].mean())
        w_all = w_all * torch.tensor(cw[data['y']], dtype=torch.float32)
    model = EPICNet(cfg, data['spec'], len(data['classes']), len(users), len(devs), X.shape[1]).to(DEVICE)
    fit_norm_stats(model, X, tr_idx)
    sub = np.random.default_rng(cfg.seed).choice(tr_idx, size=min(5000, len(tr_idx)), replace=False)
    raw_sd = X[sub].float().reshape(-1, X.shape[-1]).std(0).to(DEVICE)

    _fg, _sg, learned_gates = gate_flags(cfg)
    feat_gates_learned = _fg and cfg.fixed_groups is None
    gate_p = [p for n, p in model.named_parameters() if 'log_alpha' in n and p.requires_grad]
    other = [p for n, p in model.named_parameters() if 'log_alpha' not in n and p.requires_grad]
    opt = torch.optim.AdamW(other, lr=cfg.lr, weight_decay=cfg.weight_decay)
    spe = max(1, len(tr_idx) // cfg.batch_size)                    # steps per epoch
    # constant LR, large enough that a gate can cross open<->closed within ~2 epochs even on small
    # datasets (segmented LMDHG has only ~6 steps per epoch, HHAR several hundred)
    opt_g = torch.optim.Adam(gate_p or [nn.Parameter(torch.zeros(1, device=DEVICE))],
                             lr=min(0.3, max(cfg.gate_lr, 1.5 / spe)))
    steps = cfg.epochs * spe
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=steps + 10, pct_start=0.1)
    use_amp = DEVICE.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    q = torch.ones(len(users), device=DEVICE) / len(users)            # group-DRO weights
    lam = 0.0
    tr_idx_t = torch.as_tensor(tr_idx)
    best, bad, hist = -1, 0, []
    best_any, best_feas = (-1.0, None, -1), (-1.0, None, -1)      # (val F1, state, epoch)
    step = 0
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    t_train = time.perf_counter()
    for ep in range(cfg.epochs):
        t_ep = time.perf_counter()
        model.train()
        perm = tr_idx_t[torch.randperm(len(tr_idx_t))]
        budget_on = learned_gates and ep >= cfg.gate_warmup_epochs
        tot = {'loss': 0., 'task': 0., 'cost': 0., 'n': 0}
        for i in range(0, len(perm) - 1, cfg.batch_size):
            b = perm[i:i + cfg.batch_size]
            if len(b) < 2:
                continue
            xb, yb = X[b].to(DEVICE).float(), y[b].to(DEVICE)
            ub, db, wb = u_all[b].to(DEVICE), d_all[b].to(DEVICE), w_all[b].to(DEVICE)
            p_prog = step / max(steps, 1)
            grl = (2 / (1 + math.exp(-10 * p_prog)) - 1) if cfg.use_adv else 0.0
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_amp):
                v1 = device_physics_augment(xb, data['spec'], raw_sd, cfg)
                o1 = model(v1, grl=grl if cfg.use_adv else None)
                per = gce_loss(o1['logits'].float(), yb, cfg.gce_q) * wb
                if cfg.use_dro:
                    gl = torch.zeros(len(users), device=DEVICE).index_add_(0, ub, per)
                    gc = torch.zeros(len(users), device=DEVICE).index_add_(0, ub, torch.ones_like(per))
                    present = gc > 0
                    gl = gl / gc.clamp_min(1)
                    with torch.no_grad():
                        q[present] *= torch.exp(cfg.dro_eta * gl[present].detach()); q /= q.sum()
                    task = (q[present] * gl[present]).sum() / q[present].sum()
                else:
                    task = per.mean()
                loss = task
                if cfg.use_consistency:
                    v2 = device_physics_augment(xb, data['spec'], raw_sd, cfg)
                    o2 = model(v2)
                    loss = loss + cfg.cons_weight * jsd(o1['logits'].float(), o2['logits'].float())
                if cfg.use_adv:
                    loss = loss + cfg.adv_weight * (F.cross_entropy(o1['adv_user'].float(), ub) +
                                                    F.cross_entropy(o1['adv_dev'].float(), db))
                cr = model.budget_ratio('expected') if learned_gates else torch.tensor(1.0)
                if budget_on:
                    loss = loss + lam * (cr - cfg.budget)
                    if feat_gates_learned:                     # data-minimisation price on kept groups
                        loss = loss + cfg.minimization_weight * model.in_gate.p_open().mean()
            opt.zero_grad(set_to_none=True); opt_g.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            gate_has_grad = any(p.grad is not None for p in gate_p)   # none when no gate is learned
            scaler.unscale_(opt)
            if gate_has_grad:
                scaler.unscale_(opt_g)
            torch.nn.utils.clip_grad_norm_(other, 1.0)
            scaler.step(opt)
            if gate_has_grad:
                scaler.step(opt_g)   # GradScaler asserts on an optimizer without gradients (CUDA AMP)
            scaler.update()
            if step < steps + 9:
                sched.step()
            step += 1
            if budget_on:                                              # dual ascent on the multiplier
                lam = max(0.0, lam + cfg.lambda_lr / spe * float(cr.detach() - cfg.budget))
            tot['loss'] += float(loss); tot['task'] += float(task); tot['cost'] += float(cr); tot['n'] += 1
        logits, _ = predict(model, X, idx=va_idx)
        f1 = f1_score(y[va_idx].numpy(), logits.argmax(1).numpy(), average='macro')
        hc = float(model.budget_ratio('hard')) if learned_gates else 1.0
        feasible = (not learned_gates) or hc <= cfg.budget * 1.1
        n = max(tot['n'], 1)
        hist.append(dict(epoch=ep, loss=tot['loss'] / n, task=tot['task'] / n, exp_cost=tot['cost'] / n,
                         hard_cost=hc, lam=lam, val_f1=f1, epoch_s=time.perf_counter() - t_ep))
        if verbose:
            print(f'  ep {ep:02d} loss {tot["loss"]/n:.3f} val-F1 {f1:.3f} cost {hc:.2f} '
                  f'(target {cfg.budget if learned_gates else 1:.2f}) lambda {lam:.2f}')
        if f1 > best_any[0]:
            best_any = (f1, copy.deepcopy(model.state_dict()), ep)
        if feasible and f1 > best_feas[0]:
            best_feas = (f1, copy.deepcopy(model.state_dict()), ep)
        score = f1 + (1.0 if feasible else 0.0)
        if score > best:
            best, bad = score, 0
        else:
            bad += 1
            has_feasible = best_feas[1] is not None                    # never stop before the budget is met
            if bad >= cfg.patience and ep >= cfg.gate_warmup_epochs + 2 and has_feasible:
                break
    # prefer the cheapest-feasible checkpoint unless it costs more than budget_f1_tol of validation F1
    if best_feas[1] is not None and best_feas[0] >= best_any[0] - cfg.budget_f1_tol:
        chosen = best_feas
    else:
        chosen = best_any
        if learned_gates:
            print(f'  ! budget {cfg.budget} not met without losing > {cfg.budget_f1_tol} val-F1 — '
                  f'kept the best-F1 checkpoint (epoch {chosen[2]}); consider a larger budget or more epochs')
    model.load_state_dict(chosen[1])
    hist = pd.DataFrame(hist)
    hist.attrs['train_s'] = time.perf_counter() - t_train
    hist.attrs['peak_gpu_mb_train'] = torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == 'cuda' else float('nan')
    return model, hist


@torch.no_grad()
def push_prototypes(model: EPICNet, X: torch.Tensor, y: torch.Tensor, idx):
    """ProtoPNet projection: replace each prototype by the closest real training window
    embedding of its class, so every explanation points at an actual recorded example."""
    _, Z = predict(model, X, idx=idx)
    Zn = F.normalize(Z, dim=-1); yi = y[idx]
    P = model.head.protos
    for k in range(model.head.K):
        cls = (yi == k).nonzero().squeeze(1)
        if len(cls) == 0:
            continue
        used = set()
        for mi in range(model.head.M):
            j = k * model.head.M + mi
            s = Zn[cls] @ F.normalize(P[j].cpu(), dim=0)
            for u in used:
                s[u] = -2.0                                    # distinct windows per prototype
            jj = int(s.argmax()); used.add(jj); best = cls[jj]
            P[j].copy_(Z[best].to(P.device)); model.head.proto_src[j] = int(idx[best])


### Cell 5 — Trust layer: calibration, conformal sets, guardrails, fairness, explanations, ledger, feedback auditor

In [ ]:
# ============================================================================
# CELL 5 — Pillar C: the HCAI trust layer
# ============================================================================
def fit_temperature(logits: torch.Tensor, y: torch.Tensor) -> float:
    t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad(); l = F.cross_entropy(logits / t.exp(), y); l.backward(); return l
    opt.step(closure)
    return float(t.exp())


def expected_calibration_error(p: np.ndarray, y: np.ndarray, bins=15) -> float:
    conf, pred = p.max(1), p.argmax(1)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)


class ConformalPredictor:
    """Split conformal with LAC scores; 'class' = Mondrian (class-conditional) calibration,
    which gives every activity its own coverage guarantee instead of letting rare
    classes (e.g. stairs) absorb all the misses. Guarantee assumes exchangeability
    between calibration and test windows — exactly what an unseen subject or device
    breaks, so coverage on the held-out group is REPORTED, not assumed."""
    def __init__(self, alpha=0.1, mode='class'):
        self.alpha, self.mode = alpha, mode

    def fit(self, p: np.ndarray, y: np.ndarray):
        self.s_cal, self.y_cal, self.K = 1 - p[np.arange(len(y)), y], y, p.shape[1]
        self.q_marg, self.q = self._thresholds(self.alpha)
        return self

    def _thresholds(self, alpha):
        def qhat(v):
            n = len(v)
            if n == 0:
                return 1.0
            return float(np.quantile(v, min(1.0, math.ceil((n + 1) * (1 - alpha)) / n), method='higher'))
        qm = qhat(self.s_cal)
        q = np.array([qhat(self.s_cal[self.y_cal == k]) if (self.y_cal == k).sum() >= 10 else qm for k in range(self.K)])
        return qm, q

    def sets(self, p: np.ndarray, alpha: Optional[float] = None, mode: Optional[str] = None) -> np.ndarray:
        qm, q = (self.q_marg, self.q) if alpha is None else self._thresholds(alpha)
        thr = q[None, :] if (mode or self.mode) == 'class' else qm
        return (1 - p) <= thr


class EmbeddingOOD:
    """Mahalanobis distance to the nearest class mean (tied, shrunk covariance)."""
    def fit(self, Z: np.ndarray, y: np.ndarray, Zcal: np.ndarray, q=0.99):
        self.mu = np.stack([Z[y == k].mean(0) if (y == k).any() else Z.mean(0) for k in range(y.max() + 1)])
        Zc = Z - self.mu[y]
        cov = np.cov(Zc.T) + 1e-3 * np.eye(Z.shape[1])
        self.P = np.linalg.inv(cov)
        self.thr = float(np.quantile(self.score(Zcal), q))
        return self

    def score(self, Z):
        d = Z[:, None, :] - self.mu[None]
        return np.sqrt(np.einsum('nkd,de,nke->nk', d, self.P, d).min(1))


def physical_guardrails(X: np.ndarray, spec: Dict) -> List[List[str]]:
    """Boundary conditions that must hold before ANY automated decision is released."""
    reasons = [[] for _ in range(len(X))]
    X = np.asarray(X, dtype=np.float32)
    bad = ~np.isfinite(X).all((1, 2))
    for i in np.where(bad)[0]:
        reasons[i].append('non_finite_input')
    Xf = np.nan_to_num(X)
    if spec['name'] in ('hhar', 'synthetic'):
        accmag = np.linalg.norm(Xf[..., 0:3], axis=-1)
        for i in np.where(accmag.max(1) > spec['acc_sat'])[0]:
            reasons[i].append('accelerometer_saturation')
        absent = (Xf[..., 3:6] == 0).all((1, 2))
        for i in np.where(absent)[0]:
            reasons[i].append('gyroscope_absent')                     # known missing sensor: logged, not blocking
        for name, sl in (('accelerometer', slice(0, 3)), ('gyroscope', slice(3, 6))):
            flat = Xf[..., sl].std(1).max(1) < 1e-6
            if name == 'gyroscope':
                flat &= ~absent
            for i in np.where(flat)[0]:
                reasons[i].append(f'{name}_flatline')
    elif spec['name'] == 'lmdhg':
        L = np.abs(Xf[..., :69]).sum(-1) == 0; R = np.abs(Xf[..., 69:]).sum(-1) == 0
        for i in np.where((L & R).mean(1) > 0.5)[0]:
            reasons[i].append('hands_not_tracked')
        for i in np.where(np.abs(Xf).max((1, 2)) > spec['fov_mm'])[0]:
            reasons[i].append('outside_sensor_field_of_view')
    return reasons


NONBLOCKING = {'gyroscope_absent'}


def decide(sets: np.ndarray, ood: np.ndarray, bc: List[List[str]], pred: np.ndarray,
           classes: List[str], critical: Tuple[str, ...]) -> Tuple[List[str], List[List[str]]]:
    """Decision policy (the 'contract' every output must satisfy):
    reject_input (BC violated) > abstain_ood > abstain_ambiguous (|set| != 1)
    > human_review (critical class, GDPR Art. 22 / Recital 71) > auto."""
    dec, why = [], []
    for i in range(len(pred)):
        r = list(bc[i])
        if any(x not in NONBLOCKING for x in r):
            dec.append('reject_input')
        elif ood[i]:
            dec.append('abstain_ood'); r.append('out_of_distribution_embedding')
        elif sets[i].sum() != 1:
            dec.append('abstain_ambiguous'); r.append(f'conformal_set_size_{int(sets[i].sum())}')
        elif classes[pred[i]] in critical:
            dec.append('human_review'); r.append('critical_class_requires_human')
        else:
            dec.append('auto')
        why.append(r)
    return dec, why


# ----------------------------- fairness audit -----------------------------
def fairness_audit(df: pd.DataFrame, group_col: str, K: int, min_n: int = 30) -> Dict:
    """df: y, pred, answered (bool). Demographic parity is reported but flagged: in HAR the
    base rates of activities legitimately differ across people, so DP-parity would force
    wrong predictions. Equalised odds and abstention parity are the meaningful criteria
    (selective classification can MAGNIFY group disparities — Jones et al., ICLR'21)."""
    rows = []
    sizes = df.groupby(group_col).size()
    small = sizes[sizes < min_n]
    if len(small) and len(small) < len(sizes):                 # groups too small for per-class rates are noise
        df = df[~df[group_col].isin(small.index)]
    for g, d in df.groupby(group_col):
        tpr = [((d.pred == k) & (d.y == k)).sum() / max((d.y == k).sum(), 1) if (d.y == k).any() else np.nan for k in range(K)]
        fpr = [((d.pred == k) & (d.y != k)).sum() / max((d.y != k).sum(), 1) for k in range(K)]
        dp = [(d.pred == k).mean() for k in range(K)]
        ans = d[d.answered]
        rows.append(dict(group=g, n=len(d), acc=(d.y == d.pred).mean(),
                         macro_f1=f1_score(d.y, d.pred, average='macro', labels=list(range(K)), zero_division=0),
                         abstain_rate=1 - d.answered.mean(),
                         selective_acc=(ans.y == ans.pred).mean() if len(ans) else np.nan,
                         tpr=tpr, fpr=fpr, dp=dp))
    t = pd.DataFrame(rows)
    T, Fp, D = np.array(t.tpr.tolist(), float), np.array(t.fpr.tolist(), float), np.array(t.dp.tolist(), float)
    summary = dict(
        group=group_col, n_groups=len(t), groups_excluded_lt_min_n=int(len(small)) if len(small) < len(sizes) else 0,
        worst_group_f1=float(t.macro_f1.min()), best_group_f1=float(t.macro_f1.max()),
        f1_std=float(t.macro_f1.std()),
        disparate_impact_acc=float(t.acc.min() / max(t.acc.max(), 1e-9)),          # four-fifths rule: >= 0.8
        eo_tpr_gap=float(np.nanmax(np.nanmax(T, 0) - np.nanmin(T, 0))),
        eo_fpr_gap=float(np.nanmax(np.nanmax(Fp, 0) - np.nanmin(Fp, 0))),
        dp_gap_FLAGGED=float(np.nanmax(np.nanmax(D, 0) - np.nanmin(D, 0))),
        abstention_gap=float(t.abstain_rate.max() - t.abstain_rate.min()),
        selective_acc_gap=float(t.selective_acc.max() - t.selective_acc.min()))
    return dict(summary=summary, table=t.drop(columns=['tpr', 'fpr', 'dp']))


def aurc(conf: np.ndarray, correct: np.ndarray) -> float:
    o = np.argsort(-conf); c = correct[o].astype(float)
    risk = 1 - np.cumsum(c) / np.arange(1, len(c) + 1)
    return float(risk.mean())


# ----------------------------- explanations -----------------------------
@torch.no_grad()
def group_occlusion(model: EPICNet, x: torch.Tensor, cls: int) -> np.ndarray:
    """Faithful attribution: logit drop when one feature group is removed (only for
    groups the model actually kept — pruned groups have zero influence by construction)."""
    model.eval(); x = x.to(DEVICE).float()
    G = len(model.group_names)
    base = model(x[None])['logits'][0, cls]
    masks = torch.ones(G, G, device=DEVICE) - torch.eye(G, device=DEVICE)
    out = model(x[None].expand(G, *x.shape), group_mask=masks)['logits'][:, cls]
    return (base - out).cpu().numpy()


def counterfactual(model: EPICNet, x: torch.Tensor, target: int, bounds: Tuple[torch.Tensor, torch.Tensor],
                   ood: EmbeddingOOD, steps=300, lr=0.02, l1=0.05, tv=0.05, plaus=0.01) -> Dict:
    """'What would the signal have to look like to be classified as <target>?'
    Sparse (L1), temporally smooth (TV) perturbation, restricted to raw channels the model
    uses, pulled towards the target class's embedding region, clipped to the physical range seen in training (boundary conditions), and
    accepted only if the result is in-distribution (plausible recourse, not an adversarial example)."""
    model.eval(); x = x.to(DEVICE).float()
    lo, hi = bounds[0].to(DEVICE).float(), bounds[1].to(DEVICE).float()
    scale = (hi - lo).clamp_min(1e-6)
    delta = torch.zeros_like(x, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    mu_t = torch.tensor(ood.mu[target], dtype=torch.float32, device=DEVICE)
    Pm = torch.tensor(ood.P, dtype=torch.float32, device=DEVICE)
    for _ in range(steps):
        xc = torch.max(torch.min(x + delta * scale, hi), lo)
        o = model(xc[None])
        loss = F.cross_entropy(o['logits'], torch.tensor([target], device=DEVICE)) \
            + l1 * delta.abs().mean() + tv * (delta[1:] - delta[:-1]).abs().mean()
        dz = o['z'][0] - mu_t
        loss = loss + plaus * (dz @ Pm @ dz)                    # stay inside the target class's data region
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        xc = torch.max(torch.min(x + delta * scale, hi), lo)
        o = model(xc[None]); p = torch.softmax(o['logits'], -1)[0]
        dist = float(ood.score(o['z'].cpu().numpy())[0])
        change = (xc - x).abs() / scale
    return dict(x_cf=xc.cpu(), success=int(p.argmax()) == target, p_target=float(p[target]),
                plausible=dist <= ood.thr, ood_score=dist,
                channel_change=change.mean(0).cpu().numpy(),
                most_changed_time=(int(change.mean(1).argmax()),))


# ----------------------------- accountability -----------------------------
def pseudonymise(user: str, salt: str) -> str:
    """GDPR Art. 4(5)/5(1)(c): the ledger never stores a raw subject identifier."""
    return hmac.new(salt.encode(), str(user).encode(), hashlib.sha256).hexdigest()[:16]


def model_fingerprint(model) -> str:
    h = hashlib.sha256()
    for k, v in sorted(model.state_dict().items()):
        h.update(k.encode()); h.update(v.detach().cpu().numpy().tobytes())
    return h.hexdigest()[:20]


class HashChainLedger:
    """Append-only, tamper-evident decision log (STRIDE: Tampering + Repudiation).
    Same integrity property a permissioned smart-contract ledger provides; anchor the
    head hash on a chain (e.g. Hyperledger Fabric) for third-party verifiability."""
    def __init__(self):
        self.entries, self.head = [], '0' * 64

    def append(self, record: Dict):
        body = json.dumps(record, sort_keys=True, default=str)
        h = hashlib.sha256((self.head + body).encode()).hexdigest()
        self.entries.append(dict(prev=self.head, hash=h, record=record)); self.head = h

    def verify(self) -> bool:
        prev = '0' * 64
        for e in self.entries:
            body = json.dumps(e['record'], sort_keys=True, default=str)
            if e['prev'] != prev or hashlib.sha256((prev + body).encode()).hexdigest() != e['hash']:
                return False
            prev = e['hash']
        return True


def explain_text(rec: Dict) -> str:
    """Grounded verbaliser: every number in the text comes from the decision record.
    (If you add an LLM to rephrase this, check that it introduces no new facts.)"""
    s = (f"Decision: {rec['decision']}. Predicted '{rec['pred']}' with calibrated confidence "
         f"{rec['confidence']:.2f}; conformal set = {rec['set']}.")
    if not rec['set']:
        s += ' No activity reached its class-specific calibrated threshold, so the system does not commit to any label.'
    if rec.get('reasons'):
        s += f" Reasons: {', '.join(rec['reasons'])}."
    if rec.get('top_groups'):
        s += f" Most influential signals: {', '.join(rec['top_groups'])}."
    if rec.get('prototype'):
        s += f" Most similar recorded example: {rec['prototype']}."
    if rec['decision'] != 'auto':
        s += ' You can appeal this outcome; a human reviewer will decide (GDPR Art. 22(3)).'
    return s


class FeedbackAuditor:
    """Bias-audited human-override loop (formal rule; see notebook text).
    1) Conformal surprise test (reviewer-bias / group-shift detector).
       s_i = 1[ y_i^h not in C_marg(x_i) ]   — the human label lies outside the model's conformal set.
       For each group g: Fisher exact test of rate(s | g) > rate(s | not g); flag g by Holm's step-down
       procedure at family-wise level beta. Overrides concentrated on one group beyond what the
       calibrated model finds plausible indicate reviewer bias OR a group-specific shift —
       both must go to a person, never be silently learned (performative feedback loops).
    2) Candidate update, prototype head only (cheap, reversible), with replay to prevent forgetting:
       theta' = argmin  sum_{i in R} w_i CE(f(x_i), y_i^h) + (1/|B|) sum_{j in B} CE(f(x_j), y_j),
       w_i = 1 / (|G_R| * |R_g(i)|)   (no single group's overrides dominate the update)
    3) ACCEPT iff  no flagged group  and  dEO_A <= eps_eo  and  dF1worst_A >= -delta  and  KL_A <= eps_kl
       on a fixed anchor set A; every decision is written to the hash-chained ledger."""
    def __init__(self, beta=0.05, eps_eo=0.02, delta=0.01, eps_kl=0.05, steps=60, lr=5e-3, min_n=5):
        self.beta, self.eps_eo, self.delta, self.eps_kl = beta, eps_eo, delta, eps_kl
        self.steps, self.lr, self.min_n = steps, lr, min_n

    def _metrics(self, model, XA, yA, gA, K):
        logits, _ = predict(model, XA)
        df = pd.DataFrame(dict(y=yA, pred=logits.argmax(1).numpy(), g=gA, answered=True))
        return fairness_audit(df, 'g', K)['summary'], torch.softmax(logits, -1)

    def surprise_test(self, rv: pd.DataFrame) -> pd.DataFrame:
        from scipy.stats import fisher_exact
        rows = []
        groups = [g for g, n in rv.group.value_counts().items() if n >= self.min_n]
        for g in groups:
            a = rv.surprise[rv.group == g]; b = rv.surprise[rv.group != g]
            _, p = fisher_exact([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]], alternative='greater')
            rows.append(dict(group=g, n=len(a), surprise_rate=a.mean(), rest_rate=b.mean() if len(b) else np.nan, p=p))
        t = pd.DataFrame(rows)
        if len(t):                                                    # Holm step-down (uniformly more powerful than Bonferroni)
            o = np.argsort(t.p.values); m = len(t); flag = np.zeros(m, bool)
            for r, i in enumerate(o):
                if t.p.values[i] < self.beta / (m - r):
                    flag[i] = True
                else:
                    break
            t['flag'] = flag
        return t

    def review(self, model, reviewed: pd.DataFrame, X: torch.Tensor, y: torch.Tensor, replay_idx,
               anchor_idx, g_anchor, K: int, ledger: HashChainLedger):
        """reviewed: columns idx, y_human, group, surprise (bool)."""
        st = self.surprise_test(reviewed)
        flagged = st.group[st.flag].tolist() if len(st) else []
        before, pA = self._metrics(model, X[anchor_idx], y[anchor_idx].numpy(), g_anchor, K)
        cand = copy.deepcopy(model).eval()
        opt = torch.optim.Adam(cand.head.parameters(), lr=self.lr)
        cnt = reviewed.group.value_counts()
        w = torch.tensor([1.0 / (len(cnt) * cnt[g]) for g in reviewed.group], dtype=torch.float32, device=DEVICE)
        yo = torch.tensor(reviewed.y_human.values, device=DEVICE)
        with torch.no_grad():                                      # backbone frozen: reuse embeddings
            zo = cand(X[reviewed.idx.values].to(DEVICE))['z']
            rp = np.random.default_rng(0).choice(replay_idx, size=min(2048, len(replay_idx)), replace=False)
            zr, yr = predict(cand, X[rp])[1].to(DEVICE), y[rp].to(DEVICE)
        for _ in range(self.steps):
            loss = (F.cross_entropy(cand.head(zo)[0], yo, reduction='none') * w).sum() \
                + F.cross_entropy(cand.head(zr)[0], yr)
            opt.zero_grad(); loss.backward(); opt.step()
        after, pA2 = self._metrics(cand, X[anchor_idx], y[anchor_idx].numpy(), g_anchor, K)
        kl = float(F.kl_div(pA2.clamp_min(1e-7).log(), pA, reduction='batchmean'))
        d_eo = after['eo_tpr_gap'] - before['eo_tpr_gap']
        d_wf = after['worst_group_f1'] - before['worst_group_f1']
        checks = dict(flagged_groups=(len(flagged), len(flagged) == 0), eo_gap_change=(d_eo, d_eo <= self.eps_eo),
                      worst_group_f1_change=(d_wf, d_wf >= -self.delta), prediction_shift_kl=(kl, kl <= self.eps_kl))
        accept = all(ok for _, ok in checks.values())
        ledger.append(dict(type='feedback_update', accepted=accept, n_reviewed=len(reviewed), flagged=flagged,
                           checks={k: [float(v), bool(ok)] for k, (v, ok) in checks.items()},
                           model_before=model_fingerprint(model), model_after=model_fingerprint(cand)))
        return (cand if accept else model), accept, checks, st


STRIDE = [
    ('Spoofing', 'Forged sensor stream / impersonated device', 'Device attestation; physical BC guardrails; OOD gate'),
    ('Tampering', 'Model weights or decision log altered', 'SHA-256 model fingerprint in every record; hash-chained ledger'),
    ('Repudiation', 'Operator denies an override or decision', 'Ledger stores decision, reasons, reviewer action, model hash'),
    ('Information disclosure', 'Re-identification from motion signatures (surveillance capitalism)',
     'Subject-adversarial training; HMAC pseudonyms; gate-driven data minimisation; on-device inference'),
    ('Denial of service', 'Inputs crafted to force abstention floods', 'Rate-limit abstentions per device; alert on abstention drift'),
    ('Elevation of privilege', 'Override channel used to retrain model at will', 'FeedbackAuditor acceptance rule; head-only, reversible updates'),
]


def model_card(cfg: EPICConfig, data: Dict, summary: Dict, gate_rep: Dict, eff: Dict) -> str:
    stride = '\n'.join(f'| {a} | {b} | {c} |' for a, b, c in STRIDE)
    return f"""# Model card — EPIC-HAR ({cfg.dataset}, {cfg.protocol})
**Intended use:** activity / gesture recognition on pervasive devices with human oversight. Not for covert monitoring.
**Out of scope:** decisions with legal or similarly significant effect without human review (GDPR Art. 22).

## Performance (held-out {('subjects' if cfg.protocol == 'LOSO' else 'device models')})
{json.dumps(summary, indent=2, default=float)}

## Data minimisation (GDPR Art. 5(1)(c)) — learned by the input gates
- Feature groups kept: {gate_rep['n_groups_kept']}/{gate_rep['n_groups']} -> {gate_rep['feature_groups_kept']}
- Sensors required: {gate_rep['sensors_needed']} (of {gate_rep['sensors_all']})

## Efficiency
{json.dumps(eff, indent=2, default=float)}

## Value-Sensitive Design (Friedman & Kahn) stakeholder map
| Stakeholder | Value | Mechanism |
|---|---|---|
| Monitored person | privacy, autonomy, recourse | minimisation, pseudonyms, counterfactuals, appeal |
| Clinician / operator | trust, accountability | calibrated sets, abstention, ledger |
| Under-served users/devices | fairness | group-DRO, EO + abstention-parity audit |
| Organisation / regulator | compliance (GDPR, EU AI Act) | model card, STRIDE, audited feedback loop |

## STRIDE threat model
| Threat | Example | Mitigation |
|---|---|---|
{stride}
"""


### Cell 6 — Protocols (LOSO / LODO) and experiment runner

In [ ]:
# ============================================================================
# CELL 6 — Protocols and experiment runner
#   LOSO: test on an unseen subject; LODO: test on an unseen device MODEL.
# ============================================================================
def make_folds(meta: pd.DataFrame, cfg: EPICConfig):
    rng = np.random.default_rng(cfg.split_seed)
    folds = []
    if cfg.protocol == 'LOSO':
        users = sorted(meta.user.unique())
        if len(users) > 12:                                        # many subjects -> GroupKFold
            perm = rng.permutation(users); chunks = np.array_split(perm, cfg.group_kfold)
            test_sets = [set(c) for c in chunks]
        else:
            test_sets = [{u} for u in users]
        for i, ts in enumerate(test_sets):
            rest = [u for u in users if u not in ts]
            if len(test_sets) == len(users):                       # true LOSO: one held-out calibration subject
                cal = {rest[i % len(rest)]}
            else:                                                  # many small groups (LMDHG): ~1/8 of the rest
                cal = set(rng.choice(rest, size=max(3, len(rest) // 8), replace=False))
            te = np.where(meta.user.isin(ts))[0]
            ca = np.where(meta.user.isin(cal))[0]
            tr = np.where(~meta.user.isin(ts) & ~meta.user.isin(cal))[0]
            folds.append(dict(name=f'test={sorted(map(str, ts))}', tr=tr, ca=ca, te=te))
    elif cfg.protocol == 'LODO':
        models = sorted(meta.model.unique())
        for i, m in enumerate(models):
            rest = [x for x in models if x != m]
            cal_dev = meta.device[meta.model == rest[i % len(rest)]].value_counts().index[0]   # largest device of that model
            te = np.where(meta.model == m)[0]
            ca = np.where(meta.device == cal_dev)[0]
            tr = np.where((meta.model != m) & (meta.device != cal_dev))[0]
            folds.append(dict(name=f'test_model={m}', tr=tr, ca=ca, te=te))
    elif cfg.protocol == 'LOSDO':
        # Leave-One-Subject-AND-Device-model-Out: test = subject u on phone model m; training contains
        # NEITHER u (any device) NOR m (any subject) -> no moment of the test recording is ever seen through
        # another phone. Each subject is paired with one model in rotation (same #folds as LOSO).
        # Calibration = another unseen subject on the seen models (subject shift, as in LOSO).
        users, models = sorted(meta.user.unique()), sorted(meta.model.unique())
        for i, u in enumerate(users):
            m = models[i % len(models)]
            cal_u = users[(i + 1) % len(users)]
            te = np.where((meta.user == u) & (meta.model == m))[0]
            ca = np.where((meta.user == cal_u) & (meta.model != m))[0]
            tr = np.where((meta.user != u) & (meta.user != cal_u) & (meta.model != m))[0]
            if len(te) and len(ca) and len(tr):
                folds.append(dict(name=f'test={u}@{m}', tr=tr, ca=ca, te=te))
    elif cfg.protocol in ('XDEV_P2W', 'XDEV_W2P'):                   # strongest device shift: other body site + device class
        src, tgt = ('Phones', 'Watch') if cfg.protocol == 'XDEV_P2W' else ('Watch', 'Phones')
        cal_dev = meta.device[meta.stream == src].value_counts().index[0]
        te = np.where(meta.stream == tgt)[0]
        ca = np.where(meta.device == cal_dev)[0]
        tr = np.where((meta.stream == src) & (meta.device != cal_dev))[0]
        folds.append(dict(name=f'{src}->{tgt}', tr=tr, ca=ca, te=te))
    else:
        raise ValueError(cfg.protocol)
    if 'purity' in meta:                                           # train / calibrate on clean windows only;
        ok = meta.purity.values >= cfg.min_purity                  # test keeps ALL windows (frame-level eval)
        for f in folds:
            f['tr'], f['ca'] = f['tr'][ok[f['tr']]], f['ca'][ok[f['ca']]]
    return folds[:cfg.max_folds] if cfg.max_folds else folds


def ensure_data(c: EPICConfig, both: bool = False):
    """Download / copy raw data only if the parsed cache for this config does not exist yet."""
    if c.dataset == 'hhar':
        streams = ('Phones', 'Watch') if both else (c.hhar_stream,)
        need = []
        for st in streams:
            cc = copy.deepcopy(c); cc.hhar_stream = st
            if not os.path.exists(os.path.join(CACHE_DIR, _cache_key(cc)) + '.npz'):
                need.append(st)
        if need:
            fetch_hhar(HHAR_LOCAL)
        else:
            print('HHAR: using cached windows (no download needed)')
    elif c.dataset == 'lmdhg':
        copy_lmdhg_from_drive(LMDHG_DRIVE, LMDHG_LOCAL)


@torch.no_grad()
def cpu_latency_ms(model, T, C_raw, reps=50):
    m = copy.deepcopy(model).cpu().eval(); x = torch.randn(1, T, C_raw)
    for _ in range(5):
        m(x)
    t0 = time.perf_counter()
    for _ in range(reps):
        m(x)
    return (time.perf_counter() - t0) / reps * 1000


def model_size_mb(model):
    import io
    b = io.BytesIO(); torch.save({k: v for k, v in model.state_dict().items() if not k.startswith(('adv_', ))}, b)
    return b.tell() / 1e6


def quantize_int8(model):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')                       # torch.ao.quantization deprecation notice
            from torch.ao.quantization import quantize_dynamic
            return quantize_dynamic(copy.deepcopy(model).cpu().eval(), {nn.Linear}, dtype=torch.qint8)
    except Exception as e:
        print(f'[int8] dynamic quantisation unavailable: {e}'); return None


class _LogitsOnly(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m

    def forward(self, x):
        return self.m(x)['logits']


def measured_flops(model, T, C) -> float:
    """FLOPs for ONE window counted by torch's FlopCounterMode (same counter for every model).
    LSTM/GRU kernels are not counted by FlopCounterMode, so they are added analytically."""
    from torch.utils.flop_counter import FlopCounterMode
    # train() disables the fused nn.TransformerEncoder fast path (invisible to the counter); frozen params
    # avoid autograd-tracker errors. Dropout / BatchNorm mode does not change matmul / conv FLOPs.
    m = copy.deepcopy(model).cpu().train(); x = torch.randn(2, T, C)
    for p in m.parameters():
        p.requires_grad_(False)
    rnn = []
    def hook(mod, inp, out):
        L = inp[0].shape[1] if mod.batch_first else inp[0].shape[0]
        Bn = inp[0].shape[0] if mod.batch_first else inp[0].shape[1]
        H, D = mod.hidden_size, 2 if mod.bidirectional else 1
        tot, insz = 0, mod.input_size
        for _ in range(mod.num_layers):
            gates = 4 if isinstance(mod, nn.LSTM) else 3
            tot += Bn * D * L * gates * 2 * (insz + H) * H; insz = H * D
        rnn.append(tot)
    hs = [mm.register_forward_hook(hook) for mm in m.modules() if isinstance(mm, (nn.LSTM, nn.GRU))]
    with FlopCounterMode(display=False) as fc:
        m(x)
    for h in hs:
        h.remove()
    return float(fc.get_total_flops() + sum(rnn)) / 2                  # batch of 2 (BatchNorm in train mode) -> per window


ALPHA_SWEEP = (0.05, 0.1, 0.2, 0.3)


def alpha_sweep_metrics(cp, pt, yt, is_ood, bc, pred, classes, critical):
    out = {}
    for a in ALPHA_SWEEP:
        S = cp.sets(pt, alpha=a); dec, _ = decide(S, is_ood, bc, pred, classes, critical)
        ans = np.isin(dec, ['auto', 'human_review'])
        out[f'cov@{a}'] = float(S[np.arange(len(yt)), yt].mean()); out[f'set@{a}'] = float(S.sum(1).mean())
        out[f'answer@{a}'] = float(ans.mean())
        out[f'selacc@{a}'] = float((pred[ans] == yt[ans]).mean()) if ans.any() else np.nan
    return out


@torch.no_grad()
def gpu_throughput(model, X, bs=512, reps=5):
    if DEVICE.type != 'cuda':
        return float('nan'), float('nan')
    model = model.to(DEVICE).eval(); xb = X[:bs].to(DEVICE)
    model(xb); torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    for _ in range(reps):
        model(xb)
    torch.cuda.synchronize()
    return reps * len(xb) / (time.perf_counter() - t0), torch.cuda.max_memory_allocated() / 1e6


def peak_ram_mb():
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3       # Linux: KB -> MB


def run_fold(cfg: EPICConfig, data: Dict, fold: Dict, verbose=True):
    X = torch.from_numpy(data['X']); y = torch.from_numpy(data['y']).long(); meta = data['meta']
    K = len(data['classes'])
    rng = np.random.default_rng(cfg.split_seed)
    ca = rng.permutation(fold['ca']); va, cal = ca[: len(ca) // 2], ca[len(ca) // 2:]
    model, hist = train_model(cfg, data, fold['tr'], va, verbose=verbose)
    push_prototypes(model, X, y, fold['tr'])
    # calibration on the held-out group (after the push, so it calibrates the final model)
    lv, _ = predict(model, X, idx=va); T = fit_temperature(lv, y[va])
    lc, zc = predict(model, X, idx=cal); pc = torch.softmax(lc / T, -1).numpy()
    cp = ConformalPredictor(cfg.alpha, cfg.conformal_mode).fit(pc, y[cal].numpy())
    sub = rng.choice(fold['tr'], size=min(20000, len(fold['tr'])), replace=False)
    _, ztr = predict(model, X, idx=sub); ysub = y[sub].numpy()
    for t in data['spec'].get('optional_triads', []):             # support also covers 'sensor absent' inputs
        h = sub[: len(sub) // 2]; Xz = X[h].clone(); Xz[..., list(data['spec']['triads'][t])] = 0
        _, zz = predict(model, Xz); ztr = torch.cat([ztr, zz]); ysub = np.concatenate([ysub, y[h].numpy()])
    ood = EmbeddingOOD().fit(ztr.numpy(), ysub, zc.numpy(), cfg.ood_quantile)
    # test
    lt, zt = predict(model, X, idx=fold['te']); pt = torch.softmax(lt / T, -1).numpy()
    yt = y[fold['te']].numpy(); pred = pt.argmax(1)
    S = cp.sets(pt); is_ood = ood.score(zt.numpy()) > ood.thr
    bc = physical_guardrails(data['X'][fold['te']], data['spec'])
    Sd = S if cfg.decision_alpha is None else cp.sets(pt, alpha=cfg.decision_alpha)
    dec, why = decide(Sd, is_ood, bc, pred, data['classes'], cfg.critical_classes)
    sweep = alpha_sweep_metrics(cp, pt, yt, is_ood, bc, pred, data['classes'], cfg.critical_classes)
    m = meta.iloc[fold['te']].reset_index(drop=True)
    df = pd.DataFrame(dict(idx=fold['te'], user=m.user, device=m.device, model=m.model, y=yt, pred=pred,
                           stream=m['stream'] if 'stream' in m else 'na',
                           conf=pt.max(1), set_size=S.sum(1), covered=S[np.arange(len(yt)), yt],
                           decision=dec, reasons=[';'.join(r) for r in why], fold=fold['name']))
    df['answered'] = df.decision.isin(['auto', 'human_review'])
    pruned = export_pruned(model)
    eff = dict(params_dense=count_params(model), params_pruned=count_params(pruned),
               size_mb_dense=model_size_mb(model), size_mb_pruned=model_size_mb(pruned),
               flops_ratio=float(model.cost_ratio('hard')),
               flops_dense=measured_flops(_LogitsOnly(model), X.shape[1], X.shape[2]),
               flops_pruned=measured_flops(_LogitsOnly(pruned), X.shape[1], X.shape[2]),
               cpu_ms_dense=cpu_latency_ms(model, X.shape[1], X.shape[2]),
               cpu_ms_pruned=cpu_latency_ms(pruned, X.shape[1], X.shape[2]))
    with torch.no_grad():                                         # exactness of the gate folding
        xs = X[fold['te'][:256]]
        eff['export_max_abs_diff'] = float((model.cpu()(xs)['logits'] - pruned(xs)['logits']).abs().max())
        q8 = quantize_int8(pruned)
        if q8 is not None:
            eff['size_mb_int8'] = model_size_mb(q8)
            eff['cpu_ms_int8'] = cpu_latency_ms(q8, X.shape[1], X.shape[2])
            eff['int8_agreement'] = float((q8(xs)['logits'].argmax(1) == pruned(xs)['logits'].argmax(1)).float().mean())
    model.to(DEVICE)
    eff['gpu_win_per_s_dense'], eff['gpu_infer_mb_dense'] = gpu_throughput(model, X[fold['te'][:512]])
    eff['gpu_win_per_s_pruned'], eff['gpu_infer_mb_pruned'] = gpu_throughput(copy.deepcopy(pruned), X[fold['te'][:512]])
    eff['train_s'] = hist.attrs.get('train_s', np.nan); eff['peak_gpu_mb_train'] = hist.attrs.get('peak_gpu_mb_train', np.nan)
    eff['s_per_epoch'] = float(hist.epoch_s.mean()); eff['peak_ram_mb'] = peak_ram_mb()
    fs = dict(fold=fold['name'], acc=accuracy_score(yt, pred), macro_f1=f1_score(yt, pred, average='macro'),
              majority_baseline=float(np.bincount(yt).max() / len(yt)),
              ece=expected_calibration_error(pt, yt), temperature=T,
              coverage=float(df.covered.mean()), target_coverage=1 - cfg.alpha, avg_set=float(df.set_size.mean()),
              answer_rate=float(df.answered.mean()),
              selective_acc=float((df[df.answered].y == df[df.answered].pred).mean()) if df.answered.any() else np.nan,
              aurc=aurc(pt.max(1), pred == yt), ood_rate=float(is_ood.mean()), **sweep, **eff)
    art = dict(model=model, pruned=pruned, T=T, cp=cp, ood=ood, hist=hist, gate=model.gate_report(), fold=fold)
    return df, fs, art


def run_experiment(cfg: EPICConfig, data: Dict, tag='EPIC', verbose=True):
    os.makedirs(cfg.out_dir, exist_ok=True)
    folds = make_folds(data['meta'], cfg)
    preds, sums, arts = [], [], []
    for f in folds:
        print(f'\n=== [{tag}] {cfg.protocol} fold {f["name"]} | train {len(f["tr"])} cal {len(f["ca"])} test {len(f["te"])}')
        df, fs, art = run_fold(cfg, data, f, verbose)
        print(f'  -> F1 {fs["macro_f1"]:.3f} | cover {fs["coverage"]:.3f} | answer {fs["answer_rate"]:.2f} '
              f'| sel-acc {fs["selective_acc"]:.3f} | FLOPs {fs["flops_ratio"]:.2f} | params {fs["params_pruned"]:,}')
        preds.append(df); sums.append(fs); arts.append(art)
    P = pd.concat(preds, ignore_index=True); Sm = pd.DataFrame(sums)
    K = len(data['classes'])
    fair = {g: fairness_audit(P, g, K) for g in ('user', 'model') if P[g].nunique() > 1}
    P.to_csv(f'{cfg.out_dir}/{tag}_{cfg.dataset}_{cfg.protocol}_predictions.csv', index=False)
    Sm.to_csv(f'{cfg.out_dir}/{tag}_{cfg.dataset}_{cfg.protocol}_folds.csv', index=False)
    return dict(preds=P, folds=Sm, fair=fair, arts=arts, cfg=cfg)


def summarise(res) -> Dict:
    S, P = res['folds'], res['preds']
    out = {k: f'{S[k].mean():.3f} ± {S[k].std(ddof=0):.3f}' for k in
           ('macro_f1', 'acc', 'majority_baseline', 'ece', 'coverage', 'avg_set', 'answer_rate', 'selective_acc',
            'aurc', 'flops_ratio')}
    out['params_pruned'] = int(S.params_pruned.mean()); out['cpu_ms_pruned'] = round(float(S.cpu_ms_pruned.mean()), 2)
    for g, f in res['fair'].items():
        out[f'worst_{g}_f1'] = round(f['summary']['worst_group_f1'], 3)
        out[f'eo_tpr_gap_{g}'] = round(f['summary']['eo_tpr_gap'], 3)
        out[f'abstention_gap_{g}'] = round(f['summary']['abstention_gap'], 3)
        out[f'DI_{g}'] = round(f['summary']['disparate_impact_acc'], 3)
    return out


### Cell 7 — Run all selected experiments on the real data

In [ ]:
# ============================================================================
# CELL 7 — Run the real-data experiments
# ============================================================================
RESULTS_DIR = f'{PROJECT_DIR}/results'; os.makedirs(RESULTS_DIR, exist_ok=True)

QUICK = True
SELECT = ['LMDHG-segmented', 'LMDHG-continuous', 'HHAR-phones-LOSO', 'HHAR-phones-LODO']
FOCUS = SELECT[0]  # which run Cells 8–9 visualise

BASE = dict(epochs=30, seed=42)
RUNS = {
    'HHAR-phones-LOSO': dict(dataset='hhar', data_dir=HHAR_LOCAL, hhar_stream='Phones', protocol='LOSO',
                             critical_classes=('stairsdown',)),
    'HHAR-phones-LODO': dict(dataset='hhar', data_dir=HHAR_LOCAL, hhar_stream='Phones', protocol='LODO',
                             critical_classes=('stairsdown',)),
    'HHAR-watch-LOSO':  dict(dataset='hhar', data_dir=HHAR_LOCAL, hhar_stream='Watch', protocol='LOSO',
                             critical_classes=('stairsdown',)),
    'HHAR-phones-LOSDO': dict(dataset='hhar', data_dir=HHAR_LOCAL, hhar_stream='Phones', protocol='LOSDO',
                              critical_classes=('stairsdown',)),
    # isolated gestures, time-normalised to 64 frames
    'LMDHG-segmented':  dict(dataset='lmdhg', data_dir=LMDHG_LOCAL, lmdhg_mode='segmented', protocol='LOSO',
                             win=64, lmdhg_T=64, patch=8, patch_stride=4, max_rot_deg=15, epochs=60, batch_size=64,
                             decision_alpha=0.3),
    # unsegmented stream: same window/stride as your GMN run (60 / 5) so the numbers are comparable.
    # Real LMDHG: ~137k windows, ~2.3 GB RAM (float16) — fits a standard Colab runtime.
    'LMDHG-continuous': dict(dataset='lmdhg', data_dir=LMDHG_LOCAL, lmdhg_mode='continuous', protocol='LOSO',
                             win=60, stride=5, patch=10, patch_stride=5, max_rot_deg=15, epochs=40, min_purity=0.8,
                             decision_alpha=0.3),
}

ALL = {}
for name in SELECT:
    c = EPICConfig(**{**BASE, **RUNS[name]})
    c.out_dir = f'{RESULTS_DIR}/{name}'
    if QUICK:
        c.max_folds = 2
        if not (c.dataset == 'lmdhg' and c.lmdhg_mode == 'segmented'):   # segmented LMDHG is cheap: keep 60 epochs
            c.epochs = min(c.epochs, 10)
    ensure_data(c)
    d = load_dataset(c, CACHE_DIR)
    print(f'\n######## {name}: {d["X"].shape}  ({d["X"].nbytes / 1e6:.0f} MB in RAM) ########')
    r = run_experiment(c, d, tag=name)
    s = summarise(r)
    json.dump(dict(config=asdict(c), summary=s, quick=QUICK), open(f'{c.out_dir}/summary.json', 'w'), indent=2, default=str)
    print(json.dumps(s, indent=2))
    ALL[name] = dict(cfg=c, data=d, res=r, summary=s)
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

cfg, data, res, summary = (ALL[FOCUS][k] for k in ('cfg', 'data', 'res', 'summary'))


### Cell 7b — Targeted run (instead of Cell 7): only the protocols you need, from cached data

In [ ]:
# ============================================================================
# CELL 7b — TARGETED RUN (use INSTEAD of Cell 7 when you only need specific protocols)
# Minimal path:  Cells 1, 2, 2b (definitions), 3, 4, 5, 6  ->  this cell  ->  optionally 8 / 11.
# Uses the cached HHAR windows in Drive (no download, no CSV parsing).
# ============================================================================
RESULTS_DIR = f'{PROJECT_DIR}/results'; os.makedirs(RESULTS_DIR, exist_ok=True)

TARGET = {   # name: EPICConfig overrides
    'HHAR-phones-LOSDO': dict(dataset='hhar', data_dir=HHAR_LOCAL, hhar_stream='Phones', protocol='LOSDO',
                              critical_classes=('stairsdown',), epochs=15, max_folds=None),
    'HHAR-phones-LODO':  dict(dataset='hhar', data_dir=HHAR_LOCAL, hhar_stream='Phones', protocol='LODO',
                              critical_classes=('stairsdown',), epochs=15, max_folds=None),   # includes samsungold fold
}

ALL = globals().get('ALL', {})
for name, ov in TARGET.items():
    c = EPICConfig(**ov); c.out_dir = f'{RESULTS_DIR}/{name}'
    ensure_data(c)
    d = load_dataset(c, CACHE_DIR)
    r = run_experiment(c, d, tag=name, verbose=False)
    s = summarise(r)
    os.makedirs(c.out_dir, exist_ok=True)
    json.dump(dict(config=asdict(c), summary=s), open(f'{c.out_dir}/summary.json', 'w'), indent=2, default=str)
    print(f'\n{name}:', json.dumps(s, indent=2))
    print(r['folds'][['fold', 'macro_f1', 'coverage', 'answer_rate', 'selective_acc', 'flops_ratio']].round(3).to_string(index=False))
    ALL[name] = dict(cfg=c, data=d, res=r, summary=s)
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

FOCUS = list(TARGET)[0]
cfg, data, res, summary = (ALL[FOCUS][k] for k in ('cfg', 'data', 'res', 'summary'))
QUICK = False


### Cell 8 — Figures and fairness report

In [ ]:
# ============================================================================
# CELL 8 — Figures: per-group fairness, risk–coverage, gates, training dynamics
# ============================================================================
import matplotlib.pyplot as plt

P = res['preds']; K = len(data['classes'])
fig, ax = plt.subplots(2, 2, figsize=(14, 9))

# (a) per-subject macro-F1 and abstention rate — heterogeneity made visible
if 'user' in res['fair']:
    t = res['fair']['user']['table'].sort_values('macro_f1')
    ax[0, 0].bar(t.group.astype(str), t.macro_f1, color='#4C72B0', label='macro-F1 (all windows)')
    ax[0, 0].bar(t.group.astype(str), t.selective_acc, color='none', edgecolor='#DD8452', lw=2, label='selective acc.')
    ax2 = ax[0, 0].twinx(); ax2.plot(t.group.astype(str), t.abstain_rate, 'k.--', label='abstention rate')
    ax2.set_ylabel('abstention rate'); ax[0, 0].set_ylim(0, 1.05)
    ax[0, 0].set_title('(a) Per-subject performance and abstention'); ax[0, 0].legend(loc='lower right')

# (b) risk–coverage curve (selective prediction)
o = np.argsort(-P.conf.values); c = (P.y.values == P.pred.values)[o]
cov = np.arange(1, len(c) + 1) / len(c); risk = 1 - np.cumsum(c) / np.arange(1, len(c) + 1)
ax[0, 1].plot(cov, risk, color='#4C72B0')
ax[0, 1].axvline(P.answered.mean(), color='#DD8452', ls='--', label=f'operating point ({P.answered.mean():.2f})')
ax[0, 1].set_xlabel('coverage'); ax[0, 1].set_ylabel('selective risk'); ax[0, 1].legend()
ax[0, 1].set_title('(b) Risk–coverage')

# (c) which feature groups survive across folds (learned data minimisation)
G = res['arts'][0]['model'].group_names
kept = np.zeros(len(G))
for a in res['arts']:
    kept += np.isin(G, a['gate']['feature_groups_kept'])
ax[1, 0].barh(G, kept / len(res['arts']), color='#55A868')
ax[1, 0].set_xlabel('fraction of folds in which the group is kept'); ax[1, 0].set_title('(c) Learned feature selection')

# (d) budget dynamics
for i, a in enumerate(res['arts']):
    h = a['hist']
    ax[1, 1].plot(h.epoch, h.hard_cost, color='#C44E52', alpha=.5, label='FLOPs ratio' if i == 0 else None)
    ax[1, 1].plot(h.epoch, h.val_f1, color='#4C72B0', alpha=.5, label='val macro-F1' if i == 0 else None)
ax[1, 1].axhline(cfg.budget, color='#C44E52', ls=':', label='budget')
ax[1, 1].set_xlabel('epoch'); ax[1, 1].legend(); ax[1, 1].set_title('(d) Accuracy–compute co-optimisation')
plt.tight_layout(); plt.savefig(f'{cfg.out_dir}/EPIC_{cfg.dataset}_{cfg.protocol}_overview.png', dpi=200); plt.show()

for g, f in res['fair'].items():
    print(f'\nFairness over {g}:'); print(pd.Series(f['summary']).to_string())
print('\nDecision mix:'); print(P.decision.value_counts(normalize=True).round(3).to_string())


### Cell 9 — HCAI demonstration

In [ ]:
# ============================================================================
# CELL 9 — HCAI in action: decision records, counterfactual recourse,
#          audited human overrides (honest vs. biased reviewer), model card
# ============================================================================
art = res['arts'][0]; model = art['model'].to(DEVICE).eval(); fold = art['fold']
X = torch.from_numpy(data['X']); y = torch.from_numpy(data['y']).long(); classes = data['classes']
Pf = res['preds'][res['preds'].fold == fold['name']].reset_index(drop=True)
ledger = HashChainLedger(); fp = model_fingerprint(model)

def decision_record(i_row):
    r = Pf.iloc[i_row]; x = X[int(r.idx)]
    with torch.no_grad():
        o = model(x[None].to(DEVICE)); p = torch.softmax(o['logits'] / art['T'], -1)[0].cpu().numpy()
    S = art['cp'].sets(p[None])[0]
    occ = group_occlusion(model, x, int(r.pred))
    top = [model.group_names[j] for j in np.argsort(-occ)[:3] if occ[j] > 0]
    M = model.head.M; cls = int(r.pred)
    j = cls * M + int(o['sim'][0, cls * M:(cls + 1) * M].argmax()); src = int(model.head.proto_src[j])
    proto = (f"prototype {j} = training window #{src} (subject {pseudonymise(data['meta'].user.iloc[src], cfg.pseudonym_salt)}, "
             f"'{classes[y[src]]}')") if src >= 0 else None
    rec = dict(model=fp, subject=pseudonymise(r.user, cfg.pseudonym_salt), device_model=r.model,
               decision=r.decision, pred=classes[int(r.pred)], confidence=float(p.max()),
               set=[classes[k] for k in np.where(S)[0]], reasons=[s for s in r.reasons.split(';') if s],
               top_groups=top, prototype=proto)
    ledger.append(rec)
    return rec

for d in ['auto', 'abstain_ambiguous', 'abstain_ood', 'human_review']:
    rows = np.where(Pf.decision == d)[0]
    if len(rows):
        print(f'--- {d} ---\n' + explain_text(decision_record(int(rows[0]))) + '\n')

# Counterfactual recourse for one misclassified window
wrong = np.where(Pf.y != Pf.pred)[0]
if len(wrong):
    r = Pf.iloc[int(wrong[0])]
    tr = torch.as_tensor(fold['tr'][:20000])
    bounds = (X[tr].float().amin((0, 1)), X[tr].float().amax((0, 1)))
    cf = counterfactual(model, X[int(r.idx)], int(r.y), bounds, art['ood'])
    raw = data['spec']['raw_names']
    print(f"Counterfactual: '{classes[int(r.pred)]}' -> '{classes[int(r.y)]}': success={cf['success']} "
          f"p={cf['p_target']:.2f} plausible={cf['plausible']}")
    print('  largest changes:', sorted(zip(raw, cf['channel_change'].round(3)), key=lambda t: -t[1])[:3])

# Override loop: humans review every non-automated window; the auditor decides whether to learn
rng = np.random.default_rng(0)
rev = Pf[~Pf.answered | (Pf.decision == 'human_review')].copy()
if len(rev) >= 10:
    with torch.no_grad():
        prv = torch.softmax(predict(model, X[rev.idx.values])[0] / art['T'], -1).numpy()
    in_marg = (1 - prv) <= art['cp'].q_marg                       # marginal conformal set membership
    gcol = 'device' if Pf.device.nunique() > 1 else 'user'          # LMDHG: one sensor -> audit by subject
    anchor = np.asarray(fold['ca']); gA = data['meta'][gcol].iloc[anchor].values
    base = rev.assign(group=rev[gcol])
    honest = base.assign(y_human=np.where(rng.random(len(base)) < 0.05,
                                          rng.integers(0, len(classes), len(base)), base.y))
    victim = base.group.value_counts().index[0]                   # one group gets systematically relabelled
    biased = honest.copy(); vm = (biased.group == victim).values
    biased.loc[vm, 'y_human'] = (biased.loc[vm, 'y'] + 1) % len(classes)
    for name, ov in (('honest reviewer', honest), ('biased reviewer', biased)):
        ov = ov.assign(surprise=~in_marg[np.arange(len(ov)), ov.y_human.values])
        _, ok, checks, st = FeedbackAuditor().review(model, ov[['idx', 'y_human', 'group', 'surprise']], X, y,
                                                     fold['tr'], anchor, gA, len(classes), ledger)
        print(f'{name}: update {"ACCEPTED" if ok else "REJECTED"}  ' +
              ', '.join(f'{k}={v:.3f}{"" if passed else " ✗"}' for k, (v, passed) in checks.items()))
        if len(st):
            print(st.round(3).to_string(index=False))
print(f'\nLedger entries: {len(ledger.entries)} | chain intact: {ledger.verify()}')
ledger.entries[0]['record']['pred'] = 'tampered'
print(f'After tampering with entry 0: chain intact: {ledger.verify()}')

eff = {k: float(res['folds'][k].mean()) for k in ('params_dense', 'params_pruned', 'size_mb_dense', 'size_mb_pruned',
                                                    'flops_ratio', 'cpu_ms_dense', 'cpu_ms_pruned')}
card = model_card(cfg, data, summary, art['gate'], eff)
open(f'{cfg.out_dir}/MODEL_CARD_{cfg.dataset}_{cfg.protocol}.md', 'w').write(card)
print(card[:1500])


### Cell 10 — One-at-a-time ablations (the interaction study is Cells 19–20)

In [ ]:
# ============================================================================
# CELL 10 — Ablations
# ============================================================================
ABLATIONS = {
    'EPIC (full)':            {},
    'no gates (dense)':       dict(use_gates=False),
    'no group-DRO':           dict(use_dro=False),
    'no adversarial inv.':    dict(use_adv=False),
    'no consistency':         dict(use_consistency=False),
    'plain CE (no GCE)':      dict(gce_q=0.0),
    'marginal conformal':     dict(conformal_mode='marginal'),
    'LLM backbone (GPT-2)':   dict(backbone='llm', llm_name='gpt2', llm_layers=6, batch_size=64),
}
RUN_ABLATIONS = False        # set True: ~(#ablations x #folds) trainings — use a GPU runtime
QUICK_FOLDS = 3              # None for the full protocol (needed for the paper)

if RUN_ABLATIONS:
    rows = []
    for name, ov in ABLATIONS.items():
        if ov.get('backbone') == 'llm':
            import subprocess; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'peft'])
        c = copy.deepcopy(cfg)
        for k, v in ov.items():
            setattr(c, k, v)
        c.max_folds = QUICK_FOLDS
        r = run_experiment(c, data, tag=name.replace(' ', '_'), verbose=False)
        rows.append({'variant': name, **summarise(r)})
    table = pd.DataFrame(rows).set_index('variant')
    table.to_csv(f'{cfg.out_dir}/ablations_{cfg.dataset}_{cfg.protocol}.csv')
    display(table)


### Cell 11 — Memory / speed / accuracy benchmark (saved to Drive)

In [ ]:
# ============================================================================
# CELL 11 — Memory / speed / accuracy benchmark across all runs -> Drive
#   Accuracy : macro-F1, accuracy, majority-class baseline, worst-group F1, coverage, selective acc.
#   Memory   : parameters, model size (dense / pruned / int8), peak GPU memory (train, inference), peak RAM
#   Speed    : FLOPs ratio, CPU latency per window (dense / pruned / int8), GPU throughput, train time
# ============================================================================
def ms(S, k, f='{:.3f}'):
    return (f + ' ± ' + f).format(S[k].mean(), S[k].std(ddof=0)) if k in S and S[k].notna().any() else '—'

rows = []
for name, R in ALL.items():
    S, fair = R['res']['folds'], R['res']['fair']
    wg = min(v['summary']['worst_group_f1'] for v in fair.values()) if fair else np.nan
    rows.append({
        'run': name, 'folds': len(S),
        # accuracy
        'macro_F1': ms(S, 'macro_f1'), 'accuracy': ms(S, 'acc'), 'majority_baseline': ms(S, 'majority_baseline'),
        'worst_group_F1': f'{wg:.3f}', 'coverage(target {:.2f})'.format(1 - R['cfg'].alpha): ms(S, 'coverage'),
        'answer_rate': ms(S, 'answer_rate'), 'selective_acc': ms(S, 'selective_acc'), 'ECE': ms(S, 'ece'),
        # memory
        'params_dense': f"{S.params_dense.mean():,.0f}", 'params_pruned': f"{S.params_pruned.mean():,.0f}",
        'MB_dense': ms(S, 'size_mb_dense', '{:.2f}'), 'MB_pruned': ms(S, 'size_mb_pruned', '{:.2f}'),
        'MB_int8': ms(S, 'size_mb_int8', '{:.2f}'), 'int8_agreement': ms(S, 'int8_agreement'),
        'peak_GPU_MB_train': ms(S, 'peak_gpu_mb_train', '{:.0f}'), 'peak_GPU_MB_infer': ms(S, 'gpu_infer_mb_pruned', '{:.0f}'),
        'peak_RAM_MB': f"{S.peak_ram_mb.max():.0f}",
        # speed
        'FLOPs_ratio': ms(S, 'flops_ratio'), 'CPU_ms_dense': ms(S, 'cpu_ms_dense', '{:.2f}'),
        'CPU_ms_pruned': ms(S, 'cpu_ms_pruned', '{:.2f}'), 'CPU_ms_int8': ms(S, 'cpu_ms_int8', '{:.2f}'),
        'GPU_win/s_dense': ms(S, 'gpu_win_per_s_dense', '{:.0f}'), 'GPU_win/s_pruned': ms(S, 'gpu_win_per_s_pruned', '{:.0f}'),
        's/epoch': ms(S, 's_per_epoch', '{:.1f}'), 'train_min/fold': ms(S.assign(m=S.train_s / 60), 'm', '{:.1f}'),
    })
bench = pd.DataFrame(rows).set_index('run')
stamp = time.strftime('%Y%m%d-%H%M')
bench.to_csv(f'{RESULTS_DIR}/benchmark_{stamp}.csv')
try:
    open(f'{RESULTS_DIR}/benchmark_{stamp}.md', 'w').write(bench.T.to_markdown())
except ImportError:
    pass
open(f'{RESULTS_DIR}/benchmark_{stamp}.tex', 'w').write(bench.T.to_latex(escape=True))
print(f'Saved benchmark_{stamp}.csv / .md / .tex to {RESULTS_DIR}' + ('   (QUICK mode — not paper numbers)' if QUICK else ''))
display(bench.T)


### Cell 12 — Baseline zoo

In [ ]:
# ============================================================================
# CELL 12 — Baseline zoo (same splits, same calibration / trust layer, same efficiency counters)
# ============================================================================
class Standardize(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.register_buffer('mu', torch.zeros(C)); self.register_buffer('sd', torch.ones(C))

    def forward(self, x):
        return (x - self.mu) / self.sd


class BaselineNet(nn.Module):
    """Common wrapper: float32 cast, train-fold standardisation, returns dict(logits, z)."""
    def __init__(self, core, C, standardize=True):
        super().__init__()
        self.std = Standardize(C) if standardize else nn.Identity(); self.core = core

    def forward(self, x, group_mask=None, grl=None):
        logits, z = self.core(self.std(x.float()))
        return dict(logits=logits, z=z)


class DeepConvLSTM(nn.Module):                       # Ordóñez & Roggen, Sensors 2016
    def __init__(self, C, K, F_=64, H=128):
        super().__init__()
        self.conv = nn.Sequential(*[m for i in range(4) for m in (nn.Conv2d(1 if i == 0 else F_, F_, (5, 1)), nn.ReLU())])
        self.lstm = nn.LSTM(F_ * C, H, num_layers=2, batch_first=True, dropout=0.5)
        self.dp, self.fc = nn.Dropout(0.5), nn.Linear(H, K)

    def forward(self, x):
        h = self.conv(x.unsqueeze(1))                                   # [B,F,T',C]
        h = h.permute(0, 2, 1, 3).flatten(2)                            # [B,T',F*C]
        o, _ = self.lstm(h); z = o[:, -1]
        return self.fc(self.dp(z)), z


class TinyHAR(nn.Module):                            # Zhou et al., ISWC 2022 (re-implementation)
    def __init__(self, C, K, F_=24):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(1, F_, (5, 1), (1, 1), (2, 0)), nn.ReLU(), nn.BatchNorm2d(F_),
                                  nn.Conv2d(F_, F_, (5, 1), (2, 1), (2, 0)), nn.ReLU(), nn.BatchNorm2d(F_),
                                  nn.Conv2d(F_, F_, (5, 1), (1, 1), (2, 0)), nn.ReLU(), nn.BatchNorm2d(F_),
                                  nn.Conv2d(F_, F_, (5, 1), (2, 1), (2, 0)), nn.ReLU(), nn.BatchNorm2d(F_))
        self.cross = nn.TransformerEncoderLayer(F_, 2, 2 * F_, 0.1, batch_first=True)   # channel interaction
        self.fuse = nn.Linear(C * F_, 2 * F_)                                          # channel fusion
        self.lstm = nn.LSTM(2 * F_, 2 * F_, batch_first=True)                          # temporal
        self.att = nn.Linear(2 * F_, 1); self.fc = nn.Linear(2 * F_, K)

    def forward(self, x):
        h = self.conv(x.unsqueeze(1))                                   # [B,F,T',C]
        B, F_, T_, C = h.shape
        h = self.cross(h.permute(0, 2, 3, 1).reshape(B * T_, C, F_)).reshape(B, T_, C * F_)
        h, _ = self.lstm(F.relu(self.fuse(h)))
        w = torch.softmax(self.att(h), 1)
        z = h[:, -1] + (w * h).sum(1)
        return self.fc(z), z


class _ResBlock1d(nn.Module):
    def __init__(self, ci, co, stride=1, dil=1, k=5):
        super().__init__()
        self.c1 = nn.Conv1d(ci, co, k, stride, dil * (k // 2), dil); self.b1 = nn.BatchNorm1d(co)
        self.c2 = nn.Conv1d(co, co, k, 1, dil * (k // 2), dil); self.b2 = nn.BatchNorm1d(co)
        self.sk = nn.Identity() if (ci == co and stride == 1) else nn.Conv1d(ci, co, 1, stride)

    def forward(self, x):
        return F.relu(self.sk(x) + self.b2(self.c2(F.relu(self.b1(self.c1(x))))))


class ResNet1D(nn.Module):
    def __init__(self, C, K, w=64):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(C, w, 7, 1, 3), nn.BatchNorm1d(w), nn.ReLU(),
                                 _ResBlock1d(w, w), _ResBlock1d(w, 2 * w, 2), _ResBlock1d(2 * w, 2 * w, 2))
        self.fc = nn.Linear(2 * w, K)

    def forward(self, x):
        z = self.net(x.transpose(1, 2)).mean(-1); return self.fc(z), z


class TCN(nn.Module):                                # Bai et al. 2018 style (non-causal for classification)
    def __init__(self, C, K, w=64):
        super().__init__()
        self.inp = nn.Conv1d(C, w, 1)
        self.net = nn.Sequential(*[_ResBlock1d(w, w, 1, d) for d in (1, 2, 4, 8)]); self.fc = nn.Linear(w, K)

    def forward(self, x):
        z = self.net(self.inp(x.transpose(1, 2))).mean(-1); return self.fc(z), z


class PlainViT(nn.Module):                           # same tokenizer as EPIC, no gates / prototypes / HCAI training
    def __init__(self, C, K, P, S, d=128, L=4, H=4):
        super().__init__()
        self.tok = PatchTokenizer(C, P, S, d)
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, H, 4 * d, 0.1, batch_first=True, norm_first=True), L)
        self.ln, self.fc = nn.LayerNorm(d), nn.Linear(d, K)

    def forward(self, x):
        z = self.ln(self.enc(self.tok(x)).mean(1)); return self.fc(z), z


class BiLSTM(nn.Module):
    def __init__(self, C, K, H=128):
        super().__init__()
        self.lstm = nn.LSTM(C, H, 2, batch_first=True, bidirectional=True, dropout=0.2); self.fc = nn.Linear(2 * H, K)

    def forward(self, x):
        o, _ = self.lstm(x); z = o.mean(1); return self.fc(z), z


class DDNet(nn.Module):                              # Yang et al., ACM MM Asia 2019 (JCD + two-scale motion)
    def __init__(self, T, J, K, F_=16):
        super().__init__()
        self.J = J; iu = torch.triu_indices(J, J, 1); self.register_buffer('iu', iu)
        def emb(ci):
            return nn.Sequential(nn.Conv1d(ci, 2 * F_, 1), nn.BatchNorm1d(2 * F_), nn.LeakyReLU(0.2),
                                 nn.Conv1d(2 * F_, F_, 3, 1, 1), nn.BatchNorm1d(F_), nn.LeakyReLU(0.2),
                                 nn.Conv1d(F_, F_, 1), nn.BatchNorm1d(F_), nn.LeakyReLU(0.2))
        self.e_jcd, self.e_slow, self.e_fast = emb(iu.shape[1]), emb(3 * J), emb(3 * J)
        def blk(ci, co):
            return nn.Sequential(nn.Conv1d(ci, co, 3, 1, 1), nn.BatchNorm1d(co), nn.LeakyReLU(0.2),
                                 nn.Conv1d(co, co, 3, 1, 1), nn.BatchNorm1d(co), nn.LeakyReLU(0.2))
        self.body = nn.Sequential(blk(3 * F_, 2 * F_), nn.MaxPool1d(2), blk(2 * F_, 4 * F_), nn.MaxPool1d(2), blk(4 * F_, 8 * F_))
        self.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(8 * F_, K))

    def forward(self, x):
        B, T, _ = x.shape; p = x.view(B, T, self.J, 3)
        jcd = torch.cdist(p, p)[:, :, self.iu[0], self.iu[1]]            # [B,T,J(J-1)/2]
        slow = torch.diff(x, dim=1, prepend=x[:, :1])
        fast = torch.diff(x[:, ::2], dim=1, prepend=x[:, :1]).repeat_interleave(2, 1)[:, :T]
        h = torch.cat([self.e_jcd(jcd.transpose(1, 2)), self.e_slow(slow.transpose(1, 2)), self.e_fast(fast.transpose(1, 2))], 1)
        z = self.body(h).amax(-1); return self.fc(z), z


class AGCNBlock(nn.Module):                          # adaptive graph: A = I + B (learned) + C(x) (data-dependent), Shi et al. CVPR 2019
    def __init__(self, ci, co, V, stride=1, e=16):
        super().__init__()
        self.B = nn.Parameter(torch.zeros(V, V)); self.th, self.ph = nn.Conv2d(ci, e, 1), nn.Conv2d(ci, e, 1)
        self.gc = nn.Conv2d(ci, co, 1); self.bn = nn.BatchNorm2d(co)
        self.tc = nn.Sequential(nn.Conv2d(co, co, (9, 1), (stride, 1), (4, 0)), nn.BatchNorm2d(co))
        self.res = nn.Identity() if (ci == co and stride == 1) else nn.Sequential(nn.Conv2d(ci, co, 1, (stride, 1)), nn.BatchNorm2d(co))
        self.register_buffer('I', torch.eye(V))

    def forward(self, x):                                                # x [B,C,T,V]
        Cx = torch.softmax(torch.einsum('bctv,bctw->bvw', self.th(x), self.ph(x)) / x.shape[2], -1)
        A = self.I + self.B + Cx
        h = F.relu(self.bn(self.gc(torch.einsum('bctv,bvw->bctw', x, A))))
        return F.relu(self.tc(h) + self.res(x))


class AGCNLite(nn.Module):
    def __init__(self, J, K):
        super().__init__()
        self.J = J; self.bn0 = nn.BatchNorm1d(3 * J)
        self.blocks = nn.Sequential(AGCNBlock(3, 64, J), AGCNBlock(64, 64, J), AGCNBlock(64, 128, J, 2), AGCNBlock(128, 128, J))
        self.fc = nn.Linear(128, K)

    def forward(self, x):
        B, T, _ = x.shape
        x = self.bn0(x.transpose(1, 2)).transpose(1, 2).reshape(B, T, self.J, 3).permute(0, 3, 1, 2)
        z = self.blocks(x).mean((2, 3)); return self.fc(z), z


# ---------------- original CoordMind / ViT-MLP-LLM-FS-Or (from the user's notebook, unchanged logic) ----------
class _CMPatch(nn.Module):
    def __init__(self, num_coords, patch_size, embed_dim):
        super().__init__(); self.num_coords, self.patch_size = num_coords, patch_size
        self.projection = nn.Linear(num_coords * patch_size, embed_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, 1000, embed_dim))

    def forward(self, x):
        B, T, _ = x.shape; n = T // self.patch_size
        x = x[:, :n * self.patch_size].reshape(B, n, self.patch_size * self.num_coords)
        return self.projection(x) + self.pos_embedding[:, :n]


class CoordMindOriginal(nn.Module):
    """Faithful port of ViTMLPLLMFSOr as in ForensicAI2.ipynb (ViT -> layer-gumbel selection (unused by the
    logits, as in the original) -> GCN representor -> cross-modal fusion with a learned prompt -> LoRA -> MLP)."""
    def __init__(self, C, K, patch=8, d=256, L=6, H=8, llm=768):
        super().__init__()
        self.patch = _CMPatch(C, patch, d)
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(d, H, 4 * d, 0.1, batch_first=True, norm_first=True) for _ in range(L)])
        self.norm = nn.LayerNorm(d)
        self.layer_scores = nn.Parameter(torch.randn(L)); self.temperature = 5.0
        self.gcn = nn.ModuleList([nn.Linear(d, d) for _ in range(3)]); self.gln = nn.ModuleList([nn.LayerNorm(d) for _ in range(3)])
        self.rel = nn.MultiheadAttention(d, 4, batch_first=True)
        self.msg = nn.Sequential(nn.Linear(2 * d, d), nn.ReLU(), nn.Linear(d, d)); self.outp = nn.Linear(d, d)
        self.vproj = nn.Sequential(nn.Linear(d, llm), nn.LayerNorm(llm), nn.GELU())
        self.xattn = nn.MultiheadAttention(llm, 8, batch_first=True)
        self.gate = nn.Sequential(nn.Linear(2 * llm, llm), nn.Sigmoid()); self.fnorm = nn.LayerNorm(llm)
        self.lora_down, self.lora_up = nn.Linear(llm, 16, bias=False), nn.Linear(16, llm, bias=False)
        nn.init.zeros_(self.lora_up.weight)
        self.cls = nn.Sequential(nn.Linear(llm, llm // 2), nn.ReLU(), nn.Dropout(0.1), nn.Linear(llm // 2, K))
        self.task_prompt = nn.Parameter(torch.randn(1, 10, llm))

    def anneal_temperature(self, epoch, max_epochs):
        self.temperature = max(1.0, 5.0 * (1 - epoch / max_epochs))

    def forward(self, x):
        h = self.patch(x)
        for l in self.layers:
            h = l(h)
        h = self.norm(h)
        f = F.normalize(h, dim=-1); A = (f @ f.transpose(1, 2) > 0.3).float() + torch.eye(h.shape[1], device=h.device)
        g = h
        for lin, ln in zip(self.gcn, self.gln):
            g = F.relu(ln(lin((A / A.sum(-1, keepdim=True)) @ g)))
        att, _ = self.rel(g, g, g)
        r = self.outp(g + att + self.msg(torch.cat([g, A @ g], -1)))
        v = self.vproj(r); t = self.task_prompt.expand(x.shape[0], -1, -1)
        fu, _ = self.xattn(t, v, v); gw = self.gate(torch.cat([t, fu], -1))
        e = self.fnorm(gw * fu + (1 - gw) * t)
        e = e + 0.5 * self.lora_up(self.lora_down(e))
        z = e.mean(1); return self.cls(z), z


class HARNetAdapter(nn.Module):
    """SSL foundation model (Yuan et al., npj Digit. Med. 2024): ResNet pretrained on 700k person-days of
    UK-Biobank wrist accelerometry; input = 3-axis acceleration in g at 30 Hz, 5 s (harnet5).
    Uses ONLY the accelerometer; windows shorter than 5 s are tiled (prefer win=250 @ 50 Hz = exactly 5 s)."""
    def __init__(self, K, fs, T, linear_probe=False):
        super().__init__()
        hub = torch.hub.load('OxWearables/ssl-wearables', 'harnet5', class_num=K, pretrained=True, trust_repo=True)
        self.fs = fs
        self.feat = getattr(hub, 'feature_extractor', None)
        if self.feat is None:                                           # fallback: use the hub model's own head
            print('[HARNet5] no .feature_extractor attribute; fine-tuning the full hub model')
            self.feat, self.fc = hub, nn.Identity(); return
        if linear_probe:
            for p in self.feat.parameters():
                p.requires_grad = False
        with torch.no_grad():
            d = self.feat(torch.zeros(1, 3, 150)).flatten(1).shape[1]
        self.fc = nn.Linear(d, K)

    def forward(self, x):
        a = x[..., :3].transpose(1, 2) / 9.81                            # m/s^2 -> g
        L30 = max(8, int(round(a.shape[-1] * 30 / self.fs)))
        a = F.interpolate(a, size=L30, mode='linear', align_corners=False)
        if L30 < 150:
            a = a.repeat(1, 1, math.ceil(150 / L30))
        a = a[..., :150]
        z = self.feat(a).flatten(1); return self.fc(z), z


BASELINES = {   # name: (builder(C, K, T, cfg), datasets it applies to, standardise inputs?)
    'DeepConvLSTM': (lambda C, K, T, c: DeepConvLSTM(C, K), ('hhar', 'synthetic'), True),
    'TinyHAR':      (lambda C, K, T, c: TinyHAR(C, K), ('hhar', 'synthetic', 'lmdhg'), True),
    'ResNet1D':     (lambda C, K, T, c: ResNet1D(C, K), ('hhar', 'synthetic', 'lmdhg'), True),
    'TCN':          (lambda C, K, T, c: TCN(C, K), ('hhar', 'synthetic', 'lmdhg'), True),
    'ViT':          (lambda C, K, T, c: PlainViT(C, K, c.patch, c.patch_stride, c.d_model, c.n_layers, c.n_heads), ('hhar', 'synthetic', 'lmdhg'), True),
    'BiLSTM':       (lambda C, K, T, c: BiLSTM(C, K), ('lmdhg',), True),
    'CoordMind':    (lambda C, K, T, c: CoordMindOriginal(C, K, patch=8), ('hhar', 'synthetic', 'lmdhg'), True),
    'HARNet5':      (lambda C, K, T, c: HARNetAdapter(K, c.fs, T), ('hhar',), False),
    'DDNet':        (lambda C, K, T, c: DDNet(T, C // 3, K), ('lmdhg',), True),
    'AGCN':         (lambda C, K, T, c: AGCNLite(C // 3, K), ('lmdhg',), True),
}


def train_baseline(name, cfg: EPICConfig, data: Dict, tr_idx, va_idx, verbose=False):
    set_seed(cfg.seed, cfg.deterministic)
    X = torch.from_numpy(data['X']); y = torch.from_numpy(data['y']).long()
    C, T, K = X.shape[2], X.shape[1], len(data['classes'])
    builder, _, stdz = BASELINES[name]
    model = BaselineNet(builder(C, K, T, cfg), C, stdz).to(DEVICE)
    if stdz:
        sub = np.random.default_rng(cfg.seed).choice(tr_idx, size=min(20000, len(tr_idx)), replace=False)
        xs = X[sub].float().reshape(-1, C)
        model.std.mu.copy_(xs.mean(0).to(DEVICE)); model.std.sd.copy_(xs.std(0).clamp_min(1e-6).to(DEVICE))
    cnt = np.bincount(data['y'][tr_idx], minlength=K).astype(float)
    cw = torch.tensor(1 / np.sqrt(np.maximum(cnt, 1)), dtype=torch.float32, device=DEVICE); cw = cw / cw.mean()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.weight_decay)
    nb = sum(1 for i in range(0, len(tr_idx) - 1, cfg.batch_size) if len(tr_idx[i:i + cfg.batch_size]) >= 2)  # exact batches/epoch
    total_steps = cfg.epochs * max(1, nb) + 1
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=total_steps, pct_start=0.1)
    n_steps = 0
    use_amp = DEVICE.type == 'cuda'; scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    best, state, bad, hist, t0 = -1, None, 0, [], time.perf_counter()
    tri = torch.as_tensor(tr_idx)
    for ep in range(cfg.epochs):
        model.train(); te = time.perf_counter()
        if hasattr(model.core, 'anneal_temperature'):
            model.core.anneal_temperature(ep, cfg.epochs)
        perm = tri[torch.randperm(len(tri))]
        for i in range(0, len(perm) - 1, cfg.batch_size):
            b = perm[i:i + cfg.batch_size]
            if len(b) < 2:
                continue
            xb, yb = X[b].to(DEVICE).float(), y[b].to(DEVICE)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(model(xb)['logits'].float(), yb, weight=cw if cfg.class_balance else None)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            if n_steps < total_steps - 1:                              # never step past the schedule
                sched.step(); n_steps += 1
        f1 = f1_score(y[va_idx].numpy(), predict(model, X, idx=va_idx)[0].argmax(1).numpy(), average='macro')
        hist.append(dict(epoch=ep, val_f1=f1, epoch_s=time.perf_counter() - te))
        if verbose:
            print(f'  [{name}] ep {ep:02d} val-F1 {f1:.3f}')
        if f1 > best:
            best, state, bad = f1, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= cfg.patience:
                break
    model.load_state_dict(state)
    hist = pd.DataFrame(hist); hist.attrs['train_s'] = time.perf_counter() - t0
    hist.attrs['peak_gpu_mb_train'] = torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == 'cuda' else float('nan')
    return model, hist


def run_fold_baseline(name, cfg: EPICConfig, data: Dict, fold: Dict, verbose=False):
    """Identical evaluation to run_fold (temperature, conformal, OOD, guardrails, decision policy):
    the trust layer is model-agnostic, so differences isolate what EPIC's training/architecture add."""
    X = torch.from_numpy(data['X']); y = torch.from_numpy(data['y']).long(); meta = data['meta']
    rng = np.random.default_rng(cfg.split_seed)
    ca = rng.permutation(fold['ca']); va, cal = ca[: len(ca) // 2], ca[len(ca) // 2:]
    model, hist = train_baseline(name, cfg, data, fold['tr'], va, verbose)
    lv, _ = predict(model, X, idx=va); T = fit_temperature(lv, y[va])
    lc, zc = predict(model, X, idx=cal); pc = torch.softmax(lc / T, -1).numpy()
    cp = ConformalPredictor(cfg.alpha, cfg.conformal_mode).fit(pc, y[cal].numpy())
    sub = rng.choice(fold['tr'], size=min(20000, len(fold['tr'])), replace=False)
    _, ztr = predict(model, X, idx=sub)
    ood = EmbeddingOOD().fit(ztr.numpy(), y[sub].numpy(), zc.numpy(), cfg.ood_quantile)
    lt, zt = predict(model, X, idx=fold['te']); pt = torch.softmax(lt / T, -1).numpy()
    yt = y[fold['te']].numpy(); pred = pt.argmax(1)
    S = cp.sets(pt); is_ood = ood.score(zt.numpy()) > ood.thr
    bc = physical_guardrails(data['X'][fold['te']], data['spec'])
    Sd = S if cfg.decision_alpha is None else cp.sets(pt, alpha=cfg.decision_alpha)
    dec, why = decide(Sd, is_ood, bc, pred, data['classes'], cfg.critical_classes)
    sweep = alpha_sweep_metrics(cp, pt, yt, is_ood, bc, pred, data['classes'], cfg.critical_classes)
    m = meta.iloc[fold['te']].reset_index(drop=True)
    df = pd.DataFrame(dict(idx=fold['te'], user=m.user, device=m.device, model=m.model, y=yt, pred=pred,
                           stream=m['stream'] if 'stream' in m else 'na',
                           conf=pt.max(1), set_size=S.sum(1), covered=S[np.arange(len(yt)), yt],
                           decision=dec, reasons=[';'.join(r) for r in why], fold=fold['name']))
    df['answered'] = df.decision.isin(['auto', 'human_review'])
    model.cpu()
    eff = dict(params_dense=count_params(model), params_pruned=count_params(model),
               size_mb_dense=model_size_mb(model), size_mb_pruned=model_size_mb(model), flops_ratio=1.0,
               flops_dense=measured_flops(_LogitsOnly(model), X.shape[1], X.shape[2]),
               cpu_ms_dense=cpu_latency_ms(model, X.shape[1], X.shape[2]),
               train_s=hist.attrs['train_s'], peak_gpu_mb_train=hist.attrs['peak_gpu_mb_train'],
               s_per_epoch=float(hist.epoch_s.mean()))
    eff['flops_pruned'], eff['cpu_ms_pruned'] = eff['flops_dense'], eff['cpu_ms_dense']
    model.to(DEVICE)
    fs = dict(fold=fold['name'], acc=accuracy_score(yt, pred), macro_f1=f1_score(yt, pred, average='macro'),
              majority_baseline=float(np.bincount(yt).max() / len(yt)),
              ece=expected_calibration_error(pt, yt), temperature=T, coverage=float(df.covered.mean()),
              target_coverage=1 - cfg.alpha, avg_set=float(df.set_size.mean()), answer_rate=float(df.answered.mean()),
              selective_acc=float((df[df.answered].y == df[df.answered].pred).mean()) if df.answered.any() else np.nan,
              aurc=aurc(pt.max(1), pred == yt), ood_rate=float(is_ood.mean()), **sweep, **eff)
    return df, fs, dict(model=model, T=T, cp=cp, ood=ood, hist=hist, fold=fold)


### Cell 13 — Resumable grid: protocols × models × seeds × folds

In [ ]:
# ============================================================================
# CELL 13 — Resumable experiment grid: protocols x models x seeds x folds
#   * every finished (protocol, model, seed, fold) is appended to grid/results.csv immediately,
#     and per-window predictions to grid/preds/....
# ============================================================================
if 'BASELINES' not in globals():
    raise RuntimeError('Run Cell 12 (baseline zoo) before Cell 13. Order: Cells 1-6, 2b, 12, 13, 14, 16.')
RESULTS_DIR = globals().get('RESULTS_DIR', f'{PROJECT_DIR}/results'); os.makedirs(RESULTS_DIR, exist_ok=True)
GRID_DIR = f'{RESULTS_DIR}/grid'; os.makedirs(f'{GRID_DIR}/preds', exist_ok=True)
GRID_LOG = f'{GRID_DIR}/results.jsonl'      # one JSON object per finished fold: safe to append, schema-free
GRID_CSV = f'{GRID_DIR}/results.csv'        # snapshot written by load_results() for convenience


def load_results() -> pd.DataFrame:
    if not os.path.exists(GRID_LOG):
        return pd.DataFrame()
    R = pd.read_json(GRID_LOG, lines=True, dtype={'fold': str})
    R.to_csv(GRID_CSV, index=False)
    return R

# 5 s windows at 50 Hz (= exactly harnet5's 150 samples at 30 Hz) for EVERY HHAR model, so all models see
# identical windows. Set HHAR_WINDOW = {} to keep the 2.56 s windows of Cell 7.
HHAR_WINDOW = dict(win=250, stride=125, patch=25, patch_stride=12)
PROTOCOLS = {
    'HHAR-P-LOSO': dict(dataset='hhar', data_dir=HHAR_LOCAL, hhar_stream='Phones', protocol='LOSO'),
    'HHAR-P-LODO': dict(dataset='hhar', data_dir=HHAR_LOCAL, hhar_stream='Phones', protocol='LODO'),
    'HHAR-W-LOSO': dict(dataset='hhar', data_dir=HHAR_LOCAL, hhar_stream='Watch', protocol='LOSO'),
    'HHAR-W-LODO': dict(dataset='hhar', data_dir=HHAR_LOCAL, hhar_stream='Watch', protocol='LODO'),
    'HHAR-P-LOSDO': dict(dataset='hhar', data_dir=HHAR_LOCAL, hhar_stream='Phones', protocol='LOSDO'),
    'HHAR-P2W':    dict(dataset='hhar', data_dir=HHAR_LOCAL, protocol='XDEV_P2W', _both=True),
    'HHAR-W2P':    dict(dataset='hhar', data_dir=HHAR_LOCAL, protocol='XDEV_W2P', _both=True),
    'LMDHG-seg':   dict(dataset='lmdhg', data_dir=LMDHG_LOCAL, lmdhg_mode='segmented', protocol='LOSO',
                        win=64, lmdhg_T=64, patch=8, patch_stride=4, max_rot_deg=15, epochs=60, batch_size=64,
                        decision_alpha=0.3),
    'LMDHG-cont':  dict(dataset='lmdhg', data_dir=LMDHG_LOCAL, lmdhg_mode='continuous', protocol='LOSO',
                        win=60, stride=5, patch=10, patch_stride=5, max_rot_deg=15, epochs=40, min_purity=0.8,
                        decision_alpha=0.3),
}
# Paper-grade grid (expensive — see time estimate printed below). Trim lists to run in several sessions.
GRID_PROTOCOLS = ['HHAR-P-LOSO', 'HHAR-P-LODO', 'HHAR-P-LOSDO', 'HHAR-P2W', 'LMDHG-seg', 'LMDHG-cont']
GRID_MODELS = ['EPIC', 'ViT', 'DeepConvLSTM', 'TinyHAR', 'ResNet1D', 'TCN', 'CoordMind', 'HARNet5', 'DDNet', 'AGCN', 'BiLSTM']
GRID_SEEDS = [0, 1, 2]
GRID_EPOCHS = None          # e.g. 15 to shorten everything; None = protocol default (30 / 40 / 60)
GRID_MAX_FOLDS = None       # e.g. 3 for a pilot
RUN_GRID = False            # set True to execute


def grid_cfg(pname: str, seed: int, **over) -> EPICConfig:
    ov = {k: v for k, v in PROTOCOLS[pname].items() if not k.startswith('_')}
    if ov['dataset'] == 'hhar':
        ov = {**ov, **HHAR_WINDOW}
    c = EPICConfig(**{**dict(epochs=30), **ov, **over}); c.seed = seed
    if GRID_EPOCHS:
        c.epochs = GRID_EPOCHS
    c.max_folds = GRID_MAX_FOLDS; c.out_dir = GRID_DIR
    return c


def grid_load(pname: str, c: EPICConfig) -> Dict:
    both = bool(PROTOCOLS[pname].get('_both'))
    ensure_data(c, both)
    if c.dataset == 'hhar' and both:
        return load_hhar_both(c, CACHE_DIR)
    return load_dataset(c, CACHE_DIR)


def _done_keys():
    r = load_results()
    return set() if r.empty else set(zip(r.protocol, r.model, r.seed, r.fold))


def run_grid(protocols=None, models=None, seeds=None, model_cfg: Optional[Dict] = None, tag_suffix=''):
    """model_cfg: extra EPICConfig overrides for EPIC (e.g. dict(budget=0.15)); tag_suffix names that variant."""
    protocols, models, seeds = protocols or GRID_PROTOCOLS, models or GRID_MODELS, seeds or GRID_SEEDS
    done = _done_keys()
    for pname in protocols:
        c0 = grid_cfg(pname, seeds[0], **(model_cfg or {}))
        data = grid_load(pname, c0)
        folds = make_folds(data['meta'], c0)
        ok_models = [m for m in models if m == 'EPIC' or c0.dataset in BASELINES[m][1]]
        todo = [(s, f['name'], m) for s in seeds for f in folds for m in ok_models
                if (pname, m + tag_suffix, s, f['name']) not in done]
        print(f'\n##### {pname}: {len(folds)} folds x {len(ok_models)} models x {len(seeds)} seeds -> {len(todo)} runs to do')
        broken = set()                                           # models that failed to build (e.g. no internet)
        for seed in seeds:
            c = grid_cfg(pname, seed, **(model_cfg or {}))
            for f in folds:
                for mname in ok_models:
                    key = (pname, mname + tag_suffix, seed, f['name'])
                    if key in done or mname in broken:
                        continue
                    t0 = time.perf_counter()
                    try:
                        if mname == 'EPIC':
                            df, fs, _ = run_fold(c, data, f, verbose=False)
                        else:
                            df, fs, _ = run_fold_baseline(mname, c, data, f)
                    except Exception as e:                       # e.g. HARNet hub download failure: log and skip model
                        print(f'  ✗ {key}: {type(e).__name__}: {e} — skipping {mname} for {pname}')
                        broken.add(mname); continue
                    row = dict(protocol=pname, model=mname + tag_suffix, seed=seed, **fs,
                               budget=c.budget if mname == 'EPIC' else np.nan, wall_s=time.perf_counter() - t0)
                    with open(GRID_LOG, 'a') as fh:
                        fh.write(json.dumps({k: (float(v) if isinstance(v, (np.floating, np.integer)) else v)
                                             for k, v in row.items()}, default=str) + '\n')
                    df.assign(protocol=pname, model_name=mname + tag_suffix, seed=seed).to_csv(
                        f'{GRID_DIR}/preds/{pname}__{mname}{tag_suffix}__s{seed}.csv.gz', mode='a', index=False,
                        header=not os.path.exists(f'{GRID_DIR}/preds/{pname}__{mname}{tag_suffix}__s{seed}.csv.gz'))
                    done.add(key)
                    print(f'  ✓ {pname} | {mname + tag_suffix:14s} | seed {seed} | {f["name"]:28s} | F1 {fs["macro_f1"]:.3f} '
                          f'| MFLOPs {fs["flops_pruned"] / 1e6:8.2f} | {time.perf_counter() - t0:6.0f}s')
                if DEVICE.type == 'cuda':
                    torch.cuda.empty_cache()
        del data


def estimate_grid_hours(sec_per_run: Dict[str, float] = None):
    """Rough cost from your QUICK benchmark (train_min/fold x epochs/10). Edit to taste."""
    base = sec_per_run or {'HHAR-P-LOSO': 600, 'HHAR-P-LODO': 700, 'HHAR-P-LOSDO': 450, 'HHAR-W-LOSO': 300, 'HHAR-W-LODO': 300,
                           'HHAR-P2W': 900, 'HHAR-W2P': 600, 'LMDHG-seg': 90, 'LMDHG-cont': 1000}
    nf = {'HHAR-P-LOSO': 9, 'HHAR-P-LODO': 4, 'HHAR-P-LOSDO': 9, 'HHAR-W-LOSO': 9, 'HHAR-W-LODO': 2, 'HHAR-P2W': 1, 'HHAR-W2P': 1,
          'LMDHG-seg': 5, 'LMDHG-cont': 5}
    tot = 0
    for p in GRID_PROTOCOLS:
        ds = PROTOCOLS[p]['dataset']
        nm = sum(1 for m in GRID_MODELS if m == 'EPIC' or ds in BASELINES[m][1])
        h = base[p] * nf[p] * nm * len(GRID_SEEDS) / 3600; tot += h
        print(f'  {p:12s} ~{h:5.1f} GPU-h  ({nf[p]} folds x {nm} models x {len(GRID_SEEDS)} seeds)')
    print(f'  TOTAL ~{tot:.0f} GPU-hours on a T4 (EPIC ~1.5x a baseline; LSTM/GCN baselines vary).')


estimate_grid_hours()
if RUN_GRID:
    run_grid()


### Cell 14 — Statistics: Wilcoxon + Holm, Friedman + Nemenyi

In [ ]:
# ============================================================================
# CELL 14 — Statistics: mean ± std, paired Wilcoxon (Holm), Friedman + Nemenyi critical-difference diagram
#   Unit of analysis = held-out FOLD (seeds averaged within a fold first): seeds share the same test
#   subjects, so treating fold x seed as independent pairs would inflate n (pseudo-replication).
# ============================================================================
from scipy.stats import wilcoxon, friedmanchisquare, studentized_range
import matplotlib.pyplot as plt


def holm(pvals: Dict[str, float]) -> Dict[str, float]:
    items = sorted(pvals.items(), key=lambda t: t[1]); m = len(items); out, run = {}, 0.0
    for i, (k, p) in enumerate(items):
        run = max(run, min(1.0, (m - i) * p)); out[k] = run
    return out


def paired_tests(R: pd.DataFrame, metric='macro_f1', ref='EPIC'):
    rows = []
    blocks = R.groupby(['protocol', 'fold', 'model'])[metric].mean().unstack('model')   # seeds averaged per fold
    for prot in list(R.protocol.unique()) + ['POOLED']:
        B = blocks if prot == 'POOLED' else blocks.loc[prot]
        if ref not in B:
            continue
        pv = {}
        for m in [c for c in B.columns if c != ref]:
            d = (B[ref] - B[m]).dropna()
            if len(d) < 2 or np.allclose(d, 0):
                continue
            p = wilcoxon(d, zero_method='zsplit').pvalue
            r = d.rank(); rbc = ((r[d > 0].sum() - r[d < 0].sum()) / r.sum())          # rank-biserial effect size
            pv[m] = p
            rows.append(dict(protocol=prot, baseline=m, n_folds=len(d), mean_diff=d.mean(), median_diff=d.median(),
                             wins=int((d > 0).sum()), losses=int((d < 0).sum()), p_raw=p, rank_biserial=rbc,
                             min_attainable_p=2 / 2 ** len(d)))
        for k, v in holm(pv).items():
            for r_ in rows:
                if r_['protocol'] == prot and r_['baseline'] == k:
                    r_['p_holm'] = v
    return pd.DataFrame(rows)


def friedman_nemenyi(R: pd.DataFrame, metric='macro_f1', models=None, title='', fname=None):
    B = R.groupby(['protocol', 'fold', 'model'])[metric].mean().unstack('model')
    if models:
        B = B[[m for m in models if m in B]]
    B = B.dropna()
    k, n = B.shape[1], B.shape[0]
    if k < 3 or n < 3:
        print('Friedman needs >= 3 models and >= 3 blocks with complete results'); return None
    stat, p = friedmanchisquare(*[B[c] for c in B])
    ranks = B.rank(axis=1, ascending=False).mean().sort_values()
    q = studentized_range.ppf(0.95, k, np.inf) / np.sqrt(2)
    cd = q * np.sqrt(k * (k + 1) / (6 * n))
    print(f'Friedman chi2={stat:.2f}, p={p:.2g} over n={n} (protocol, fold) blocks, k={k} models | Nemenyi CD = {cd:.2f}')
    fig, ax = plt.subplots(figsize=(9, 0.45 * k + 1.2))
    ax.set_xlim(0.5, k + 0.5); ax.set_ylim(-1, k); ax.invert_xaxis(); ax.set_yticks([])
    for i, (m, rk) in enumerate(ranks.items()):
        ax.plot([rk, rk], [k - 1 - i, k], color='k', lw=.8); ax.text(rk, k - 1 - i - 0.1, f'{m} ({rk:.2f})', ha='center', va='top', fontsize=9)
    ax.plot([ranks.iloc[0], ranks.iloc[0] + cd], [k - 0.3, k - 0.3], color='#C44E52', lw=3)
    ax.text(ranks.iloc[0] + cd / 2, k - 0.15, f'CD = {cd:.2f}', ha='center', color='#C44E52')
    ax.set_xlabel('average rank (1 = best)'); ax.set_title(title or f'Critical difference — {metric}')
    plt.tight_layout()
    if fname:
        plt.savefig(fname, dpi=200)
    plt.show()
    return dict(chi2=stat, p=p, cd=cd, ranks=ranks)


def results_table(R: pd.DataFrame, metrics=('macro_f1', 'coverage', 'answer_rate', 'flops_pruned', 'params_pruned', 'cpu_ms_pruned')):
    g = R.groupby(['protocol', 'model'])
    T = pd.DataFrame({m: g[m].mean().map('{:.3f}'.format) + ' ± ' + g[m].std(ddof=0).map('{:.3f}'.format)
                      if m not in ('flops_pruned', 'params_pruned') else
                      (g[m].mean() / 1e6).map('{:.2f}M'.format) for m in metrics})
    return T


R = load_results()
if not R.empty:
    print(f'{len(R)} fold-level results | protocols {sorted(R.protocol.unique())} | models {sorted(R.model.unique())}')
    TAB = results_table(R); display(TAB)
    TAB.to_latex(f'{GRID_DIR}/table_main.tex', escape=True)
    WIL = paired_tests(R); display(WIL.round(4))
    WIL.to_csv(f"{GRID_DIR}/wilcoxon_holm.csv", index=False)
    FR = friedman_nemenyi(R, models=GRID_MODELS, fname=f'{GRID_DIR}/cd_diagram.png')
else:
    print('No grid results yet — run Cell 13 with RUN_GRID = True')


### Cell 15 — Accuracy–efficiency Pareto (budget sweep)

In [ ]:
# ============================================================================
# CELL 15 — Accuracy–efficiency Pareto: EPIC budget sweep vs. baselines
#   Every EPIC budget is a separate "model" in the same results.csv (EPIC-b0.15, ...), so it also enters
#   the statistics of Cell 14. FLOPs are counted by the same FlopCounterMode for all models.
# ============================================================================
BUDGETS = [0.15, 0.25, 0.35, 0.5, 1.0]
PARETO_PROTOCOLS = ['HHAR-P-LOSO', 'LMDHG-seg']
PARETO_SEEDS = [0, 1, 2]
RUN_PARETO = False

if RUN_PARETO:
    for b in BUDGETS:
        run_grid(PARETO_PROTOCOLS, ['EPIC'], PARETO_SEEDS,
                 model_cfg=dict(budget=b) if b < 1 else dict(use_gates=False), tag_suffix=f'-b{b}')


def pareto_front(x, y):
    o = np.argsort(x); best, keep = -np.inf, []
    for i in o:
        if y[i] > best:
            keep.append(i); best = y[i]
    return np.array(keep)


def plot_pareto(R, protocol, fname=None):
    G = R[R.protocol == protocol].groupby('model').agg(f1=('macro_f1', 'mean'), f1s=('macro_f1', 'std'),
                                                        flops=('flops_pruned', 'mean'), params=('params_pruned', 'mean'),
                                                        cpu=('cpu_ms_pruned', 'mean')).reset_index()
    if G.empty:
        return
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
    for a, (col, lab) in zip(ax, [('flops', 'MFLOPs / window (log)'), ('params', 'parameters (log)'), ('cpu', 'CPU ms / window (log)')]):
        xs = G[col] / (1e6 if col == 'flops' else 1)
        is_epic = G.model.str.startswith('EPIC')
        a.errorbar(xs[~is_epic], G.f1[~is_epic], yerr=G.f1s[~is_epic], fmt='o', color='#8C8C8C', capsize=3, label='baselines')
        a.errorbar(xs[is_epic], G.f1[is_epic], yerr=G.f1s[is_epic], fmt='s', color='#C44E52', capsize=3, label='EPIC (budget sweep)')
        for _, r in G.iterrows():
            a.annotate(r.model, ((r[col] / (1e6 if col == 'flops' else 1)), r.f1), fontsize=7, xytext=(3, 3), textcoords='offset points')
        fr = pareto_front(xs.values, G.f1.values)
        a.plot(xs.values[fr], G.f1.values[fr], '--', color='#4C72B0', lw=1, label='Pareto front')
        a.set_xscale('log'); a.set_xlabel(lab); a.set_ylabel('macro-F1'); a.grid(alpha=.3)
    ax[0].legend(); fig.suptitle(f'Accuracy–efficiency trade-off — {protocol}'); plt.tight_layout()
    if fname:
        plt.savefig(fname, dpi=200)
    plt.show()


R = load_results()
if not R.empty:
    for p in R.protocol.unique():
        plot_pareto(R, p, f'{GRID_DIR}/pareto_{p}.png')


### Cell 16 — Coverage gap under heterogeneity, α sweep

In [ ]:
# ============================================================================
# CELL 16 — Coverage gap under heterogeneity (the epistemic metric)
# ============================================================================
import matplotlib.pyplot as plt


def load_preds(protocol, model):
    fs = glob.glob(f'{GRID_DIR}/preds/{protocol}__{model}__s*.csv.gz')
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()


def coverage_gap_table(protocols=None, models=None, alpha=0.1):
    R = load_results(); rows = []
    for p in protocols or R.protocol.unique():
        for m in models or R[R.protocol == p].model.unique():
            P = load_preds(p, m)
            if P.empty:
                continue
            for gcol in ('user', 'model', 'stream'):
                if gcol not in P or P[gcol].nunique() < 2 and gcol != 'stream':
                    continue
                g = P.groupby(gcol).agg(coverage=('covered', 'mean'), answer=('answered', 'mean'), n=('covered', 'size'))
                for k, r in g.iterrows():
                    rows.append(dict(protocol=p, model_name=m, group_by=gcol, group=k, n=r.n, coverage=r.coverage,
                                     gap=(1 - alpha) - r.coverage, answer_rate=r.answer))
    return pd.DataFrame(rows)


def plot_alpha_sweep(R, protocol, fname=None):
    S = R[R.protocol == protocol]
    if S.empty:
        return
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
    for m, g in S.groupby('model'):
        st = dict(color='#C44E52', lw=2.5, zorder=5) if m == 'EPIC' else dict(alpha=.6)
        ax[0].plot([1 - a for a in ALPHA_SWEEP], [g[f'cov@{a}'].mean() for a in ALPHA_SWEEP], 'o-', label=m, **st)
        ax[1].plot(ALPHA_SWEEP, [g[f'answer@{a}'].mean() for a in ALPHA_SWEEP], 'o-', **st)
        ax[2].plot([g[f'answer@{a}'].mean() for a in ALPHA_SWEEP], [g[f'selacc@{a}'].mean() for a in ALPHA_SWEEP], 'o-', **st)
    ax[0].plot([0.65, 1], [0.65, 1], 'k:', label='target = achieved'); ax[0].set_xlabel('target coverage 1-α'); ax[0].set_ylabel('achieved coverage')
    ax[1].set_xlabel('decision α'); ax[1].set_ylabel('answer rate (auto + human-review)')
    ax[2].set_xlabel('answer rate'); ax[2].set_ylabel('selective accuracy')
    ax[0].legend(fontsize=7); fig.suptitle(f'Conformal coverage & selective prediction under heterogeneity — {protocol}')
    plt.tight_layout()
    if fname:
        plt.savefig(fname, dpi=200)
    plt.show()


R = load_results()
if not R.empty:
    CG = coverage_gap_table()
    CG.to_csv(f'{GRID_DIR}/coverage_gap.csv', index=False)
    summary_gap = CG.groupby(['protocol', 'model_name', 'group_by']).agg(
        mean_gap=('gap', 'mean'), worst_gap=('gap', 'max'), groups_below_target=('gap', lambda g: int((g > 0.02).sum())),
        n_groups=('gap', 'size')).round(3)
    display(summary_gap)
    for p in R.protocol.unique():
        plot_alpha_sweep(R, p, f'{GRID_DIR}/alpha_sweep_{p}.png')


### Cell 17 — TorchScript / mobile / ONNX export and latency

In [ ]:
# ============================================================================
# CELL 17 — Deployment export: pruned EPIC -> TorchScript (+ mobile lite) and ONNX; verified and timed
#   Phone timing: copy the .ptl to an Android phone and run PyTorch's speed_benchmark / ExecuTorch.
# ============================================================================
import subprocess
EXPORT_DIR = f'{RESULTS_DIR}/export'; os.makedirs(EXPORT_DIR, exist_ok=True)


def export_model(model: nn.Module, T: int, C: int, name: str, check_X: Optional[torch.Tensor] = None) -> Dict:
    m = _LogitsOnly(copy.deepcopy(model).cpu().eval()).eval()
    x = torch.randn(1, T, C)
    out = dict(name=name)
    # TorchScript (trace) + mobile lite interpreter
    with torch.no_grad():
        ts = torch.jit.trace(m, x, check_trace=False)
    ts_path = f'{EXPORT_DIR}/{name}.pt'; ts.save(ts_path); out['torchscript_mb'] = os.path.getsize(ts_path) / 1e6
    try:
        from torch.utils.mobile_optimizer import optimize_for_mobile
        try:
            lite = optimize_for_mobile(ts); out['mobile_optimized'] = True
        except Exception:                                         # build without XNNPACK: save un-optimised
            lite = ts; out['mobile_optimized'] = False
        lite._save_for_lite_interpreter(f'{EXPORT_DIR}/{name}.ptl')
        out['mobile_ptl_mb'] = os.path.getsize(f'{EXPORT_DIR}/{name}.ptl') / 1e6
    except Exception as e:
        out['mobile_ptl_mb'] = f'n/a ({type(e).__name__})'
    # ONNX
    try:
        onnx_path = f'{EXPORT_DIR}/{name}.onnx'
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            torch.onnx.export(m, (x,), onnx_path, input_names=['window'], output_names=['logits'],
                              dynamic_axes={'window': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17, dynamo=False)
        out['onnx_mb'] = os.path.getsize(onnx_path) / 1e6
        import onnxruntime as ort
        so = ort.SessionOptions(); so.intra_op_num_threads = 1
        sess = ort.InferenceSession(onnx_path, so, providers=['CPUExecutionProvider'])
        xn = (check_X[:64].float() if check_X is not None else torch.randn(8, T, C))
        ref = m(xn).detach().numpy(); got = sess.run(None, {'window': xn.numpy()})[0]
        out['onnx_max_abs_diff'] = float(np.abs(ref - got).max())
        out['onnx_argmax_agree'] = float((ref.argmax(1) == got.argmax(1)).mean())
        x1 = xn[:1].numpy()
        for _ in range(20):
            sess.run(None, {'window': x1})
        t0 = time.perf_counter()
        for _ in range(200):
            sess.run(None, {'window': x1})
        out['onnxruntime_ms_1thread'] = (time.perf_counter() - t0) / 200 * 1000
    except Exception as e:
        out['onnx_mb'] = f'n/a ({type(e).__name__}: {str(e)[:80]})'
    torch.set_num_threads(1)
    out['torchscript_ms_1thread'] = cpu_latency_ms(ts, T, C, reps=200)
    torch.set_num_threads(os.cpu_count() or 1)
    return out


RUN_EXPORT = True
if RUN_EXPORT:
    try:
        import onnxruntime  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime', 'onnxscript'])
    art = res['arts'][0]; Xd = torch.from_numpy(data['X'])
    T_, C_ = Xd.shape[1], Xd.shape[2]
    rows = [export_model(art['pruned'], T_, C_, f'EPIC_pruned_{cfg.dataset}', Xd[art['fold']['te']]),
            export_model(art['model'], T_, C_, f'EPIC_dense_{cfg.dataset}', Xd[art['fold']['te']])]
    EXP = pd.DataFrame(rows).set_index('name'); EXP.to_csv(f'{EXPORT_DIR}/export_summary.csv'); display(EXP.T)
    print(f'Files in {EXPORT_DIR}: {sorted(os.listdir(EXPORT_DIR))}')


### Cell 18 — Override-audit power and user-study stimuli

In [ ]:
# ============================================================================
# CELL 18 — HCAI evidence: (a) power of the override audit, (b) user-study stimuli export
# ============================================================================
from scipy.stats import fisher_exact


def audit_power(n_per_group=(10, 20, 40, 80), bias=(0.1, 0.2, 0.3, 0.5), base_rate=0.05, n_groups=10,
                beta=0.05, sims=500, seed=0):
    """P(FeedbackAuditor flags the biased group) when one of n_groups reviewers' groups has a fraction `bias`
    of its labels flipped. base_rate = surprise rate of honest labels (≈ alpha for a calibrated model).
    Use it to choose how many reviewed cases to accumulate before an update is audited."""
    rng = np.random.default_rng(seed); rows = []
    for n in n_per_group:
        for b in bias:
            hit = 0
            for _ in range(sims):
                s = rng.random((n_groups, n)) < base_rate
                s[0] |= rng.random(n) < b                       # group 0 is the victim
                ps = []
                for g in range(n_groups):
                    a = s[g]; r = np.delete(s, g, 0).ravel()
                    ps.append(fisher_exact([[a.sum(), n - a.sum()], [r.sum(), len(r) - r.sum()]], alternative='greater')[1])
                o = np.argsort(ps); flagged = set()
                for k, i in enumerate(o):                       # Holm
                    if ps[i] < beta / (n_groups - k):
                        flagged.add(i)
                    else:
                        break
                hit += 0 in flagged
            rows.append(dict(n_per_group=n, bias=b, power=hit / sims))
    return pd.DataFrame(rows).pivot(index='n_per_group', columns='bias', values='power')


POW = audit_power(sims=300)
print('Detection power of the override audit (rows: reviewed cases per group; cols: fraction of biased labels)')
display(POW.round(2))
POW.to_csv(f'{RESULTS_DIR}/audit_power.csv')


def export_study_stimuli(res, data, cfg, n_per_cell=12, seed=0, fname=None):
    """Balanced trial set for a within-subject study. Cells = AI correct/incorrect x decision type.
    Each trial carries what each condition shows: C0 label only | C1 + calibrated confidence & conformal set |
    C2 + prototype ('this looks like that') | C3 + counterfactual ('what would change the decision')."""
    art = res['arts'][0]; model = art['model'].to(DEVICE).eval(); fold = art['fold']
    X = torch.from_numpy(data['X']); y = torch.from_numpy(data['y']).long(); classes = data['classes']
    P = res['preds'][res['preds'].fold == fold['name']].reset_index(drop=True)
    P['ai_correct'] = P.y == P.pred
    rng = np.random.default_rng(seed); trials = []
    tr = torch.as_tensor(fold['tr'][:20000]); bounds = (X[tr].float().amin((0, 1)), X[tr].float().amax((0, 1)))
    for (ok, dec), g in P.groupby(['ai_correct', P.decision.isin(['auto', 'human_review'])]):
        for i in rng.choice(len(g), size=min(n_per_cell, len(g)), replace=False):
            r = g.iloc[i]; x = X[int(r.idx)]
            with torch.no_grad():
                o = model(x[None].to(DEVICE)); p = torch.softmax(o['logits'] / art['T'], -1)[0].cpu().numpy()
            S = art['cp'].sets(p[None])[0]; M = model.head.M; c = int(r.pred)
            j = c * M + int(o['sim'][0, c * M:(c + 1) * M].argmax()); src = int(model.head.proto_src[j])
            alt = int(np.argsort(-p)[1])
            cf = counterfactual(model, x, alt, bounds, art['ood'], steps=150)
            raw = data['spec']['raw_names']; top = np.argsort(-cf['channel_change'])[:2]
            trials.append(dict(trial_id=f'T{len(trials):03d}', window_idx=int(r.idx), true_label=classes[int(r.y)],
                               ai_label=classes[c], ai_correct=bool(ok), decision=r.decision,
                               C1_confidence=round(float(p.max()), 3), C1_set=[classes[k] for k in np.where(S)[0]],
                               C2_prototype_window=src, C2_prototype_label=classes[int(y[src])] if src >= 0 else None,
                               C3_counterfactual_target=classes[alt], C3_valid=bool(cf['success'] and cf['plausible']),
                               C3_changes=[raw[t] for t in top]))
    T_ = pd.DataFrame(trials).sample(frac=1, random_state=seed).reset_index(drop=True)
    if fname:
        T_.to_json(fname, orient='records', indent=1)
    return T_


RUN_STIMULI = True
if RUN_STIMULI:
    STIM = export_study_stimuli(res, data, cfg, n_per_cell=8, fname=f'{RESULTS_DIR}/user_study_stimuli.json')
    print(f'{len(STIM)} trials -> {RESULTS_DIR}/user_study_stimuli.json'); display(STIM.head())


### Cell 19 — Component study: 2^(5-1) factorial + feature-selection controls

In [ ]:
# ============================================================================
# CELL 19 — Component study: what each part of EPIC-HAR contributes, alone and together
#   Requires Cells 1-6, 2b, 12 and 13 (with RUN_GRID = False, GRID_EPOCHS = None, GRID_MAX_FOLDS = None, i.e. the
#   settings of the grid, so the full configuration reproduces the grid's EPIC runs). Resumable like Cell 13.
#
#   Five training components, each on (+1) or off (-1):
#     F  feature selection: learned Hard-Concrete gates on input feature groups (pillar A)
#     S  structural sparsity: learned gates on heads / neurons / layers (pillar B)
#     D  group-DRO over training subjects
#     A  adversarial subject / device invariance (gradient reversal)
#     C  consistency between two device-physics views (JSD)
#   A 2^(5-1) fractional factorial with generator C = F*S*D*A (resolution V): 16 configurations per
#   (fold, seed). Every main effect and every two-factor interaction is estimable and none is aliased
#   with another main effect or two-factor interaction (each is aliased only with a 3- or 4-factor term).
#
#   The trust layer is post hoc, so each trained model is also calibrated with MARGINAL conformal sets
#   (the default is class-conditional): training components x calibration mode at no extra training cost.
# ============================================================================
import itertools
from sklearn.feature_selection import mutual_info_classif

# Stop at once if Cell 4 comes from an older notebook (the fix for configurations without learned gates is missing).
if 'gate_has_grad' not in train_model.__code__.co_varnames:
    raise RuntimeError('Cell 4 is from an older notebook. Open EPIC_HAR_framework_v6.ipynb and run Cells 1-6, 2b, 12 and 13 '
                       'again before Cell 19; otherwise the configurations without learned gates fail on the GPU.')

FACTORS = ['F', 'S', 'D', 'A', 'C']
FACTOR_NAMES = {'F': 'feature selection', 'S': 'structural sparsity', 'D': 'group-DRO',
                'A': 'adversarial invariance', 'C': 'view consistency'}
FACT_DIR = f'{RESULTS_DIR}/factorial'; os.makedirs(FACT_DIR, exist_ok=True)
FACT_LOG = os.environ.get('EPIC_FACT_LOG') or f'{FACT_DIR}/factorial.jsonl'   # one file per worker on Kaggle

FACT_PROTOCOLS = ['LMDHG-seg']        # suggested order: 'LMDHG-seg' (~20 min/seed), 'HHAR-P2W' (~2 h/seed),
                                      # 'HHAR-P-LOSDO' (~8 h/seed; split over sessions, it resumes)
FACT_SEEDS = [0, 1, 2]
FACT_CONTROLS = True                  # FS-random and FS-filter (FS-none always runs)
FACT_EXTRAS = False                   # also 'no GCE' and 'no sensor dropout' (one factor at a time)
FACT_MAX_HOURS = 3.0                  # stop cleanly before starting a run once this much time has passed
FACT_FOLDS = None                     # e.g. [0, 1, 2] to do only these fold indices this session (None = all)
RUN_FACTORIAL = False


FULL_CONFIG = 'FSDAC:+++++'


def fraction_design():
    """16 runs of the 2^(5-1) design, the full configuration (all +1) first."""
    runs = []
    for f, s, d, a in itertools.product([1, -1], repeat=4):
        runs.append(dict(F=f, S=s, D=d, A=a, C=f * s * d * a))
    return runs


def config_name(lv: Dict) -> str:
    return 'FSDAC:' + ''.join('+' if lv[k] > 0 else '-' for k in FACTORS)


def factor_overrides(lv: Dict) -> Dict:
    return dict(feature_gates=lv['F'] > 0, struct_gates=lv['S'] > 0, use_dro=lv['D'] > 0,
                use_adv=lv['A'] > 0, use_consistency=lv['C'] > 0)


def fact_plan(controls=True, extras=False):
    """(name, factor levels or None, selection method, config overrides, reference config for k)."""
    plan = [(config_name(lv), lv, 'learned' if lv['F'] > 0 else 'none', factor_overrides(lv), None)
            for lv in fraction_design()]
    on = dict(F=1, S=1, D=1, A=1, C=1)
    none = dict(F=-1, S=1, D=1, A=1, C=1)                      # not in the fraction (F*S*D*A = -1)
    alone = dict(F=1, S=-1, D=1, A=1, C=1)                     # feature selection alone, held to the budget
    plan.append(('FS-none', none, 'none', factor_overrides(none), None))
    plan.append(('FS-alone', alone, 'learned', factor_overrides(alone), None))
    if controls:
        # family 1 (deployed model): same k as the learned gates of the full configuration
        plan.append(('FS-random', None, 'random', factor_overrides(on), FULL_CONFIG))
        plan.append(('FS-filter', None, 'filter', factor_overrides(on), FULL_CONFIG))
        # family 2 (selection alone): same k as FS-alone, which must cut inputs to ~budget
        plan.append(('FS-alone-random', None, 'random', factor_overrides(alone), 'FS-alone'))
        plan.append(('FS-alone-filter', None, 'filter', factor_overrides(alone), 'FS-alone'))
    if extras:
        plan.append(('no-GCE', None, 'learned', {**factor_overrides(on), 'gce_q': 0.0}, None))
        plan.append(('no-sensor-dropout', None, 'learned', {**factor_overrides(on), 'sensor_dropout': 0.0}, None))
    return plan


def filter_groups(data: Dict, tr_idx, k: int, seed: int) -> List[str]:
    """Classical filter selection: rank feature groups by mutual information between the label and
    per-window mean and standard deviation of each lifted channel (training fold only)."""
    names, ch2g, gnames, _ = build_channel_groups(data['spec'])
    sub = np.random.default_rng(seed).choice(tr_idx, size=min(4000, len(tr_idx)), replace=False)
    with torch.no_grad():
        Z = FeatureLift(data['spec'])(torch.from_numpy(data['X'][sub]).float())
    feats = torch.cat([Z.mean(1), Z.std(1)], 1).numpy()
    mi = mutual_info_classif(np.nan_to_num(feats), data['y'][sub], random_state=seed)
    C = Z.shape[-1]; ch = np.maximum(mi[:C], mi[C:])
    g = np.zeros(len(gnames))
    for c, gi in enumerate(ch2g):
        g[gi] = max(g[gi], ch[c])
    return [gnames[i] for i in np.argsort(-g)[:k]]


def random_groups(data: Dict, k: int, seed: int, fold_name: str) -> List[str]:
    _, _, gnames, _ = build_channel_groups(data['spec'])
    h = int(hashlib.md5(f'{fold_name}|{seed}'.encode()).hexdigest()[:8], 16)
    return list(np.random.default_rng(h).choice(gnames, size=k, replace=False))


def trust_extras(cfg, data, fold, df, art) -> Dict:
    """Coverage of the same trained model and calibration split under class-conditional AND marginal sets."""
    X = torch.from_numpy(data['X'])
    lt, _ = predict(art['model'], X, idx=fold['te']); pt = torch.softmax(lt / art['T'], -1).numpy()
    yt = data['y'][fold['te']]
    out = {}
    for mode in ('class', 'marginal'):
        S = art['cp'].sets(pt, mode=mode); cov = S[np.arange(len(yt)), yt].astype(float)
        out[f'cov_{mode}'] = float(cov.mean()); out[f'set_{mode}'] = float(S.sum(1).mean())
        per_class = [cov[yt == c].mean() for c in np.unique(yt) if (yt == c).sum() >= 10]
        out[f'worst_class_cov_{mode}'] = float(min(per_class)) if per_class else np.nan
    users = [g for _, g in df.groupby('user') if len(g) >= 20]
    if len(users) > 1:
        out['worst_user_f1'] = float(min(f1_score(g.y, g.pred, average='macro') for g in users))
    return out


def gate_fields(rep: Dict) -> Dict:
    out = dict(groups_kept=rep['feature_groups_kept'], n_groups_kept=rep['n_groups_kept'], n_groups=rep['n_groups'],
               sensors_needed=rep['sensors_needed'], hard_cost_ratio=rep['hard_cost_ratio'])
    if 'layers' in rep:
        out.update(layers_on=sum(l['on'] for l in rep['layers']), heads_on=sum(l['heads'] for l in rep['layers']),
                   neurons_on=sum(l['neurons'] for l in rep['layers']))
    return out


def load_factorial() -> pd.DataFrame:
    return pd.read_json(FACT_LOG, lines=True, dtype={'fold': str}) if os.path.exists(FACT_LOG) else pd.DataFrame()


def run_factorial(protocols=None, seeds=None, controls=None, extras=None):
    protocols, seeds = protocols or FACT_PROTOCOLS, seeds or FACT_SEEDS
    t_start = time.perf_counter()
    plan = fact_plan(FACT_CONTROLS if controls is None else controls, FACT_EXTRAS if extras is None else extras)
    R = load_factorial()
    done = set() if R.empty else set(zip(R.protocol, R.config, R.seed, R.fold))
    refs = {r for *_, r in plan if r}
    kept_k = {} if R.empty else {(p, c, s, f): int(k) for p, c, s, f, k in
                                 zip(R.protocol, R.config, R.seed, R.fold, R.n_groups_kept) if c in refs}
    for pname in protocols:
        c0 = grid_cfg(pname, seeds[0]); data = grid_load(pname, c0); folds = make_folds(data['meta'], c0)
        if FACT_FOLDS is not None:
            folds = [folds[i] for i in FACT_FOLDS if i < len(folds)]
        todo = sum((pname, n, s, f['name']) not in done for s in seeds for f in folds for n, *_ in plan)
        print(f'\n##### {pname}: {len(folds)} folds x {len(plan)} configurations x {len(seeds)} seeds -> {todo} runs to do')
        for seed in seeds:
            for f in folds:
                for name, lv, method, over, ref in plan:
                    key = (pname, name, seed, f['name'])
                    if key in done:
                        continue
                    if (time.perf_counter() - t_start) / 3600 > FACT_MAX_HOURS:
                        print(f'\nStopped after {FACT_MAX_HOURS} h as planned. Everything finished is saved; '
                              f'run Cell 19 again in a new session to continue.'); return
                    over = dict(over); k = None
                    if method in ('random', 'filter'):
                        k = kept_k.get((pname, ref, seed, f['name']))
                        if k is None:
                            print(f'  - {name} skipped for {f["name"]} (needs {ref} first)'); continue
                        over['fixed_groups'] = (random_groups(data, k, seed, f['name']) if method == 'random'
                                                else filter_groups(data, f['tr'], k, seed))
                    c = grid_cfg(pname, seed, **over); set_seed(seed, c.deterministic)
                    t0 = time.perf_counter()
                    try:
                        df, fs, art = run_fold(c, data, f, verbose=False)
                        extra = trust_extras(c, data, f, df, art)
                    except Exception as e:
                        print(f'  ✗ {key}: {type(e).__name__}: {e}'); continue
                    g = gate_fields(art['gate'])
                    row = dict(protocol=pname, config=name, seed=seed, fs_method=method,
                               **{k_: (lv[k_] if lv else None) for k_ in FACTORS}, k_target=k, k_ref=ref,
                               fixed_groups=over.get('fixed_groups'), gce_q=c.gce_q, sensor_dropout=c.sensor_dropout,
                               **fs, **extra, **g, wall_s=time.perf_counter() - t0)
                    with open(FACT_LOG, 'a') as fh:
                        fh.write(json.dumps({k_: (float(v) if isinstance(v, (np.floating, np.integer)) else v)
                                             for k_, v in row.items()}, default=str) + '\n')
                    done.add(key)
                    if name in refs:
                        kept_k[(pname, name, seed, f['name'])] = g['n_groups_kept']
                    print(f'  ✓ {pname} | {name:12s} | seed {seed} | {f["name"][:28]:28s} | F1 {fs["macro_f1"]:.3f} '
                          f'| groups {g["n_groups_kept"]:>2}/{g["n_groups"]} | FLOPs {fs["flops_ratio"]:.2f} '
                          f'| {time.perf_counter() - t0:5.0f}s')
                    del art
                    if DEVICE.type == 'cuda':
                        torch.cuda.empty_cache()
        del data


def estimate_factorial_hours():
    """From the grid's own EPIC timings (results.jsonl): configurations without the second view run ~0.6x."""
    G = load_results(); plan = fact_plan(FACT_CONTROLS, FACT_EXTRAS)
    w = np.mean([0.6 if (lv and lv['C'] < 0) else 1.0 for _, lv, _, _, _ in plan])
    for p in FACT_PROTOCOLS:
        e = G[(G.protocol == p) & (G.model == 'EPIC')] if not G.empty else G
        if e.empty:
            print(f'  {p}: no EPIC timing in the grid log yet'); continue
        per_seed = e.groupby('seed').wall_s.sum().mean() * len(plan) * w / 3600
        print(f'  {p:13s} ~{per_seed:4.1f} GPU-h per seed x {len(FACT_SEEDS)} seeds = {per_seed * len(FACT_SEEDS):5.1f} GPU-h')


print('Plan per (fold, seed):', [n for n, *_ in fact_plan(FACT_CONTROLS, FACT_EXTRAS)])
estimate_factorial_hours()
if RUN_FACTORIAL:
    run_factorial()


### Cell 20 — Effects, interactions and feature-selection analysis

In [ ]:
import itertools
import math
import numpy as np
import pandas as pd
from scipy import stats

FACTORS = ['F', 'S', 'D', 'A', 'C']
FACTOR_NAMES = {'F': 'feature selection', 'S': 'structural sparsity', 'D': 'group-DRO',
                'A': 'adversarial invariance', 'C': 'view consistency'}
TERMS = FACTORS + [a + b for a, b in itertools.combinations(FACTORS, 2)]
FULL = 'FSDAC:+++++'
ALL_FRACTION = ['FSDAC:' + ''.join('+' if v > 0 else '-' for v in (f, s, d, a, f * s * d * a))
                for f, s, d, a in itertools.product([1, -1], repeat=4)]


def holm(p):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p); out = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * p[i])); out[i] = run
    return out


def design_rows(R: pd.DataFrame, protocol: str) -> pd.DataFrame:
    d = R[(R.protocol == protocol) & R.F.notna() & R.fs_method.isin(['learned', 'none'])
          & ~R.config.isin(['no-GCE', 'no-sensor-dropout'])].copy()
    d['block'] = d.seed.astype(str) + '|' + d.fold.astype(str)
    for f in FACTORS:
        d[f] = d[f].astype(float)
    return d


def term_matrix(d: pd.DataFrame) -> np.ndarray:
    cols = [d[t].values if len(t) == 1 else d[t[0]].values * d[t[1]].values for t in TERMS]
    return np.column_stack(cols)


def effects(R: pd.DataFrame, protocol: str, response: str) -> pd.DataFrame:
    d = design_rows(R, protocol).dropna(subset=[response])
    if d.empty or d.block.nunique() < 2:
        return pd.DataFrame()
    B = pd.get_dummies(d.block).values.astype(float)
    Xt = term_matrix(d); X = np.column_stack([B, Xt]); y = d[response].values.astype(float)
    if np.linalg.matrix_rank(X) < X.shape[1]:          # some of the 16 configurations missing: not estimable
        missing = sorted({config for config in ALL_FRACTION} - set(d.config))
        print(f'[{protocol}] effects not estimable yet; missing configurations: {missing}')
        return pd.DataFrame()
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta; dof = len(y) - np.linalg.matrix_rank(X)
    s2 = resid @ resid / max(dof, 1); cov = s2 * np.linalg.pinv(X.T @ X)
    b = beta[B.shape[1]:]; se = np.sqrt(np.diag(cov)[B.shape[1]:])
    t = b / np.where(se > 0, se, np.nan); p = 2 * stats.t.sf(np.abs(t), max(dof, 1)); tq = stats.t.ppf(0.975, max(dof, 1))
    # non-parametric check on the balanced fraction: per-block contrast for each term
    fr = d[d.config.str.startswith('FSDAC:') & (d.F * d.S * d.D * d.A == d.C)]
    wil, sign = [], []
    for j, term in enumerate(TERMS):
        xs = fr[term] if len(term) == 1 else fr[term[0]] * fr[term[1]]
        con = (fr.assign(_x=xs.values).groupby('block')
               .apply(lambda g: g[response][g._x > 0].mean() - g[response][g._x < 0].mean(), include_groups=False).dropna())
        nz = con[con != 0]
        wil.append(stats.wilcoxon(nz).pvalue if len(nz) >= 2 else np.nan)
        sign.append(float((np.sign(con) == np.sign(2 * b[j])).mean()) if len(con) else np.nan)
    out = pd.DataFrame(dict(term=TERMS, kind=['main'] * 5 + ['interaction'] * 10, effect=2 * b,
                            ci_lo=2 * (b - tq * se), ci_hi=2 * (b + tq * se), p=p, p_holm=holm(p),
                            p_wilcoxon=wil, sign_agreement=sign))
    w = np.array(wil, float); ok = ~np.isnan(w); wh = np.full(len(w), np.nan); wh[ok] = holm(w[ok])
    out['p_wilcoxon_holm'] = wh
    out['label'] = [FACTOR_NAMES[t] if len(t) == 1 else f'{FACTOR_NAMES[t[0]]} x {FACTOR_NAMES[t[1]]}' for t in TERMS]
    out.insert(0, 'response', response); out.insert(0, 'protocol', protocol)
    out['n_blocks'] = d.block.nunique(); out['n_runs'] = len(d); out['resid_dof'] = dof
    out['resid_sd'] = float(np.sqrt(s2))
    return out


def surface(eff: pd.DataFrame):
    """Fitted response (relative to the block mean) at any on/off setting of the five components."""
    b = dict(zip(eff.term, eff.effect / 2))
    def f(on: dict):
        x = {k: 1.0 if on.get(k) else -1.0 for k in FACTORS}
        return sum(b[t] * (x[t] if len(t) == 1 else x[t[0]] * x[t[1]]) for t in TERMS)
    return f


def shapley(eff: pd.DataFrame) -> pd.DataFrame:
    """Exact Shapley attribution of (all on) - (all off) on the fitted surface. With +-1 coding a pairwise
    interaction takes the same value at all-on and all-off, so the Shapley value equals the main effect;
    interactions show up as the difference between the gain of a component added ALONE to the bare model
    and its gain when removed LAST from the full model (the usual leave-one-out ablation)."""
    f = surface(eff); n = len(FACTORS); rows = []
    v = {S: f({k: k in S for k in FACTORS}) for r in range(n + 1) for S in itertools.combinations(FACTORS, r)}
    v = {frozenset(k): val for k, val in v.items()}
    for i in FACTORS:
        others = [k for k in FACTORS if k != i]; phi = 0.0
        for r in range(n):
            w = math.factorial(r) * math.factorial(n - r - 1) / math.factorial(n)
            for S in itertools.combinations(others, r):
                phi += w * (v[frozenset(S) | {i}] - v[frozenset(S)])
        alone = v[frozenset([i])] - v[frozenset()]
        last = v[frozenset(FACTORS)] - v[frozenset(others)]
        rows.append(dict(term=i, label=FACTOR_NAMES[i], shapley=phi, gain_alone=alone, gain_last=last))
    out = pd.DataFrame(rows)
    out.attrs['total'] = v[frozenset(FACTORS)] - v[frozenset()]
    out.attrs['sum_alone'] = out.gain_alone.sum()
    return out


def synergy(eff: pd.DataFrame) -> dict:
    m = eff[eff.kind == 'main']; it = eff[eff.kind == 'interaction']
    sh = shapley(eff)
    return dict(total=sh.attrs['total'], sum_alone=sh.attrs['sum_alone'],
                synergy=sh.attrs['total'] - sh.attrs['sum_alone'],
                interaction_share=float(it.effect.abs().sum() / (m.effect.abs().sum() + it.effect.abs().sum())),
                n_sig_interactions=int((it.p_holm < 0.05).sum()))


FS_FAMILIES = {
    'deployed': {'learned': FULL, 'none': 'FS-none', 'random': 'FS-random', 'filter': 'FS-filter'},
    'alone': {'learned': 'FS-alone', 'none': 'FSDAC:--+++', 'random': 'FS-alone-random', 'filter': 'FS-alone-filter'},
}


def fs_controls(R: pd.DataFrame, protocol: str,
                metrics=('macro_f1', 'ece', 'cov_class', 'flops_ratio', 'n_groups_kept', 'cpu_ms_pruned')):
    """Learned feature selection against no selection, random and filter selection at the same k.
    'deployed': all components on, k from the full configuration; 'alone': structural gates off and the
    feature gates held to the budget by themselves."""
    rows = []
    for fam, cmap in FS_FAMILIES.items():
        d = R[(R.protocol == protocol) & R.config.isin(cmap.values())].copy()
        if d.empty:
            continue
        d['method'] = d.config.map({v: k for k, v in cmap.items()})
        d['block'] = d.seed.astype(str) + '|' + d.fold.astype(str)
        for m in metrics:
            if m not in d:
                continue
            W = d.pivot_table(index='block', columns='method', values=m)
            for other in ('none', 'random', 'filter'):
                if 'learned' not in W or other not in W:
                    continue
                pr = W[['learned', other]].dropna(); diff = pr.learned - pr[other]; nz = diff[diff != 0]
                rows.append(dict(protocol=protocol, family=fam, metric=m, vs=other, n_blocks=len(pr),
                                 learned=pr.learned.mean(), other=pr[other].mean(), diff=diff.mean(),
                                 wins=int((diff > 0).sum()), losses=int((diff < 0).sum()),
                                 p_wilcoxon=stats.wilcoxon(nz).pvalue if len(nz) >= 2 else np.nan))
    out = pd.DataFrame(rows)
    if not out.empty:                                   # Holm over the three controls within family and metric
        out['p_holm'] = np.nan
        for _, idx in out.groupby(['family', 'metric']).groups.items():
            pv = out.loc[idx, 'p_wilcoxon'].values.astype(float); ok = ~np.isnan(pv)
            if ok.any():
                h = np.full(len(pv), np.nan); h[ok] = holm(pv[ok]); out.loc[idx, 'p_holm'] = h
    return out


def fs_selection(R: pd.DataFrame, protocol: str, config: str = FULL) -> tuple:
    """How often each feature group is kept by the learned gates of `config`, and how stable the selected
    subsets are across folds and seeds (mean pairwise Jaccard similarity)."""
    d = R[(R.protocol == protocol) & (R.config == config)]
    if d.empty:
        return pd.DataFrame(), np.nan
    sets = [frozenset(g) for g in d.groups_kept]
    allg = sorted(set().union(*sets)) if sets else []
    freq = pd.Series({g: np.mean([g in s for s in sets]) for g in allg}, dtype=float).sort_values(ascending=False)
    jac = [len(a & b) / len(a | b) for a, b in itertools.combinations(sets, 2) if a | b]
    return freq.rename('kept_frac').to_frame(), float(np.mean(jac)) if jac else np.nan


def calibration_interaction(R: pd.DataFrame, protocol: str, response='worst_class_cov') -> pd.DataFrame:
    """Calibration mode (class-conditional vs marginal) crossed with the training components:
    effect of each training factor on the class-minus-marginal difference of the response."""
    d = design_rows(R, protocol)
    a, b = f'{response}_class', f'{response}_marginal'
    if a not in d or b not in d:
        return pd.DataFrame()
    d = d.assign(delta=d[a] - d[b]).dropna(subset=['delta'])
    if d.empty:
        return pd.DataFrame()
    e = effects(d.assign(protocol=protocol), protocol, 'delta')
    e = e[e.kind == 'main'].copy(); e['mean_delta'] = d.delta.mean()
    return e


def replication_check(R: pd.DataFrame, G: pd.DataFrame) -> pd.DataFrame:
    """The full configuration should reproduce the grid's EPIC runs (same config, seed, deterministic mode)."""
    f = R[R.config == FULL][['protocol', 'seed', 'fold', 'macro_f1']]
    g = G[G.model == 'EPIC'][['protocol', 'seed', 'fold', 'macro_f1']]
    m = f.merge(g, on=['protocol', 'seed', 'fold'], suffixes=('_factorial', '_grid'))
    m['abs_diff'] = (m.macro_f1_factorial - m.macro_f1_grid).abs()
    return m


# ---------------------------------------------------------------- driver: tables and figures for the paper
import matplotlib.pyplot as plt

FACT_RESPONSES = ['macro_f1', 'ece', 'cov_class', 'worst_class_cov_class', 'flops_ratio', 'answer_rate', 'selective_acc']


def analyse_factorial(show=True):
    R = load_factorial()
    if R.empty:
        print('No factorial runs yet: set RUN_FACTORIAL = True in Cell 19.'); return
    E = pd.concat([effects(R, p, r) for p in R.protocol.unique() for r in FACT_RESPONSES if r in R], ignore_index=True)
    E.to_csv(f'{FACT_DIR}/effects.csv', index=False)
    SH = pd.concat([shapley(E[(E.protocol == p) & (E.response == r)]).assign(protocol=p, response=r)
                    for p, r in E[['protocol', 'response']].drop_duplicates().itertuples(index=False)], ignore_index=True)
    SH.to_csv(f'{FACT_DIR}/shapley.csv', index=False)
    FC = pd.concat([fs_controls(R, p) for p in R.protocol.unique()], ignore_index=True)
    FC.to_csv(f'{FACT_DIR}/fs_controls.csv', index=False)
    CI = pd.concat([calibration_interaction(R, p) for p in R.protocol.unique()], ignore_index=True)
    CI.to_csv(f'{FACT_DIR}/calibration_interaction.csv', index=False)
    G = load_results()
    if not G.empty:
        rep = replication_check(R, G); rep.to_csv(f'{FACT_DIR}/replication.csv', index=False)
        if len(rep):
            print(f'Replication of the grid by the full configuration: {len(rep)} runs, '
                  f'max |dF1| = {rep.abs_diff.max():.4f}, mean {rep.abs_diff.mean():.4f}')
    for p in R.protocol.unique():
        e = E[(E.protocol == p) & (E.response == 'macro_f1')]
        if e.empty:
            continue
        syn = synergy(e); freq, jac = fs_selection(R, p); freq_a, jac_a = fs_selection(R, p, 'FS-alone')
        print(f'\n=== {p}: {e.n_blocks.iloc[0]} blocks (fold x seed), {e.n_runs.iloc[0]} runs, residual SD {e.resid_sd.iloc[0]:.3f}')
        print(f'all-on minus all-off: {syn["total"]:+.3f} F1 | sum of gains alone: {syn["sum_alone"]:+.3f} '
              f'| synergy {syn["synergy"]:+.3f} | interaction share {syn["interaction_share"]:.2f} '
              f'| Holm-significant interactions: {syn["n_sig_interactions"]}')
        if show:
            display(e[['term', 'label', 'effect', 'ci_lo', 'ci_hi', 'p_holm', 'p_wilcoxon_holm', 'sign_agreement']].round(4))
            display(FC[FC.protocol == p].round(4))
            print(f'groups kept, full configuration (stability: mean pairwise Jaccard {jac:.2f})'); display(freq.T.round(2))
            print(f'groups kept, feature selection alone (Jaccard {jac_a:.2f})'); display(freq_a.T.round(2))
        # figure: effects with 95% CI, main effects and interactions
        fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw=dict(width_ratios=[1, 1.3]))
        for a, kind in zip(ax, ['main', 'interaction']):
            q = e[e.kind == kind].iloc[::-1]
            a.errorbar(q.effect, range(len(q)), xerr=[q.effect - q.ci_lo, q.ci_hi - q.effect], fmt='o', color='k', capsize=2)
            a.axvline(0, color='grey', lw=0.8, ls='--'); a.set_yticks(range(len(q))); a.set_yticklabels(q.term)
            a.set_title(f'{kind} effects on macro-F1'); a.set_xlabel('effect (on minus off)')
        fig.suptitle(p); fig.tight_layout(); fig.savefig(f'{FACT_DIR}/effects_{p}.png', dpi=200)
        plt.show() if show else plt.close(fig)
    return E


if os.path.exists(FACT_LOG):
    FACT_EFFECTS = analyse_factorial()


## 6 · Key references:
Angelopoulos, A. N., & Bates, S. (2023). Conformal prediction: A gentle introduction. Foundations and Trends in Machine Learning, 16(4), 494–591. https://doi.org/10.1561/2200000101

Bai, S., Kolter, J. Z., & Koltun, V. (2018). An empirical evaluation of generic convolutional and recurrent networks for sequence modeling (arXiv:1803.01271). arXiv. https://doi.org/10.48550/arXiv.1803.01271

Barber, R. F., Candès, E. J., Ramdas, A., & Tibshirani, R. J. (2023). Conformal prediction beyond exchangeability. The Annals of Statistics, 51(2), 816–845. https://doi.org/10.1214/23-AOS2276

Chen, C., Li, O., Tao, D., Barnett, A., Rudin, C., & Su, J. K. (2019). This looks like that: Deep learning for interpretable image recognition. In Advances in Neural Information Processing Systems 32 (NeurIPS 2019). https://arxiv.org/abs/1806.10574


Abdullahi, S. B., et al. (2024). Spatial–temporal feature-based end-to-end Fourier network for 3D sign language recognition. Expert Systems with Applications, 238, 123258. https://doi.org/10.1016/j.eswa.2024.123258


Demšar, J. (2006). Statistical comparisons of classifiers over multiple data sets. Journal of Machine Learning Research, 7, 1–30. https://jmlr.org/papers/v7/demsar06a.html.


Ganin, Y., & Lempitsky, V. (2015). Unsupervised domain adaptation by backpropagation. In Proceedings of the 32nd International Conference on Machine Learning (Vol. 37, pp. 1180–1189). PMLR. https://proceedings.mlr.press/v37/ganin15.html

Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). On calibration of modern neural networks. In Proceedings of the 34th International Conference on Machine Learning (Vol. 70, pp. 1321–1330). PMLR. http://proceedings.mlr.press/v70/guo17a.html

Hardt, M., Price, E., & Srebro, N. (2016). Equality of opportunity in supervised learning. In Advances in Neural Information Processing Systems 29 (pp. 3315–3323). https://arxiv.org/abs/1610.02413

Hart, S. G., & Staveland, L. E. (1988). Development of NASA-TLX (Task Load Index): Results of empirical and theoretical research. Advances in Psychology, 52, 139–183. https://doi.org/10.1016/S0166-4115(08)62386-9

Hoffman, R. R., Mueller, S. T., Klein, G., & Litman, J. (2018). Metrics for explainable AI: Challenges and prospects (arXiv:1812.04608). arXiv. https://doi.org/10.48550/arXiv.1812.04608.

Jian, J.-Y., Bisantz, A. M., & Drury, C. G. (2000). Foundations for an empirically determined scale of trust in automated systems. International Journal of Cognitive Ergonomics, 4(1), 53–71. https://doi.org/10.1207/S15327566IJCE0401_04.

Abdullahi, S. B., & Mahbub, T. (2026). A blockchain-verified explainable AI pipeline for multi-scale agri-food carbon accounting. Computers & Industrial Engineering, 112304. https://doi.org/10.1016/j.cie.2026.112304

Jin, M., Wang, S., Ma, L., Chu, Z., Zhang, J. Y., Shi, X., Chen, P.-Y., Liang, Y., Li, Y., Pan, S., & Wen, Q. (2024). Time-LLM: Time series forecasting by reprogramming large language models. In The Twelfth International Conference on Learning Representations (ICLR 2024). OpenReview. https://openreview.net/forum?id=Unb5CVP.

Kim, T., Kim, J., Tae, Y., Park, C., Choi, J.-H., & Choo, J. (2022). Reversible instance normalization for accurate time-series forecasting against distribution shift. In 10th International Conference on Learning Representations (ICLR 2022). OpenReview. https://openreview.net/forum?id=cGDAkQo1C0p

Kleinberg, J., Mullainathan, S., & Raghavan, M. (2017). Inherent trade-offs in the fair determination of risk scores. In 8th Innovations in Theoretical Computer Science Conference (LIPIcs, Vol. 67, pp. 43:1–43:23). Schloss Dagstuhl. https://doi.org/10.4230/LIPIcs.ITCS.2017.43.

Stisen, A., Blunck, H., Bhattacharya, S., Prentow, T. S., Kjærgaard, M. B., Dey, A. K., Sonne, T., & Jensen, M. M. (2015). Smart devices are different: Assessing and mitigating mobile sensing heterogeneities for activity recognition. In Proceedings of the 13th ACM Conference on Embedded Networked Sensor Systems (pp. 127–140). ACM. https://doi.org/10.1145/2809695.2809718


Abdullahi, S. B., & Bolon-Canedo, V. (2026). Minimizing redundancy in hand dynamic features for enhanced sign language recognition. Intelligent Data Analysis, 30(3), 745–767. https://doi.org/10.1177/1088467X251367228


Voita, E., Talbot, D., Moiseev, F., Sennrich, R., & Titov, I. (2019). Analyzing multi-head self-attention: Specialized heads do the heavy lifting, the rest can be pruned. In Proceedings of the 57th Annual Meeting of the Association for Computational Linguistics (pp. 5797–5808). ACL. https://doi.org/10.18653/v1/P19-1580

Wachter, S., Mittelstadt, B., & Russell, C. (2018). Counterfactual explanations without opening the black box: Automated decisions and the GDPR. Harvard Journal of Law & Technology, 31(2), 841–887. https://jolt.law.harvard.edu/assets/articlePDFs/v31/Counterfactual-Explanations-without-Opening-the-Black-Box-Sandra-Wachter-et-al.pdf.

Alamri, F. S., Abdullahi, S. B., Ayesha, N., Alshannaq, F., & Saba, T. (2026). Bayesian-optimized convex neural networks with embedded feature selection for high-dimensional anthropometric health prediction. PeerJ Computer Science, 12, e3752. https://doi.org/10.7717/peerj-cs.3752

Zhang, Z., & Sabuncu, M. R. (2018). Generalized cross entropy loss for training deep neural networks with noisy labels. In Advances in Neural Information Processing Systems 31 (NeurIPS 2018). https://arxiv.org/abs/1805.07836

Zhou, T., Niu, P., Wang, X., Sun, L., & Jin, R. (2023). One fits all: Power general time series analysis by pretrained LM. In Advances in Neural Information Processing Systems 36 (NeurIPS 2023). https://doi.org/10.52202/075280-1877

Zhou, Y., Zhao, H., Huang, Y., Riedel, T., Hefenbrock, M., & Beigl, M. (2022). TinyHAR: A lightweight deep learning model designed for human activity recognition. In Proceedings of the 2022 ACM International Symposium on Wearable Computers (pp. 89–93). ACM. https://doi.org/10.1145/3544794.3558467.

Rafeal, Evangelista. (2019). Review of Zuboff’s The Age of Surveillance Capitalism. Sueveillance & Society. https://doi.org/10.24908/ss.v17i1/2.13132

Boulahia, S. Y., Anquetil, E., Multon, F., & Kulpa, R. (2017). Dynamic hand gesture recognition based on 3D pattern assembled trajectories. In 2017 Seventh International Conference on Image Processing Theory, Tools and Applications (IPTA) (pp. 1–6). IEEE. https://doi.org/10.1109/IPTA.2017.8310146.